# Legumin hexamer: GROMACS setup, runs and analysis (Colab)

Working notebook for the pea legumin hexamer simulations in this repository: system setup, 1 ns runs at 300 K, 373 K and 373 K with box shear (two replicates each), and the analysis of every run.

- **Software:** GROMACS 2026.3 (conda-forge CUDA build) on a Colab T4 GPU; MDAnalysis, SciPy, pandas and Matplotlib for analysis. See `environment.yml`.
- **Layout:** the sections follow the order the work was done in. Sections 1–3 are setup and minimisation (3b is troubleshooting), 4–7 are the runs, 8–10 the first analysis, 11–13 replicate 2 and the all-runs comparison.
- **Paths:** file paths point to the author's Google Drive folder (`/content/drive/MyDrive/Plant Protein Extrusion MD`). Change them to your own folder before running.
- **Outputs are cleared.** Figures and tables are in `analysis/`, and the run settings are in `simulation_setup/`.

## 1. Mount Drive and inspect the AlphaFold3 structure
Mount Google Drive and check the legumin hexamer PDB (6 chains, 496 residues each).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Update this path if Legumin_MD is inside another folder
PDB_PATH = "/content/drive/MyDrive/Legumin_MD/legumin_hexamer.pdb"

if os.path.exists(PDB_PATH):
    print("PDB found!")
    print("File size:", round(os.path.getsize(PDB_PATH) / 1024, 2), "KB")
else:
    print("PDB not found. Check the folder path.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

PDB_PATH = "/content/drive/MyDrive/Legumin MD/legumin_hexamer.pdb"

if os.path.isfile(PDB_PATH):
    print("PDB found!")
    print("Path:", PDB_PATH)
    print("File size:", round(os.path.getsize(PDB_PATH) / 1024, 2), "KB")
else:
    print("Still not found. Folders in MyDrive:")
    for item in os.listdir("/content/drive/MyDrive"):
        print(item)

In [ ]:
import os

FOLDER = "/content/drive/MyDrive/Legumin MD "
PDB_PATH = os.path.join(FOLDER, "legumin_hexamer.pdb")

print("Folder exists:", os.path.isdir(FOLDER))

if os.path.isdir(FOLDER):
    print("\nFiles in the folder:")
    for item in os.listdir(FOLDER):
        print(repr(item))

    if os.path.isfile(PDB_PATH):
        print("\nPDB FOUND:", PDB_PATH)
        print("File size:", round(os.path.getsize(PDB_PATH) / 1024, 2), "KB")
    else:
        print("\nThe folder exists, but legumin_hexamer.pdb is not directly inside it.")

In [ ]:
from collections import defaultdict

PDB_PATH = "/content/drive/MyDrive/Legumin MD /legumin_hexamer.pdb"

chains = defaultdict(set)
atom_counts = defaultdict(int)

with open(PDB_PATH, "r") as f:
    for line in f:
        if line.startswith("ATOM  "):
            chain = line[21].strip() or "_"
            residue_number = line[22:27].strip()
            residue_name = line[17:20].strip()

            chains[chain].add((residue_number, residue_name))
            atom_counts[chain] += 1

print("Number of chains:", len(chains))
print("Chain IDs:", sorted(chains.keys()))

for chain in sorted(chains):
    print(
        f"Chain {chain}: "
        f"{len(chains[chain])} residues, "
        f"{atom_counts[chain]} atoms"
    )

print("Total protein atoms:", sum(atom_counts.values()))

In [ ]:
from collections import Counter, defaultdict

PDB_PATH = "/content/drive/MyDrive/Legumin MD /legumin_hexamer.pdb"

residues = defaultdict(dict)
record_counts = Counter()
atom_names = defaultdict(Counter)
other_residues = Counter()

with open(PDB_PATH) as f:
    for line in f:
        record = line[:6].strip()

        if record:
            record_counts[record] += 1

        if record == "ATOM":
            chain = line[21].strip() or "_"
            resid = (line[22:27].strip(), line[17:20].strip())
            atom_name = line[12:16].strip()

            residues[chain][resid] = True
            atom_names[chain][atom_name] += 1

        elif record == "HETATM":
            other_residues[line[17:20].strip()] += 1

print("PDB record types:", record_counts)
print("\nResidue numbering by chain:")

for chain in sorted(residues):
    numbers = [int(r[0].split()[0]) for r in residues[chain]]
    print(
        f"Chain {chain}: first={min(numbers)}, "
        f"last={max(numbers)}, unique residues={len(numbers)}"
    )

print("\nHETATM residue types:", other_residues or "None")
print("\nUnique atom names in chain A:")
print(sorted(atom_names["A"].keys()))

In [ ]:
%pip install biopython

In [ ]:
from Bio.PDB import PDBParser
from Bio.SeqUtils import seq1

PDB_PATH = "/content/drive/MyDrive/Legumin MD /legumin_hexamer.pdb"

parser = PDBParser(QUIET=True)
structure = parser.get_structure("legumin", PDB_PATH)

sequences = {}

for chain in structure[0]:
    residues = [
        r for r in chain
        if r.id[0] == " " and r.resname in {
            "ALA","ARG","ASN","ASP","CYS","GLN","GLU","GLY",
            "HIS","ILE","LEU","LYS","MET","PHE","PRO","SER",
            "THR","TRP","TYR","VAL"
        }
    ]

    sequence = "".join(seq1(r.resname) for r in residues)
    numbers = [r.id[1] for r in residues]

    sequences[chain.id] = sequence
    gaps = [
        (numbers[i], numbers[i+1])
        for i in range(len(numbers)-1)
        if numbers[i+1] != numbers[i] + 1
    ]

    print(f"Chain {chain.id}: {len(sequence)} amino acids")
    print(f"  Numbering gaps: {gaps if gaps else 'None'}")
    print(f"  First 10 residues: {sequence[:10]}")
    print(f"  Last 10 residues:  {sequence[-10:]}")

print("\nAll six chain sequences identical:",
      len(set(sequences.values())) == 1)

## 2. Install GROMACS 2026.3 (CUDA build)
Check the Colab GPU, install condacolab and create the `gmx_cuda` conda environment with the conda-forge CUDA build of GROMACS 2026.3.

In [ ]:
import os
import shutil

print("gmx found:", shutil.which("gmx"))
print("gmx_mpi found:", shutil.which("gmx_mpi"))
print("Conda:", os.environ.get("CONDA_PREFIX", "Not set"))

!which conda
!ls /usr/local/bin/gmx*
!ls /usr/local/envs/

In [ ]:
import shutil
import os

for cmd in ["conda", "mamba", "micromamba", "apt-get", "bash"]:
    print(f"{cmd}: {shutil.which(cmd)}")

print("Python:", os.sys.version)
print("Platform:", os.uname().machine)

In [ ]:
!nvidia-smi
!nvcc --version

In [ ]:
%pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
import condacolab
condacolab.check()

!nvidia-smi
!conda --version

In [ ]:
!CONDA_OVERRIDE_CUDA=12.9 conda create -y \
    -n gmx_cuda \
    -c conda-forge \
    "gromacs=2026.3=nompi_cuda*"

In [ ]:
!conda run -n gmx_cuda gmx --version

In [ ]:
!conda run -n gmx_cuda gmx mdrun -version

In [ ]:
!conda run -n gmx_cuda python -c "import shutil; print('GROMACS executable:', shutil.which('gmx'))"
!nvidia-smi

## 3. System build and energy minimisation
`pdb2gmx` (AMBER99SB-ILDN, SPC/E, `-merge no`) → `editconf` (rhombic dodecahedron, `-d 1.2`) → `solvate` → `genion` → `em`.

In [ ]:
import os
import shutil

SOURCE = "/content/drive/MyDrive/Legumin MD /legumin_hexamer.pdb"
PROJECT = "/content/drive/MyDrive/Plant Protein Extrusion MD"
STRUCTURES = os.path.join(PROJECT, "structures")
ANALYSIS = os.path.join(PROJECT, "analysis")
SIMULATIONS = os.path.join(PROJECT, "simulations")

for folder in [STRUCTURES, ANALYSIS, SIMULATIONS]:
    os.makedirs(folder, exist_ok=True)

DEST = os.path.join(STRUCTURES, "legumin_hexamer_original.pdb")

if not os.path.isfile(SOURCE):
    raise FileNotFoundError(f"Source PDB not found: {SOURCE}")

if not os.path.exists(DEST):
    shutil.copy2(SOURCE, DEST)

print("Project folder:", PROJECT)
print("Original PDB preserved:", os.path.isfile(SOURCE))
print("Working copy:", DEST)
print("Working copy size:", round(os.path.getsize(DEST) / 1024, 2), "KB")
print("GROMACS environment: /usr/local/envs/gmx_cuda")

In [ ]:
!conda run -n gmx_cuda gmx pdb2gmx -h

In [ ]:
from pathlib import Path

GMX_DATA = Path("/usr/local/envs/gmx_cuda/share/gromacs/top")

print("Topology directory exists:", GMX_DATA.is_dir())

if GMX_DATA.is_dir():
    print("\nInstalled Amber force fields:")
    for item in sorted(GMX_DATA.glob("*amber*.ff")):
        print(item.name)

    print("\nAll force-field directories:")
    for item in sorted(GMX_DATA.glob("*.ff")):
        print(item.name)
else:
    print("Directory not found; checking the environment:")
    !conda run -n gmx_cuda gmx --version

In [ ]:
%cd "/content/drive/MyDrive/Plant Protein Extrusion MD"

!conda run -n gmx_cuda gmx pdb2gmx \
    -f structures/legumin_hexamer_original.pdb \
    -o structures/legumin_hexamer_processed.gro \
    -p topol.top \
    -i posre.itp \
    -ff amber99sb-ildn \
    -water spce \
    -chainsep id_or_ter \
    -merge no

In [ ]:
import os

PROJECT = "/content/drive/MyDrive/Plant Protein Extrusion MD"

files = [
    "structures/legumin_hexamer_processed.gro",
    "topol.top",
    "posre.itp",
]

for name in files:
    path = os.path.join(PROJECT, name)
    print(f"{name}: {'FOUND' if os.path.isfile(path) else 'MISSING'}")
    if os.path.isfile(path):
        print(f"  Size: {os.path.getsize(path) / 1024:.1f} KB")

print("\nTopology molecule definitions:")
with open(os.path.join(PROJECT, "topol.top")) as f:
    show = False
    for line in f:
        if line.strip() == "[ molecules ]":
            show = True
        if show:
            print(line.rstrip())

In [ ]:
import os

PROJECT = "/content/drive/MyDrive/Plant Protein Extrusion MD"

print("Checking possible position-restraint files:\n")

for root, dirs, files in os.walk(PROJECT):
    for name in files:
        if name == "posre.itp" or name.startswith("posre_"):
            path = os.path.join(root, name)
            print(path, f"({os.path.getsize(path) / 1024:.1f} KB)")

print("\nChecking topology includes:")
with open(os.path.join(PROJECT, "topol.top")) as f:
    for line in f:
        if "#include" in line or "[ molecules ]" in line:
            print(line.rstrip())

In [ ]:
import os

PROJECT = "/content/drive/MyDrive/Plant Protein Extrusion MD"

for chain in "ABCDEF":
    filename = f"topol_Protein_chain_{chain}.itp"
    path = os.path.join(PROJECT, filename)

    print(f"\n--- Chain {chain}: {filename} ---")

    if not os.path.isfile(path):
        print("MISSING")
        continue

    with open(path) as f:
        for line in f:
            if "posre" in line.lower() or line.strip() == "[ moleculetype ]":
                print(line.rstrip())

In [ ]:
%cd "/content/drive/MyDrive/Plant Protein Extrusion MD"

!conda run -n gmx_cuda gmx editconf \
    -f structures/legumin_hexamer_processed.gro \
    -o structures/legumin_hexamer_box.gro \
    -c \
    -d 1.2 \
    -bt dodecahedron

In [ ]:
%cd "/content/drive/MyDrive/Plant Protein Extrusion MD"

!conda run -n gmx_cuda gmx solvate \
    -cp structures/legumin_hexamer_box.gro \
    -cs spc216.gro \
    -o structures/legumin_hexamer_solv.gro \
    -p topol.top

In [ ]:
from pathlib import Path

project = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")
mdp = project / "em.mdp"

mdp.write_text("""; Energy minimization
integrator      = steep
emtol           = 1000.0
emstep          = 0.01
nsteps          = 50000

cutoff-scheme   = Verlet
nstlist         = 20
coulombtype     = PME
rcoulomb        = 1.0
rvdw            = 1.0
pbc             = xyz

constraints     = h-bonds
""")

print(f"Created: {mdp}")
print(mdp.read_text())

In [ ]:
from pathlib import Path

project = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")
top = project / "topol.top"

print("Topology solvent and ion entries:")
for line in top.read_text().splitlines():
    if "spce.itp" in line or "ions.itp" in line or line.strip().startswith("SOL"):
        print(line)

ions = Path("/usr/local/envs/gmx_cuda/share/gromacs/top/amber99sb-ildn.ff/ions.itp")
print("\nIon definitions file exists:", ions.exists())

if ions.exists():
    for line in ions.read_text().splitlines():
        if line.strip().startswith("[ ") or line.strip().startswith("NA") or line.strip().startswith("CL"):
            print(line)

In [ ]:
%cd "/content/drive/MyDrive/Plant Protein Extrusion MD"

!conda run -n gmx_cuda gmx grompp \
    -f em.mdp \
    -c structures/legumin_hexamer_solv.gro \
    -p topol.top \
    -o ions.tpr \
    -maxwarn 1

In [ ]:
%cd "/content/drive/MyDrive/Plant Protein Extrusion MD"

!printf "SOL\n" | conda run -n gmx_cuda gmx genion \
    -s ions.tpr \
    -o structures/legumin_hexamer_ions.gro \
    -p topol.top \
    -pname NA \
    -nname CL \
    -neutral

In [ ]:
%cd "/content/drive/MyDrive/Plant Protein Extrusion MD"

!conda run -n gmx_cuda gmx genion \
    -s ions.tpr \
    -o structures/legumin_hexamer_ions.gro \
    -p topol.top \
    -pname NA \
    -nname CL \
    -neutral \
    --select 'group "SOL"'

In [ ]:
%cd "/content/drive/MyDrive/Plant Protein Extrusion MD"

!bash -c 'printf "13\n" | conda run -n gmx_cuda gmx genion \
    -s ions.tpr \
    -o structures/legumin_hexamer_ions.gro \
    -p topol.top \
    -pname NA \
    -nname CL \
    -neutral'

In [ ]:
%cd "/content/drive/MyDrive/Plant Protein Extrusion MD"

!printf '13\n' | /usr/local/envs/gmx_cuda/bin.AVX2_256/gmx genion \
    -s ions.tpr \
    -o structures/legumin_hexamer_ions.gro \
    -p topol.top \
    -pname NA \
    -nname CL \
    -neutral

In [ ]:
from pathlib import Path

project = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")
gro = project / "structures/legumin_hexamer_ions.gro"
top = project / "topol.top"

print("Ionized structure exists:", gro.exists())

if gro.exists():
    with gro.open() as f:
        f.readline()
        atom_count = int(f.readline().strip())
    print("Total atoms in ionized structure:", atom_count)

print("\nUpdated topology:")
for line in top.read_text().splitlines():
    if line.strip().startswith(("SOL", "NA", "CL")):
        print(line)

In [ ]:
%cd "/content/drive/MyDrive/Plant Protein Extrusion MD"

!conda run -n gmx_cuda gmx grompp \
    -f em.mdp \
    -c structures/legumin_hexamer_ions.gro \
    -p topol.top \
    -o em.tpr \
    -maxwarn 0

In [ ]:
%cd "/content/drive/MyDrive/Plant Protein Extrusion MD"

!conda run -n gmx_cuda gmx mdrun \
    -deffnm em \
    -ntmpi 1 \
    -ntomp 4 \
    -pin off

In [ ]:
import os

PROJECT = "/content/drive/MyDrive/Plant Protein Extrusion MD"

for name in ["em.gro", "em.edr", "em.log", "em.tpr"]:
    path = os.path.join(PROJECT, name)
    print(
        f"{name}: {'FOUND' if os.path.exists(path) else 'MISSING'}"
        + (f" | {os.path.getsize(path)/1024:.1f} KB"
           if os.path.exists(path) else "")
    )

In [ ]:
%cd "/content/drive/MyDrive/Plant Protein Extrusion MD"

!grep -c "LINCS WARNING" em.log
!tail -n 60 em.log

In [ ]:
%cd "/content/drive/MyDrive/Plant Protein Extrusion MD"

!conda run -n gmx_cuda gmx check -f em.gro

## 3b. Energy minimisation troubleshooting (diagnostic)
Checks after the first minimisation reported LINCS warnings: a second and third minimisation (`em2`, `em3`), atom-clash and topology checks. The NVT runs below start from `em.gro`.

In [ ]:
from pathlib import Path

project = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")

mdp_text = """integrator      = steep
emtol           = 100.0
emstep          = 0.001
nsteps          = 10000

nstlist         = 20
cutoff-scheme   = Verlet
coulombtype     = PME
rcoulomb        = 1.0
rvdw            = 1.0
pbc             = xyz
"""

(project / "em2.mdp").write_text(mdp_text)
print("Created:", project / "em2.mdp")
print(mdp_text)

In [ ]:
%cd "/content/drive/MyDrive/Plant Protein Extrusion MD"

!conda run -n gmx_cuda gmx grompp \
    -f em2.mdp \
    -c em.gro \
    -p topol.top \
    -o em2.tpr \
    -maxwarn 0

In [ ]:
%cd "/content/drive/MyDrive/Plant Protein Extrusion MD"

!conda run -n gmx_cuda gmx mdrun \
    -deffnm em2 \
    -ntmpi 1 \
    -ntomp 2 \
    -pin off

In [ ]:
%cd "/content/drive/MyDrive/Plant Protein Extrusion MD"

!grep -n -E "LINCS WARNING|constraint|converged|Maximum force|Potential Energy" em2.log | tail -n 40

In [ ]:
from pathlib import Path

project = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")

# Show the residue and atom information for the atom with the
# highest reported force (GROMACS atom numbers start at 1).
target = 12174
gro = project / "em2.gro"

with gro.open() as f:
    lines = f.readlines()

atom_line = lines[2 + target - 1]
print("Atom number:", target)
print("Atom name:", atom_line[10:15].strip())
print("Residue number:", atom_line[0:5].strip())
print("Residue name:", atom_line[5:10].strip())
print("Coordinates (nm):", atom_line[20:44].strip())

In [ ]:
import numpy as np
from pathlib import Path

gro = Path("/content/drive/MyDrive/Plant Protein Extrusion MD/em2.gro")

with gro.open() as f:
    lines = f.readlines()

target_atom = 12174
target_line = lines[2 + target_atom - 1]
target_xyz = np.array([
    float(target_line[20:28]),
    float(target_line[28:36]),
    float(target_line[36:44])
])

print("Target:", target_line[:20].strip())
print("Nearby atoms within 0.30 nm:")

found = 0
for i, line in enumerate(lines[2:-1], start=1):
    if i == target_atom:
        continue
    try:
        xyz = np.array([
            float(line[20:28]),
            float(line[28:36]),
            float(line[36:44])
        ])
    except ValueError:
        continue

    delta = xyz - target_xyz
    distance = np.linalg.norm(delta)

    # The box may require periodic-boundary handling, so this is
    # only a preliminary, non-periodic neighbor check.
    if distance < 0.30:
        print(f"Atom {i:6d} | {line[:20].strip():20s} | {distance:.3f} nm")
        found += 1

print("Neighbors found:", found)

In [ ]:
from pathlib import Path

project = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")

# Search saved logs and backup logs for LINCS warnings
for log in sorted(project.glob("*em*.log")):
    print(f"\n--- {log.name} ---")
    text = log.read_text(errors="ignore").splitlines()
    matches = [
        (i, line) for i, line in enumerate(text)
        if "LINCS WARNING" in line or "relative constraint deviation" in line
    ]

    if not matches:
        print("No LINCS warnings recorded in this log.")
    else:
        for i, line in matches[:20]:
            print(f"Line {i + 1}: {line}")
            for j in range(i + 1, min(i + 4, len(text))):
                print("   ", text[j])

In [ ]:
%cd "/content/drive/MyDrive/Plant Protein Extrusion MD"

!conda run -n gmx_cuda gmx energy -f em2.edr -o em2_potential.xvgv

In [ ]:
from pathlib import Path

project = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")

for name in ["em.gro", "em.edr", "em.log", "em2.gro", "em2.edr", "em2.log"]:
    p = project / name
    print(f"{name}: {'FOUND' if p.exists() else 'MISSING'}")

print("\nFirst minimization:")
print("Potential energy: -3649198.2 kJ/mol")
print("Maximum force: 838.92 kJ mol^-1 nm^-1")

print("\nSecond minimization:")
print("Potential energy: -3666665.2 kJ/mol")
print("Maximum force: 3599.47 kJ mol^-1 nm^-1")

In [ ]:
from pathlib import Path
import numpy as np

project = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")

def read_gro(path):
    with open(path) as f:
        lines = f.readlines()
    n = int(lines[1].strip())
    coords = np.array([
        [float(line[20:28]), float(line[28:36]), float(line[36:44])]
        for line in lines[2:2+n]
    ])
    return lines, n, coords

for name in ["em.gro", "em2.gro"]:
    lines, n, coords = read_gro(project / name)
    print(f"\n{name}: {n} atoms")
    print("All coordinates finite:", np.isfinite(coords).all())
    print("Coordinate bounds (nm):")
    print("  min:", coords.min(axis=0))
    print("  max:", coords.max(axis=0))

lines1, n1, xyz1 = read_gro(project / "em.gro")
lines2, n2, xyz2 = read_gro(project / "em2.gro")

if n1 == n2:
    displacement = np.linalg.norm(xyz2 - xyz1, axis=1)
    print("\nDifference between em and em2:")
    print("Mean atom displacement:", displacement.mean(), "nm")
    print("Maximum atom displacement:", displacement.max(), "nm")
    print("Atoms displaced > 0.1 nm:", int((displacement > 0.1).sum()))
else:
    print("Atom counts differ; cannot compare coordinates directly.")

In [ ]:
from pathlib import Path
import numpy as np

project = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")

def read_gro(path):
    with open(path) as f:
        lines = f.readlines()
    n = int(lines[1].strip())
    atom_lines = lines[2:2+n]
    coords = np.array([
        [float(line[20:28]), float(line[28:36]), float(line[36:44])]
        for line in atom_lines
    ])
    # Protein residues are listed before solvent in this system.
    protein_mask = np.array([
        line[5:10].strip() not in {"SOL", "NA", "CL"}
        for line in atom_lines
    ])
    box = np.array([float(x) for x in lines[2+n].split()[:3]])
    return atom_lines, coords, protein_mask, box

l1, x1, protein1, box1 = read_gro(project / "em.gro")
l2, x2, protein2, box2 = read_gro(project / "em2.gro")

mask = protein1 & protein2
delta = x2[mask] - x1[mask]
box = box1
delta -= box * np.round(delta / box)
d = np.linalg.norm(delta, axis=1)

print("Protein atoms compared:", int(mask.sum()))
print("Mean periodic-corrected displacement:", round(float(d.mean()), 4), "nm")
print("Median displacement:", round(float(np.median(d)), 4), "nm")
print("Maximum displacement:", round(float(d.max()), 4), "nm")
print("Protein atoms displaced > 0.1 nm:", int((d > 0.1).sum()))
print("Protein atoms displaced > 0.3 nm:", int((d > 0.3).sum()))

In [ ]:
from pathlib import Path
import numpy as np

project = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")

def read_gro(path):
    with open(path) as f:
        lines = f.readlines()
    n = int(lines[1].strip())
    atoms = lines[2:2+n]
    xyz = np.array([
        [float(a[20:28]), float(a[28:36]), float(a[36:44])]
        for a in atoms
    ])
    box = np.array([float(x) for x in lines[2+n].split()[:3]])
    return atoms, xyz, box

a1, x1, box1 = read_gro(project / "em.gro")
a2, x2, box2 = read_gro(project / "em2.gro")

protein = np.array([
    a[5:10].strip() not in {"SOL", "NA", "CL"}
    for a in a1
])

delta = x2 - x1
delta -= box1 * np.round(delta / box1)
d = np.linalg.norm(delta, axis=1)

indices = np.where(protein & (d > 0.1))[0]

print("Atoms with apparent displacement > 0.1 nm:")
for i in indices:
    print(
        f"Atom {i+1}: {a1[i][:20].strip():20s} "
        f"displacement={d[i]:.3f} nm"
    )

In [ ]:
from pathlib import Path
import numpy as np

project = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")

def inspect_gro(filename):
    path = project / filename
    with open(path) as f:
        lines = f.readlines()

    n = int(lines[1].strip())
    atoms = lines[2:2+n]
    xyz = np.array([
        [float(a[20:28]), float(a[28:36]), float(a[36:44])]
        for a in atoms
    ])
    box = np.array([float(v) for v in lines[2+n].split()[:3]])

    return n, xyz, box

for filename in ["em.gro", "em2.gro"]:
    n, xyz, box = inspect_gro(filename)
    print(f"\n{filename}")
    print("Atom count:", n)
    print("All coordinates finite:", np.isfinite(xyz).all())
    print("Box dimensions (nm):", box)
    print("Coordinate range (nm):", xyz.min(axis=0), "to", xyz.max(axis=0))

n1, x1, b1 = inspect_gro("em.gro")
n2, x2, b2 = inspect_gro("em2.gro")

print("\nAtom counts match:", n1 == n2)
print("Box dimensions match:", np.allclose(b1, b2))

In [ ]:
from pathlib import Path

project = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")

for filename in ["em.log", "em2.log"]:
    path = project / filename
    text = path.read_text(errors="replace")

    print(f"\n{'='*20} {filename} {'='*20}")
    for line in text.splitlines():
        if any(term in line.lower() for term in [
            "converged to", "maximum force", "potential energy",
            "stopped because", "converged"
        ]):
            print(line.strip())

In [ ]:
from pathlib import Path

project = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")

def show_atom(filename, atom_number):
    lines = (project / filename).read_text().splitlines()
    atom_count = int(lines[1])
    line = lines[1 + atom_number]  # GRO atom number is 1-based
    print(
        f"{filename}: atom {atom_number} | "
        f"residue/name: {line[:20].strip()} | "
        f"coordinates: {line[20:44].strip()}"
    )

for filename in ["em.gro", "em2.gro"]:
    show_atom(filename, 12174)

In [ ]:
from pathlib import Path
import numpy as np

project = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")
lines = (project / "em2.gro").read_text().splitlines()
n = int(lines[1].strip())
atoms = lines[2:2+n]

def atom_info(atom_number):
    line = atoms[atom_number - 1]
    xyz = np.array([
        float(line[20:28]),
        float(line[28:36]),
        float(line[36:44])
    ])
    return line[:20].strip(), xyz

center = 12174
center_name, center_xyz = atom_info(center)

print("Central atom:", center_name)
print("\nNearby ASP 294 side-chain atoms:")

for i, line in enumerate(atoms, start=1):
    residue = line[5:10].strip()
    atom_name = line[12:16].strip()

    if residue == "294ASP" and atom_name in {
        "CB", "CG", "OD1", "OD2", "HB1", "HB2"
    }:
        name, xyz = atom_info(i)
        distance = np.linalg.norm(xyz - center_xyz)
        print(f"Atom {i:6d} | {name:20s} | distance = {distance:.3f} nm")

In [ ]:
from pathlib import Path
import numpy as np

path = Path("/content/drive/MyDrive/Plant Protein Extrusion MD/em2.gro")
lines = path.read_text().splitlines()
n = int(lines[1].strip())
atoms = lines[2:2+n]

center_idx = 12174 - 1
center_line = atoms[center_idx]
center_xyz = np.array([
    float(center_line[20:28]),
    float(center_line[28:36]),
    float(center_line[36:44])
])

print("Central atom:", center_line[:20])
print("\nAtoms in the same residue:")

for i, line in enumerate(atoms):
    if line[0:5] == center_line[0:5]:
        xyz = np.array([
            float(line[20:28]),
            float(line[28:36]),
            float(line[36:44])
        ])
        distance = np.linalg.norm(xyz - center_xyz)
        print(f"Atom {i+1:6d} | {line[:20].strip():20s} | {distance:.3f} nm")

In [ ]:
from pathlib import Path

project = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")

for filename in ["topol.top", "posre.itp"]:
    path = project / filename
    print(f"\n{filename}: exists = {path.exists()}")
    if path.exists():
        print("Size:", path.stat().st_size, "bytes")

In [ ]:
from pathlib import Path

gro = Path("/content/drive/MyDrive/Plant Protein Extrusion MD/em.gro")
if not gro.exists():
    raise FileNotFoundError(f"Cannot find {gro}")

lines = gro.read_text().splitlines()
n_atoms = int(lines[1].strip())
atoms = []

for i, line in enumerate(lines[2:2 + n_atoms], start=1):
    atoms.append({
        "id": i,
        "resid": line[0:5].strip(),
        "resname": line[5:10].strip(),
        "atomname": line[10:15].strip(),
        "x": float(line[20:28]),
        "y": float(line[28:36]),
        "z": float(line[36:44]),
    })

targets = [
    (14660, 14661),
    (13254, 13255, 13256, 13257),
    (43310, 43311, 43312),
]

print(f"File: {gro.name} | Atoms parsed: {len(atoms)}")

for group in targets:
    print("\n" + "=" * 65)
    print("LINCS warning atom IDs:", group)

    for atom_id in group:
        if not 1 <= atom_id <= len(atoms):
            print(f"Atom {atom_id}: OUT OF RANGE")
            continue

        a = atoms[atom_id - 1]
        print(
            f"\nAtom {a['id']}: {a['resname']}{a['resid']} "
            f"{a['atomname']} | "
            f"xyz=({a['x']:.3f}, {a['y']:.3f}, {a['z']:.3f}) nm"
        )

        lo = max(1, atom_id - 3)
        hi = min(len(atoms), atom_id + 3)
        print("Nearby atoms (global ID, residue, atom):")

        for j in range(lo, hi + 1):
            b = atoms[j - 1]
            print(
                f"  {b['id']:5d}  {b['resname']}{b['resid']:>4} "
                f"{b['atomname']:>4}"
            )

In [ ]:
from pathlib import Path

work = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")

print("PDB files in project folder:")
pdbs = sorted(work.glob("*.pdb"))

if pdbs:
    for p in pdbs:
        print(f"{p.name}  ({p.stat().st_size:,} bytes)")
else:
    print("No PDB files found in the project root.")

print("\nFiles mentioning LINCS steps or coordinates:")
for p in sorted(work.iterdir()):
    if any(s in p.name.lower() for s in ["step", "lincs", "prev", "current"]):
        print(p.name)

In [ ]:
from pathlib import Path
import numpy as np

work = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")

# Atom IDs are GROMACS 1-based atom numbers.
checks = {
    "step43":  (14660, 14661, 0.0960),
    "step93":  (43713, 43714, 0.1010),
    "step126": (43310, 43311, 0.1090),
    "step126_extra": (43310, 43312, 0.1090),
}

def read_gro_or_pdb(path):
    coords = {}
    if path.suffix.lower() == ".pdb":
        for line in path.read_text(errors="replace").splitlines():
            if line.startswith(("ATOM  ", "HETATM")):
                try:
                    serial = int(line[6:11])
                    xyz = np.array([
                        float(line[30:38]),
                        float(line[38:46]),
                        float(line[46:54])
                    ])
                    coords[serial] = xyz
                except ValueError:
                    pass
    return coords

print("Distances in snapshot pairs (nm)")
for step, (a1, a2, target) in checks.items():
    if "_extra" in step:
        continue
    print(f"\n{step}: atoms {a1}-{a2}; reported target {target:.3f} nm")
    for suffix in ("b", "c"):
        path = work / f"{step}{suffix}.pdb"
        if not path.exists():
            print(f"  {path.name}: missing")
            continue
        coords = read_gro_or_pdb(path)
        if a1 in coords and a2 in coords:
            d = np.linalg.norm(coords[a1] - coords[a2])
            print(f"  {suffix}: {d:.4f} nm")
        else:
            print(f"  {suffix}: atom serials not found (PDB serial numbering may differ)")

In [ ]:
from pathlib import Path

work = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")

for filename in ["step43b.pdb", "step43c.pdb",
                 "step126b.pdb", "step126c.pdb"]:
    path = work / filename
    print(f"\n===== {filename} =====")
    count = 0

    with path.open(errors="replace") as f:
        for line in f:
            if line.startswith(("ATOM  ", "HETATM")):
                if count < 5:
                    print(repr(line.rstrip()))
                count += 1

    print("Total ATOM/HETATM records:", count)

In [ ]:
from pathlib import Path
import numpy as np

work = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")

checks = [
    ("step43", 14660, 14661, 0.0960),
    ("step93", 43713, 43714, 0.1010),
    ("step126", 43310, 43311, 0.1090),
    ("step126", 43310, 43312, 0.1090),
]

def read_pdb_coords(path):
    coords = {}
    with open(path, errors="replace") as f:
        for line in f:
            if line.startswith(("ATOM  ", "HETATM")):
                try:
                    serial = int(line[6:11])
                    coords[serial] = np.array([
                        float(line[30:38]),
                        float(line[38:46]),
                        float(line[46:54])
                    ])
                except ValueError:
                    continue
    return coords

for step, a1, a2, target in checks:
    print(f"\n{step}: atoms {a1}-{a2}; target {target:.4f} nm")
    for suffix in ("b", "c"):
        path = work / f"{step}{suffix}.pdb"
        coords = read_pdb_coords(path)
        if a1 in coords and a2 in coords:
            distance_nm = np.linalg.norm(coords[a1] - coords[a2]) / 10
            print(f"  {suffix}: {distance_nm:.4f} nm")
        else:
            print(f"  {suffix}: atom IDs unavailable")

In [ ]:
from pathlib import Path

work = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")

for filename in ["step126b.pdb", "step126c.pdb"]:
    print(f"\n===== {filename} =====")
    path = work / filename

    wanted = {13254, 13255, 13256, 13257, 14660, 14661,
              43310, 43311, 43312}

    with path.open(errors="replace") as f:
        for line in f:
            if line.startswith("ATOM  "):
                try:
                    serial = int(line[6:11])
                except ValueError:
                    continue
                if serial in wanted:
                    print(line.rstrip())

In [ ]:
from pathlib import Path
import numpy as np

work = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")

def pdb_atoms(path):
    result = {}
    with open(path, errors="replace") as f:
        for line in f:
            if line.startswith("ATOM  "):
                try:
                    serial = int(line[6:11])
                    result[serial] = {
                        "name": line[12:16].strip(),
                        "res": line[17:20].strip(),
                        "resid": line[22:26].strip(),
                        "xyz": np.array([
                            float(line[30:38]),
                            float(line[38:46]),
                            float(line[46:54])
                        ]) / 10.0
                    }
                except ValueError:
                    pass
    return result

def gro_atoms(path):
    lines = path.read_text().splitlines()
    n = int(lines[1].strip())
    result = {}
    for idx, line in enumerate(lines[2:2+n], start=1):
        result[idx] = {
            "name": line[10:15].strip(),
            "res": line[5:10].strip(),
            "resid": line[0:5].strip(),
            "xyz": np.array([
                float(line[20:28]),
                float(line[28:36]),
                float(line[36:44])
            ])
        }
    return result

pairs = [(43310, 43311), (43310, 43312), (14660, 14661),
         (13254, 13255), (13254, 13256), (13254, 13257)]

sources = {
    "em.gro": gro_atoms(work / "em.gro"),
    "step126b.pdb": pdb_atoms(work / "step126b.pdb"),
    "step126c.pdb": pdb_atoms(work / "step126c.pdb"),
}

for a, b in pairs:
    print(f"\nAtoms {a}-{b}")
    for label, atoms in sources.items():
        if a in atoms and b in atoms:
            A, B = atoms[a], atoms[b]
            d = np.linalg.norm(A["xyz"] - B["xyz"])
            print(f"  {label}: {A['res']}{A['resid']} {A['name']}--"
                  f"{B['name']} = {d:.4f} nm")

In [ ]:
from pathlib import Path

work = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")

for filename in ["em.mdp", "em2.mdp"]:
    path = work / filename
    print(f"\n===== {filename} =====")
    if path.exists():
        print(path.read_text())
    else:
        print("File not found")

print("\n===== Relevant lines from em.log =====")
log = work / "em.log"
if log.exists():
    for line in log.read_text(errors="replace").splitlines():
        if any(term in line.lower() for term in [
            "lincs", "constraint", "converged", "potential energy",
            "maximum force", "fmax", "step 43", "step 126"
        ]):
            print(line)
else:
    print("em.log not found")

In [ ]:
from pathlib import Path
import numpy as np
from scipy.spatial import cKDTree

gro = Path("/content/drive/MyDrive/Plant Protein Extrusion MD/em.gro")
lines = gro.read_text().splitlines()
n_atoms = int(lines[1].strip())

# Protein atoms appear before solvent/ions in this system.
protein = []
for atom_id, line in enumerate(lines[2:2+n_atoms], start=1):
    resname = line[5:10].strip()
    atomname = line[10:15].strip()
    if resname == "SOL" or atomname.startswith(("H", "1H", "2H", "3H")):
        continue
    protein.append((
        atom_id, int(line[0:5]), resname, atomname,
        np.array([float(line[20:28]),
                  float(line[28:36]),
                  float(line[36:44])])
    ))

coords = np.array([a[4] for a in protein])
tree = cKDTree(coords)
pairs = tree.query_pairs(r=0.18)

close = []
for i, j in pairs:
    a, b = protein[i], protein[j]
    # Ignore immediate sequence neighbours within the same numbered chain
    # only when residue numbers are adjacent.
    if a[1] == b[1] and abs(a[1] - b[1]) <= 1:
        continue
    d = np.linalg.norm(a[4] - b[4])
    close.append((d, a, b))

close.sort(key=lambda x: x[0])

print("Protein heavy atoms checked:", len(protein))
print("Close pairs under 0.18 nm:", len(close))
print("\nClosest 20 pairs:")
for d, a, b in close[:20]:
    print(
        f"{d:.3f} nm | atom {a[0]} {a[2]}{a[1]}-{a[3]} "
        f" <-> atom {b[0]} {b[2]}{b[1]}-{b[3]}"
    )

In [ ]:
from pathlib import Path

p = Path("/content/drive/MyDrive/Plant Protein Extrusion MD/em2.log")
lines = p.read_text(errors="replace").splitlines()

print("\n".join(lines[-100:]))

In [ ]:
from pathlib import Path

work = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")

print("Checking the actual settings stored in each TPR:\n")

for name in ["em", "em2"]:
    print(f"\n===== {name}.tpr =====")
    import subprocess
    result = subprocess.run(
        ["conda", "run", "-n", "gmx_cuda", "gmx", "dump",
         "-s", str(work / f"{name}.tpr")],
        capture_output=True, text=True
    )

    lines = result.stdout.splitlines()
    keys = ("constraints", "constraint_algorithm", "lincs_order",
            "lincs_iter", "emtol", "emstep", "integrator")

    found = False
    for line in lines:
        if any(k in line.lower() for k in keys):
            print(line)
            found = True

    if result.returncode != 0:
        print(result.stderr[-1500:])
    elif not found:
        print("No matching settings found in gmx dump output.")

In [ ]:
import subprocess
from pathlib import Path

work = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")

for name in ["em", "em2"]:
    r = subprocess.run(
        ["conda", "run", "-n", "gmx_cuda", "gmx", "dump",
         "-s", str(work / f"{name}.tpr")],
        capture_output=True, text=True
    )
    output = r.stdout + "\n" + r.stderr
    lines = output.splitlines()

    print(f"\n===== {name}.tpr =====")
    for i, line in enumerate(lines):
        if any(k in line.lower() for k in [
            "constraint-algorithm", "lincs-order", "lincs-iter",
            "constraints =", "nconstraints", "numconstraints"
        ]):
            print(line.strip())

In [ ]:
from pathlib import Path

work = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")

for name in ["em", "em2"]:
    path = work / f"{name}.log"
    print(f"\n===== {name}.log: final 35 lines =====")
    if path.exists():
        print("\n".join(path.read_text(errors="replace").splitlines()[-35:]))
    else:
        print("Log file not found:", path)

In [ ]:
from pathlib import Path

work = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")
gro = work / "em2.gro"

lines = gro.read_text().splitlines()
atom_line = lines[2 + 12174 - 1]
print("Atom 12174 in em2.gro:")
print(atom_line)

In [ ]:
from pathlib import Path

work = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")

print("Topology files:")
for p in sorted(work.rglob("*.itp")):
    print(p.relative_to(work))

print("\nMain topology exists:", (work / "topol.top").exists())

In [ ]:
from pathlib import Path

work = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")
p = work / "topol_Protein_chain_B.itp"
lines = p.read_text().splitlines()

for i, line in enumerate(lines):
    if line.strip().lower() == "[ atoms ]":
        print("Atoms section:")
        print("\n".join(f"{j+1}: {lines[j]}" for j in range(i, min(i+12, len(lines)))))
        break

print("\nASP294 atom entries:")
for i, line in enumerate(lines):
    fields = line.split()
    if len(fields) >= 6 and fields[3] == "ASP" and fields[4] == "294":
        print(f"Line {i+1}: {line}")

In [ ]:
from pathlib import Path

work = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")
p = work / "topol_Protein_chain_B.itp"
lines = p.read_text().splitlines()

for line in lines:
    fields = line.split()
    if len(fields) >= 8 and fields[0].isdigit() and int(fields[0]) == 4345:
        print("Chain B local atom 4345:")
        print(line)
        break
else:
    print("Atom 4345 not found in chain B topology.")

In [ ]:
from pathlib import Path

work = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")
p = work / "topol_Protein_chain_B.itp"
lines = p.read_text().splitlines()

section = None
print("Bonds involving ASP294 CG (local atom 4345):")

for line in lines:
    s = line.strip()

    if s.startswith("[") and s.endswith("]"):
        section = s.lower()
        continue

    if section == "[ bonds ]" and s and not s.startswith(";"):
        fields = s.split()
        if len(fields) >= 2 and fields[0].isdigit() and fields[1].isdigit():
            a, b = int(fields[0]), int(fields[1])
            if a == 4345 or b == 4345:
                print(line)

In [ ]:
from pathlib import Path
import re

work = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")

targets = {
    (14660, 14661),
    (43310, 43311),
    (43310, 43312),
}

print("Checking topology entries for LINCS-warning atom pairs...\n")

for p in sorted(work.glob("topol_Protein_chain_*.itp")):
    section = None
    found = []

    for line in p.read_text().splitlines():
        s = line.strip()

        if s.startswith("[") and "]" in s:
            section = re.match(r"\[\s*(\w+)", s).group(1).lower()
            continue

        if not s or s.startswith(";"):
            continue

        fields = s.split()
        if not fields or not fields[0].isdigit():
            continue

        counts = {
            "bonds": 2,
            "constraints": 3,
            "angles": 3,
            "dihedrals": 4,
        }

        n = counts.get(section)
        if n is None or len(fields) < n:
            continue

        try:
            atoms = tuple(map(int, fields[:n]))
        except ValueError:
            continue

        for a, b in targets:
            if section in ("bonds", "constraints") and set(atoms[:2]) == {a, b}:
                found.append((section, line.strip()))

    if found:
        print(p.name)
        for section, entry in found:
            print(f"  [{section}] {entry}")

if not any(work.glob("topol_Protein_chain_*.itp")):
    print("No chain topology files found. Check the working directory.")

In [ ]:
atoms_per_chain = 7829

targets = [14660, 14661, 43310, 43311, 43312]

for atom in targets:
    chain_index = (atom - 1) // atoms_per_chain
    local_index = (atom - 1) % atoms_per_chain + 1
    chain = chr(ord("A") + chain_index)

    print(f"Global atom {atom} -> Chain {chain}, local atom {local_index}")

In [ ]:
from pathlib import Path
import re

work = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")

targets = {
    "B": {6831, 6832},
    "F": {4165, 4166, 4167},
}

for chain, wanted in targets.items():
    p = work / f"topol_Protein_chain_{chain}.itp"
    lines = p.read_text().splitlines()

    print(f"\n{'='*18} CHAIN {chain} {'='*18}")

    section = None
    for line in lines:
        s = line.strip()

        if s.startswith("[") and "]" in s:
            m = re.match(r"\[\s*(\w+)", s)
            section = m.group(1).lower() if m else None
            continue

        if not s or s.startswith(";"):
            continue

        fields = s.split()
        if section == "atoms" and fields[0].isdigit() and len(fields) >= 5:
            idx = int(fields[0])
            if idx in wanted:
                print("ATOM: ", line.strip())

    section = None
    for line in lines:
        s = line.strip()

        if s.startswith("[") and "]" in s:
            m = re.match(r"\[\s*(\w+)", s)
            section = m.group(1).lower() if m else None
            continue

        if not s or s.startswith(";"):
            continue

        fields = s.split()
        if not fields or not fields[0].isdigit():
            continue

        n = {"bonds": 2, "constraints": 3, "angles": 3}.get(section)
        if n is None or len(fields) < n:
            continue

        try:
            indices = [int(x) for x in fields[:n]]
        except ValueError:
            continue

        if any(i in wanted for i in indices):
            print(f"[{section}] {line.strip()}")

In [ ]:
import subprocess
from pathlib import Path

work = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")

for name in ["em", "em2"]:
    print(f"\n{'='*15} {name}.tpr {'='*15}")

    result = subprocess.run(
        ["conda", "run", "-n", "gmx_cuda", "gmx", "dump",
         "-s", str(work / f"{name}.tpr")],
        capture_output=True, text=True
    )

    lines = result.stdout.splitlines()
    keywords = (
        "constraints =", "constraint-algorithm",
        "lincs-order", "lincs-iter",
        "nconstraints", "settles"
    )

    matches = [
        line.strip() for line in lines
        if any(key in line.lower() for key in keywords)
    ]

    print("\n".join(matches) if matches else "No matching settings found.")

In [ ]:
from pathlib import Path
import re

work = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")

for name in ["em", "em2"]:
    log = work / f"{name}.log"
    print(f"\n===== {name}.log =====")

    if not log.exists():
        print("Log file not found.")
        continue

    lines = log.read_text(errors="replace").splitlines()
    matches = [
        (i, line.strip())
        for i, line in enumerate(lines)
        if re.search(r"constraint|lincs|constraining", line, re.I)
    ]

    if matches:
        for i, line in matches[:25]:
            print(f"{i+1}: {line}")
    else:
        print("No constraint-related lines found.")

In [ ]:
from pathlib import Path
import subprocess
import re

work = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")

for name in ["em", "em2"]:
    result = subprocess.run(
        ["conda", "run", "-n", "gmx_cuda", "gmx", "dump",
         "-s", str(work / f"{name}.tpr")],
        capture_output=True, text=True
    )

    print(f"\n===== {name}.tpr =====")

    # Find the integrator and constraint-related settings
    for line in result.stdout.splitlines():
        if re.search(
            r"^\s*(integrator|constraints|constraint-algorithm|lincs-order|lincs-iter)\s*=",
            line,
            re.I
        ):
            print(line.strip())

    if result.returncode != 0:
        print("GROMACS dump error:", result.stderr[-500:])

In [ ]:
from pathlib import Path
import re

work = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")

for name in ["em", "em2"]:
    p = work / f"{name}.log"
    lines = p.read_text(errors="replace").splitlines()

    print(f"\n===== {name} =====")

    patterns = [
        r"The number of constraints is",
        r"Constraint error in algorithm",
        r"Steepest Descents converged",
        r"Maximum force",
        r"Potential Energy",
        r"stopped because",
        r"did not reach the requested",
    ]

    found = False
    for line in lines:
        if any(re.search(pattern, line, re.I) for pattern in patterns):
            print(line.strip())
            found = True

    if not found:
        print("No matching lines found.")

In [ ]:
from pathlib import Path

work = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")

for name in ["em.mdp", "em2.mdp"]:
    p = work / name
    print(f"\n===== {name} =====")

    for line in p.read_text().splitlines():
        s = line.strip()
        if s and not s.startswith(";"):
            print(s)

print("\n===== Topology files =====")
for name in ["topol.top", "topol_Protein_chain_B.itp",
             "topol_Protein_chain_F.itp"]:
    p = work / name
    print(f"{name}: {'FOUND' if p.exists() else 'NOT FOUND'}")

In [ ]:
from pathlib import Path

work = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")
p = work / "topol.top"

lines = p.read_text().splitlines()

print("===== Force-field and topology includes =====")
for line in lines:
    s = line.strip()
    if s.startswith("#include"):
        print(s)

print("\n===== Molecule definitions =====")
section = None
for line in lines:
    s = line.strip()

    if s.startswith("[") and "]" in s:
        section = s.lower()
        print(s)
        continue

    if section == "[ molecules ]" and s and not s.startswith(";"):
        print(s)

In [ ]:
from pathlib import Path
import re

work = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")

files = [
    work / "amber99sb-ildn.ff" / "forcefield.itp",
    work / "amber99sb-ildn.ff" / "aminoacids.rtp",
    work / "topol_Protein_chain_B.itp",
    work / "topol_Protein_chain_F.itp",
]

for p in files:
    print(f"\n===== {p.name} =====")

    if not p.exists():
        print("File not found.")
        continue

    lines = p.read_text(errors="replace").splitlines()

    if p.name == "forcefield.itp":
        for i, line in enumerate(lines):
            if re.search(r"constraints|constraint-algorithm", line, re.I):
                print(f"{i+1}: {line.strip()}")

    elif p.name == "aminoacids.rtp":
        print("Residue template file exists; no full template dump.")

    else:
        section = None
        for line in lines:
            s = line.strip()
            if s.startswith("[") and "]" in s:
                section = re.match(r"\[\s*(\w+)", s).group(1).lower()
            if section == "constraints" and s and not s.startswith(";"):
                print(s)

In [ ]:
import subprocess
from pathlib import Path

result = subprocess.run(
    ["conda", "run", "-n", "gmx_cuda", "gmx", "--version"],
    capture_output=True,
    text=True,
    check=True
)

print(result.stdout)

root = Path("/usr/local/envs/gmx_cuda/share/gromacs/top")
matches = list(root.glob("amber99sb-ildn.ff/forcefield.itp"))

print("\nForce-field path:")
for p in matches:
    print(p)

In [ ]:
from pathlib import Path

work = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")

for name in ["em", "em2"]:
    lines = (work / f"{name}.log").read_text(
        errors="replace"
    ).splitlines()

    print(f"\n===== {name}: constraint initialization =====")
    for i, line in enumerate(lines):
        if "Initializing LINear Constraint Solver" in line:
            print("\n".join(lines[i:i+12]))
            break
    else:
        print("No LINCS initialization block found.")

In [ ]:
from pathlib import Path

ff = Path(
    "/usr/local/envs/gmx_cuda/share/gromacs/top/amber99sb-ildn.ff"
)

for filename in ["forcefield.itp", "aminoacids.rtp"]:
    p = ff / filename
    print(f"\n===== {filename} =====")

    if not p.exists():
        print("File not found")
        continue

    lines = p.read_text(errors="replace").splitlines()

    if filename == "forcefield.itp":
        for i, line in enumerate(lines):
            if "constraints" in line.lower():
                print(f"{i+1}: {line.strip()}")
    else:
        print("Residue templates are available.")

In [ ]:
import subprocess
from pathlib import Path

work = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")

for name in ["em", "em2"]:
    print(f"\n===== {name}: TPR parameter summary =====")

    result = subprocess.run(
        ["conda", "run", "-n", "gmx_cuda", "gmx", "dump",
         "-s", str(work / f"{name}.tpr")],
        capture_output=True, text=True
    )

    lines = result.stdout.splitlines()
    for i, line in enumerate(lines):
        if "constraints" in line.lower():
            print(line.strip())
            for extra in lines[i+1:i+3]:
                print(extra.strip())

    if result.returncode != 0:
        print(result.stderr[-500:])

In [ ]:
import subprocess
from pathlib import Path

work = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")

for name in ["em", "em2"]:
    print(f"\n===== {name}: resolved MDP =====")

    result = subprocess.run(
        [
            "conda", "run", "-n", "gmx_cuda", "gmx", "grompp",
            "-f", str(work / f"{name}.mdp"),
            "-c", str(work / ("ions.gro" if name == "em" else "em.gro")),
            "-p", str(work / "topol.top"),
            "-pp", str(work / f"{name}_processed.top"),
            "-po", str(work / f"{name}_resolved.mdp"),
            "-o", str(work / f"{name}_check.tpr"),
            "-maxwarn", "5"
        ],
        capture_output=True, text=True
    )

    print(result.stdout[-2500:])
    print(result.stderr[-1500:])

    resolved = work / f"{name}_resolved.mdp"
    if resolved.exists():
        for line in resolved.read_text().splitlines():
            if line.strip().startswith("constraints"):
                print("Resolved:", line.strip())

In [ ]:
from pathlib import Path

work = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")

print("Coordinate files in project directory:\n")

for p in sorted(work.glob("*.gro")):
    print(f"{p.name:35s} {p.stat().st_size / 1e6:.2f} MB")

print("\nExisting minimization inputs:")
for name in ["em.tpr", "em2.tpr", "em.gro", "em2.gro"]:
    p = work / name
    print(f"{name:15s} {'FOUND' if p.exists() else 'NOT FOUND'}")

In [ ]:
from pathlib import Path

work = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")

for pattern in ["*.gro", "*.pdb", "*.tpr", "*.mdp", "*.cpt"]:
    print(f"\n===== {pattern} =====")
    files = list(work.rglob(pattern))
    if not files:
        print("None found")
    else:
        for p in files[:60]:
            print(p.relative_to(work))
        if len(files) > 60:
            print(f"... and {len(files)-60} more")

In [ ]:
from pathlib import Path
import subprocess

work = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")
coord = work / "structures/legumin_hexamer_ions.gro"

mdp_text = """integrator      = steep
emtol           = 1000.0
emstep          = 0.001
nsteps          = 50000
cutoff-scheme   = Verlet
nstlist         = 20
coulombtype     = PME
rcoulomb        = 1.0
rvdw            = 1.0
pbc             = xyz
constraints     = h-bonds
"""

mdp = work / "em3.mdp"
mdp.write_text(mdp_text)

cmd = [
    "conda", "run", "-n", "gmx_cuda", "gmx", "grompp",
    "-f", str(mdp),
    "-c", str(coord),
    "-p", str(work / "topol.top"),
    "-o", str(work / "em3.tpr"),
    "-po", str(work / "em3_resolved.mdp"),
]

result = subprocess.run(cmd, capture_output=True, text=True)

print(result.stdout[-3000:])
if result.stderr:
    print(result.stderr[-2000:])

print("\ngrompp exit code:", result.returncode)
print("em3.tpr created:", (work / "em3.tpr").exists())

In [ ]:
import subprocess
from pathlib import Path

work = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")

cmd = [
    "conda", "run", "-n", "gmx_cuda", "gmx", "mdrun",
    "-deffnm", str(work / "em3"),
    "-ntmpi", "1",
    "-ntomp", "2",
    "-pin", "off"
]

print("Starting EM3 energy minimization...\n", flush=True)

process = subprocess.Popen(
    cmd,
    cwd=str(work),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in process.stdout:
    print(line, end="", flush=True)

return_code = process.wait()
print(f"\nEM3 exit code: {return_code}")

In [ ]:
from pathlib import Path

work = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")

# Map global GROMACS atom numbers to atom/residue records
gro = work / "structures/legumin_hexamer_ions.gro"

if not gro.exists():
    raise FileNotFoundError(f"Missing coordinate file: {gro}")

lines = gro.read_text().splitlines()
atoms = []

for line in lines[2:-1]:
    if len(line) < 44:
        continue
    try:
        atoms.append({
            "resid": line[0:5].strip(),
            "resname": line[5:10].strip(),
            "atomname": line[10:15].strip(),
            "atomnr": line[15:20].strip(),
        })
    except Exception:
        pass

print("Atoms in starting GRO:", len(atoms))
print("Warning pairs to map:")
pairs = [
    (27597, 27599),
    (42925, 42926),
    (35990, 35991),
    (28879, 28880),
    (13251, 13252),
    (4568, 4569),
    (12174, 12174),
]

for a, b in pairs:
    print(f"\nPair {a}-{b}:")
    for n in sorted(set([a, b])):
        if 1 <= n <= len(atoms):
            print(n, atoms[n-1])
        else:
            print(n, "outside atom range")

In [ ]:
import subprocess
from pathlib import Path

work = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")

cmd = [
    "conda", "run", "-n", "gmx_cuda", "gmx",
    "dump", "-s", str(work / "em3.tpr")
]

result = subprocess.run(
    cmd,
    cwd=str(work),
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print(result.stderr)
else:
    lines = result.stdout.splitlines()
    keywords = [
        "integrator", "emtol", "emstep", "nsteps",
        "constraints", "constraint_algorithm",
        "lincs_order", "lincs_iter", "continuation",
        "coulombtype", "rcoulomb", "rvdw",
        "pbc", "define"
    ]

    for i, line in enumerate(lines):
        if any(key in line.lower() for key in keywords):
            print(line)

In [ ]:
from pathlib import Path
import re

work = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")
log = work / "em3.log"

if not log.exists():
    print("em3.log not found:", log)
else:
    lines = log.read_text(errors="replace").splitlines()

    print("=== First LINCS warnings ===")
    found = 0
    for i, line in enumerate(lines):
        if "LINCS WARNING" in line or "relative constraint deviation" in line:
            start = max(0, i - 1)
            end = min(len(lines), i + 5)
            print("\n".join(lines[start:end]))
            print("---")
            found += 1
            if found >= 3:
                break

    print("\n=== Minimization summary ===")
    for line in lines:
        if any(term in line for term in [
            "converged to Fmax",
            "Potential Energy",
            "Maximum force",
            "Norm of force",
            "constraint"
        ]):
            print(line)

In [ ]:
import subprocess
from pathlib import Path

work = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")

for name in ["em2", "em3"]:
    print(f"\n===== {name.upper()} =====")

    result = subprocess.run(
        [
            "conda", "run", "-n", "gmx_cuda", "gmx",
            "dump", "-s", str(work / f"{name}.tpr")
        ],
        capture_output=True,
        text=True
    )

    if result.returncode != 0:
        print(result.stderr)
        continue

    for line in result.stdout.splitlines():
        if any(key in line.lower() for key in [
            "constraint-algorithm",
            "constraints",
            "lincs-order",
            "lincs-iter",
            "emtol",
            "emstep",
            "nsteps"
        ]):
            print(line)

In [ ]:
from pathlib import Path
import subprocess

work = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")
gro = work / "em3.gro"

print("Final EM3 structure exists:", gro.exists())

if gro.exists():
    print("File size:", round(gro.stat().st_size / 1024 / 1024, 2), "MB")

    result = subprocess.run(
        [
            "conda", "run", "-n", "gmx_cuda", "gmx",
            "check", "-f", str(gro)
        ],
        cwd=str(work),
        capture_output=True,
        text=True
    )

    print(result.stdout[-5000:])
    if result.stderr:
        print(result.stderr[-2000:])

In [ ]:
from pathlib import Path
import numpy as np

work = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")

def read_gro(path):
    lines = path.read_text().splitlines()
    n_atoms = int(lines[1].strip())
    xyz = np.array([
        [float(line[20:28]), float(line[28:36]), float(line[36:44])]
        for line in lines[2:2+n_atoms]
    ])
    return lines, xyz, n_atoms

lines, xyz, n_atoms = read_gro(work / "em3.gro")
box = np.array([float(x) for x in lines[2+n_atoms].split()[:3]])

pairs = [
    (27597, 27599, 0.1090, "GLN280 CB-HB2"),
    (42925, 42926, 0.1010, "GLU260 N-H"),
    (35990, 35991, 0.1010, "ASP314 N-H"),
    (28879, 28880, 0.1090, "ARG362 CB-HB1"),
    (13251, 13252, 0.1090, "ILE363 CG1-HG11"),
    (4568, 4569, 0.1010, "ARG307 N-H"),
]

print(f"Atoms read: {n_atoms}")
print("\nFinal EM3 bond distances (minimum-image PBC):")

for a, b, expected, label in pairs:
    delta = xyz[a-1] - xyz[b-1]
    delta -= box * np.round(delta / box)
    distance = np.linalg.norm(delta)
    print(
        f"{label:22s} atoms {a}-{b}: "
        f"{distance:.4f} nm; reference {expected:.4f} nm"
    )

In [ ]:
from pathlib import Path
import numpy as np

work = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")
top = (work / "topol_Protein_chain_B.itp").read_text().splitlines()

# Locate the atom and bond sections in chain B's topology
section = None
atom_record = None
bond_records = []

for line in top:
    s = line.strip()

    if s.startswith("[") and s.endswith("]"):
        section = s.lower()
        continue

    if not s or s.startswith(";"):
        continue

    fields = s.split()

    if section == "[ atoms ]" and len(fields) >= 5:
        try:
            if int(fields[0]) == 4345:
                atom_record = fields
        except ValueError:
            pass

    elif section == "[ bonds ]" and len(fields) >= 2:
        try:
            i, j = int(fields[0]), int(fields[1])
            if i == 4345 or j == 4345:
                bond_records.append((i, j, fields))
        except ValueError:
            pass

print("Atom 4345 in chain B topology:")
print(atom_record)
print("\nBond entries involving atom 4345:")
for record in bond_records:
    print(record)

# Coordinates around global atom 12174 in the final GRO
lines = (work / "em3.gro").read_text().splitlines()
n_atoms = int(lines[1])
for global_id in [12174, 12173, 12175, 12176, 12177]:
    if global_id <= n_atoms:
        print(f"\nGRO atom {global_id}: {lines[global_id + 1]}")

In [ ]:
from pathlib import Path
import numpy as np

work = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")
gro_lines = (work / "em3.gro").read_text().splitlines()
n_atoms = int(gro_lines[1])

coords = np.array([
    [float(line[20:28]), float(line[28:36]), float(line[36:44])]
    for line in gro_lines[2:2+n_atoms]
])

box = np.array([
    float(x) for x in gro_lines[2+n_atoms].split()[:3]
])

global_cg = 12174
global_neighbours = [12171, 12175, 12176]

labels = {
    12171: "bonded neighbour (local 4342)",
    12175: "bonded neighbour (local 4346)",
    12176: "bonded neighbour (local 4347)",
}

print("Distances from ASP294 CG, final EM3 structure:")
for atom in global_neighbours:
    delta = coords[global_cg-1] - coords[atom-1]
    delta -= box * np.round(delta / box)
    distance = np.linalg.norm(delta)
    print(f"{labels[atom]}: global {atom}, distance {distance:.4f} nm")

In [ ]:
import subprocess
from pathlib import Path

work = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")

result = subprocess.run(
    [
        "conda", "run", "-n", "gmx_cuda", "gmx",
        "energy",
        "-f", str(work / "em3.edr"),
        "-o", str(work / "em3_potential.xvg")
    ],
    cwd=str(work),
    input="Potential\n0\n",
    capture_output=True,
    text=True
)

print(result.stdout[-4000:])
if result.stderr:
    print(result.stderr[-2000:])

xvg = work / "em3_potential.xvg"
if xvg.exists():
    print("\n=== First and last energy samples ===")
    data = [
        line.strip() for line in xvg.read_text().splitlines()
        if line.strip() and not line.startswith(("#", "@"))
    ]
    print("\n".join(data[:5]))
    print("...")
    print("\n".join(data[-5:]))

In [ ]:
import subprocess
from pathlib import Path

work = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")

result = subprocess.run(
    [
        "conda", "run", "-n", "gmx_cuda", "gmx",
        "energy",
        "-f", str(work / "em3.edr"),
        "-o", str(work / "em3_potential.xvg")
    ],
    cwd=str(work),
    input="10\n0\n",
    capture_output=True,
    text=True
)

print(result.stdout[-2500:])
if result.stderr:
    print(result.stderr[-1500:])

xvg = work / "em3_potential.xvg"

if xvg.exists():
    data = [
        line.strip()
        for line in xvg.read_text().splitlines()
        if line.strip() and not line.startswith(("#", "@"))
    ]

    print("\nEnergy samples at the start:")
    print("\n".join(data[:5]))
    print("\nEnergy samples at the end:")
    print("\n".join(data[-5:]))
else:
    print("\nEnergy output file was not created.")

In [ ]:
%cd "/content/drive/MyDrive/Plant Protein Extrusion MD"
!conda run --no-capture-output -n gmx_cuda gmx energy -f em3.edr -o em3_potential.xvg

In [ ]:
from pathlib import Path
import numpy as np

work = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")
xvg = work / "em3_potential.xvg"

data = []
for line in xvg.read_text().splitlines():
    line = line.strip()
    if not line or line.startswith(("#", "@")):
        continue
    values = line.split()
    if len(values) >= 2:
        data.append([float(values[0]), float(values[1])])

data = np.array(data)
time = data[:, 0]
energy = data[:, 1]

print("Number of samples:", len(energy))
print(f"First energy: {energy[0]:.6e} kJ/mol")
print(f"Last energy:  {energy[-1]:.6e} kJ/mol")
print(f"Minimum:      {energy.min():.6e} kJ/mol")
print(f"Maximum:      {energy.max():.6e} kJ/mol")
print(f"Median:       {np.median(energy):.6e} kJ/mol")

print("\n10 largest energy samples:")
for i in np.argsort(energy)[-10:][::-1]:
    print(f"time/step={time[i]:.0f}, potential={energy[i]:.6e} kJ/mol")

In [ ]:
from pathlib import Path
import numpy as np
from scipy.spatial import cKDTree

work = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")

def read_gro(path):
    lines = path.read_text().splitlines()
    n = int(lines[1])
    xyz = np.array([
        [float(line[20:28]), float(line[28:36]), float(line[36:44])]
        for line in lines[2:2+n]
    ])
    return xyz, n

start_path = work / "structures/legumin_hexamer_ions.gro"
final_path = work / "em3.gro"

start, n_start = read_gro(start_path)
final, n_final = read_gro(final_path)

print("Starting atoms:", n_start)
print("Final atoms:", n_final)

# Detect extremely close non-identical atom positions in the starting structure.
# This is a preliminary geometric screen, not a full bonded-aware clash analysis.
tree = cKDTree(start)
pairs = tree.query_pairs(r=0.05)  # 0.05 nm = 0.5 Å

print("Starting coordinate pairs within 0.05 nm:", len(pairs))

if pairs:
    distances = []
    for i, j in pairs:
        d = np.linalg.norm(start[i] - start[j])
        distances.append((d, i+1, j+1))
    for d, i, j in sorted(distances)[:20]:
        print(f"Atoms {i}-{j}: {d:.4f} nm")

In [ ]:
from pathlib import Path
import numpy as np
from scipy.spatial import cKDTree

work = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")
path = work / "structures/legumin_hexamer_ions.gro"
lines = path.read_text().splitlines()
n = int(lines[1])

atoms = []
xyz = []

for line in lines[2:2+n]:
    atoms.append({
        "resid": line[0:5].strip(),
        "resname": line[5:10].strip(),
        "atomname": line[10:15].strip(),
        "serial": line[15:20].strip()
    })
    xyz.append([
        float(line[20:28]),
        float(line[28:36]),
        float(line[36:44])
    ])

xyz = np.asarray(xyz)
tree = cKDTree(xyz)
pairs = tree.query_pairs(r=0.05)

close = []
for i, j in pairs:
    d = np.linalg.norm(xyz[i] - xyz[j])
    close.append((d, i, j))

for d, i, j in sorted(close)[:10]:
    print(f"\nDistance: {d:.4f} nm")
    print(f"Atom {i+1}: {atoms[i]}")
    print(f"Atom {j+1}: {atoms[j]}")

## 4. Legumin 300 K: ions, NVT, NPT and production (r1 and r2)
Re-adds the ions with `genion -neutral -conc 0.15` and fixes the topology counts, then runs NVT (100 ps), NPT (500 ps) and 1 ns production. r1 uses seed 111, r2 uses 222.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd "/content/drive/MyDrive/Plant Protein Extrusion MD"
!ls

In [ ]:
import os
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin.AVX2_256:/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
%cd "/content/drive/MyDrive/Plant Protein Extrusion MD"

In [ ]:
%%bash
SEED=111; T=300; TAG=nvt_${T}_r1

cat > ${TAG}.mdp << EOF
define        = -DPOSRES
integrator    = md
dt            = 0.002
nsteps        = 50000          ; 100 ps
continuation  = no
constraints   = h-bonds
constraint-algorithm = lincs
cutoff-scheme = Verlet
coulombtype   = PME
rcoulomb      = 1.0
rvdw          = 1.0
DispCorr      = EnerPres
tcoupl        = V-rescale
tc-grps       = Protein Non-Protein
tau-t         = 0.1 0.1
ref-t         = ${T} ${T}
pcoupl        = no
gen-vel       = yes
gen-temp      = ${T}
gen-seed      = ${SEED}
nstenergy     = 500
nstlog        = 500
nstxout-compressed = 5000
EOF

gmx grompp -f ${TAG}.mdp -c em.gro -r em.gro -p topol.top -o ${TAG}.tpr
gmx mdrun -deffnm ${TAG} -ntmpi 1 -ntomp 2 -pin off

In [ ]:
!tail -n 8 topol.top

In [ ]:
%%bash
set -e
cp topol.top topol_wrong_ions.bak
sed -i '/^NA /d' topol.top
sed -i 's/^SOL .*/SOL         57108/' topol.top

cat > ions.mdp << EOF
integrator = steep
nsteps     = 0
cutoff-scheme = Verlet
coulombtype = cutoff
rcoulomb   = 1.0
rvdw       = 1.0
pbc        = xyz
continuation = yes
EOF

gmx grompp -f ions.mdp -c structures/legumin_hexamer_solv.gro -p topol.top -o ions.tpr -maxwarn 1
echo SOL | gmx genion -s ions.tpr -o structures/legumin_hexamer_ions.gro -p topol.top -pname NA -nname CL -neutral -conc 0.15
tail -n 8 topol.top
sed -n 2p structures/legumin_hexamer_ions.gro

In [ ]:
%%bash
set -e
T=300; R=1; SEED=111
GRO=$(ls structures/*_ions.gro | head -n 1)
echo "Using $GRO"

# EM
gmx grompp -f em.mdp -c $GRO -p topol.top -o em.tpr
gmx mdrun -deffnm em -ntmpi 1 -ntomp 2 -pin off
grep "converged to Fmax" em.log

# NVT (100 ps)
TAG=nvt_${T}_r${R}
cat > ${TAG}.mdp << EOF
define = -DPOSRES
integrator = md
dt = 0.002
nsteps = 50000
continuation = no
constraints = h-bonds
constraint-algorithm = lincs
cutoff-scheme = Verlet
coulombtype = PME
rcoulomb = 1.0
rvdw = 1.0
DispCorr = EnerPres
tcoupl = V-rescale
tc-grps = Protein Non-Protein
tau-t = 0.1 0.1
ref-t = ${T} ${T}
pcoupl = no
gen-vel = yes
gen-temp = ${T}
gen-seed = ${SEED}
nstenergy = 500
nstlog = 500
nstxout-compressed = 5000
EOF
gmx grompp -f ${TAG}.mdp -c em.gro -r em.gro -p topol.top -o ${TAG}.tpr
gmx mdrun -deffnm ${TAG} -ntmpi 1 -ntomp 2 -pin off

# NPT (500 ps)
PREV=${TAG}; TAG=npt_${T}_r${R}
sed -e 's/^nsteps.*/nsteps = 250000/' \
    -e 's/^continuation.*/continuation = yes/' \
    -e 's/^pcoupl .*/pcoupl = C-rescale\npcoupltype = isotropic\ntau-p = 2.0\ncompressibility = 4.5e-5\nref-p = 1.0/' \
    -e 's/^gen-vel.*/gen-vel = no/' \
    -e '/^gen-temp/d' -e '/^gen-seed/d' ${PREV}.mdp > ${TAG}.mdp
gmx grompp -f ${TAG}.mdp -c ${PREV}.gro -r ${PREV}.gro -t ${PREV}.cpt -p topol.top -o ${TAG}.tpr
gmx mdrun -deffnm ${TAG} -ntmpi 1 -ntomp 2 -pin off

In [ ]:
%%bash
set -e
T=300; R=1
PREV=nvt_${T}_r${R}; TAG=npt_${T}_r${R}

grep -q refcoord ${TAG}.mdp || echo "refcoord-scaling = com" >> ${TAG}.mdp

gmx grompp -f ${TAG}.mdp -c ${PREV}.gro -r ${PREV}.gro -t ${PREV}.cpt -p topol.top -o ${TAG}.tpr
gmx mdrun -deffnm ${TAG} -ntmpi 1 -ntomp 2 -pin off

In [ ]:
%%bash
tail -n 6 npt_300_r1.log
awk '/Step           Time/{getline; s=$1} END{printf "last step: %d / 250000\n", s}' npt_300_r1.log
tail -n 1 npt_300_r1.gro

In [ ]:
%%bash
echo "Density" | gmx energy -f npt_300_r1.edr -o npt_300_r1_density.xvg | tail -n 4
echo "Temperature" | gmx energy -f npt_300_r1.edr | tail -n 4
echo "Potential" | gmx energy -f npt_300_r1.edr | tail -n 4
echo "Pressure" | gmx energy -f npt_300_r1.edr | tail -n 4

In [ ]:
%%bash
set -e
T=300; R=1; SEED=111
PREV=npt_${T}_r${R}; TAG=md_${T}_r${R}

cat > ${TAG}.mdp << EOF
integrator = md
dt = 0.002
nsteps = 500000
continuation = yes
constraints = h-bonds
constraint-algorithm = lincs
cutoff-scheme = Verlet
coulombtype = PME
rcoulomb = 1.0
rvdw = 1.0
DispCorr = EnerPres
tcoupl = V-rescale
tc-grps = Protein Non-Protein
tau-t = 0.1 0.1
ref-t = ${T} ${T}
pcoupl = no
gen-vel = no
ld-seed = ${SEED}
nstenergy = 500
nstlog = 5000
nstxout-compressed = 500
compressed-x-grps = Protein
EOF

gmx grompp -f ${TAG}.mdp -c ${PREV}.gro -t ${PREV}.cpt -p topol.top -o ${TAG}.tpr
gmx mdrun -deffnm ${TAG} -cpt 5 -ntmpi 1 -ntomp 2 -pin off

In [ ]:
%%bash
tail -n 6 md_300_r1.log
awk '/Step           Time/{getline; s=$1} END{printf "last step: %d / 500000\n", s}' md_300_r1.log
grep -c "LINCS WARNING" md_300_r1.log || true
echo "Temperature" | gmx energy -f md_300_r1.edr 2>/dev/null | grep -E "^Temperature"
echo "Potential" | gmx energy -f md_300_r1.edr 2>/dev/null | grep -E "^Potential"
echo "Pressure" | gmx energy -f md_300_r1.edr 2>/dev/null | grep -E "^Pressure"

In [ ]:
%%bash
set -e
T=300; R=2
SEED=$((R*111))
echo "T=$T rep=$R seed=$SEED"

COMMON="integrator = md
dt = 0.002
constraints = h-bonds
constraint-algorithm = lincs
cutoff-scheme = Verlet
coulombtype = PME
rcoulomb = 1.0
rvdw = 1.0
DispCorr = EnerPres
tcoupl = V-rescale
tc-grps = Protein Non-Protein
tau-t = 0.1 0.1
ref-t = ${T} ${T}"

# NVT 100 ps
TAG=nvt_${T}_r${R}
cat > ${TAG}.mdp << EOF
define = -DPOSRES
${COMMON}
nsteps = 50000
continuation = no
pcoupl = no
gen-vel = yes
gen-temp = ${T}
gen-seed = ${SEED}
nstenergy = 500
nstlog = 500
nstxout-compressed = 5000
EOF
gmx grompp -f ${TAG}.mdp -c em.gro -r em.gro -p topol.top -o ${TAG}.tpr
gmx mdrun -deffnm ${TAG} -cpt 5 -ntmpi 1 -ntomp 2 -pin off

# NPT 500 ps
PREV=${TAG}; TAG=npt_${T}_r${R}
cat > ${TAG}.mdp << EOF
define = -DPOSRES
${COMMON}
nsteps = 250000
continuation = yes
pcoupl = C-rescale
pcoupltype = isotropic
tau-p = 2.0
compressibility = 4.5e-5
ref-p = 1.0
refcoord-scaling = com
gen-vel = no
nstenergy = 500
nstlog = 500
nstxout-compressed = 5000
EOF
gmx grompp -f ${TAG}.mdp -c ${PREV}.gro -r ${PREV}.gro -t ${PREV}.cpt -p topol.top -o ${TAG}.tpr
gmx mdrun -deffnm ${TAG} -cpt 5 -ntmpi 1 -ntomp 2 -pin off

# Production 1 ns (NVT, no restraints)
PREV=${TAG}; TAG=md_${T}_r${R}
cat > ${TAG}.mdp << EOF
${COMMON}
nsteps = 500000
continuation = yes
pcoupl = no
gen-vel = no
ld-seed = ${SEED}
nstenergy = 500
nstlog = 5000
nstxout-compressed = 500
compressed-x-grps = Protein
EOF
gmx grompp -f ${TAG}.mdp -c ${PREV}.gro -t ${PREV}.cpt -p topol.top -o ${TAG}.tpr
gmx mdrun -deffnm ${TAG} -cpt 5 -ntmpi 1 -ntomp 2 -pin off

In [ ]:
%%bash
set -e
T=300; R=2
mkdir -p old_npt_${T}_r${R}_partial
for f in log xtc cpt edr gro; do
  if [ -f npt_${T}_r${R}.$f ]; then mv npt_${T}_r${R}.$f old_npt_${T}_r${R}_partial/; fi
done
if [ -f npt_${T}_r${R}_prev.cpt ]; then mv npt_${T}_r${R}_prev.cpt old_npt_${T}_r${R}_partial/; fi

gmx grompp -f npt_${T}_r${R}.mdp -c nvt_${T}_r${R}.gro -r nvt_${T}_r${R}.gro -t nvt_${T}_r${R}.cpt -p topol.top -o npt_${T}_r${R}.tpr
gmx mdrun -deffnm npt_${T}_r${R} -cpt 5 -ntmpi 1 -ntomp 2 -pin off

In [ ]:
%%bash
awk '/Step           Time/{getline; s=$1} END{print "last step:", s, "/ 250000"}' npt_300_r2.log
tail -n 2 npt_300_r2.log
for term in Density Temperature Potential Pressure; do
  echo $term | gmx energy -f npt_300_r2.edr 2>/dev/null | grep -E "^$term"
done
tail -n 1 npt_300_r2.gro

In [ ]:
%%bash
set -e
T=300; R=2; SEED=222
PREV=npt_${T}_r${R}; TAG=md_${T}_r${R}

cat > ${TAG}.mdp << EOF
integrator = md
dt = 0.002
nsteps = 500000
continuation = yes
constraints = h-bonds
constraint-algorithm = lincs
cutoff-scheme = Verlet
coulombtype = PME
rcoulomb = 1.0
rvdw = 1.0
DispCorr = EnerPres
tcoupl = V-rescale
tc-grps = Protein Non-Protein
tau-t = 0.1 0.1
ref-t = ${T} ${T}
pcoupl = no
gen-vel = no
ld-seed = ${SEED}
nstenergy = 500
nstlog = 5000
nstxout-compressed = 500
compressed-x-grps = Protein
EOF

gmx grompp -f ${TAG}.mdp -c ${PREV}.gro -t ${PREV}.cpt -p topol.top -o ${TAG}.tpr
gmx mdrun -deffnm ${TAG} -cpt 5 -ntmpi 1 -ntomp 2 -pin off

In [ ]:
%%bash
tail -n 6 md_300_r2.log
awk '/Step           Time/{getline; s=$1} END{printf "last step: %d / 500000\n", s}' md_300_r2.log
grep -c "LINCS WARNING" md_300_r2.log || true
for term in Temperature Potential Pressure; do
  echo $term | gmx energy -f md_300_r2.edr 2>/dev/null | grep -E "^$term"
done
ls -l md_300_r2.*

## 5. First analysis steps and run checks (300 K r1)
Trajectory preparation and a test `grompp` for the 300 K r1 run. Cell 128 puts the CPU build (`gmx_cpu`) on the path; the later cells use `gmx_cuda`.

In [ ]:
import os
os.environ["PATH"] = "/usr/local/envs/gmx_cpu/bin.AVX2_256:/usr/local/envs/gmx_cpu/bin:" + os.environ["PATH"]
from google.colab import drive
drive.mount('/content/drive')
!gmx --version | head -n 4
!find /content/drive/MyDrive -name "md_300_r1.xtc" 2>/dev/null

In [ ]:
os.environ["BASE"] = "/content/drive/MyDrive/Plant Protein Extrusion MD"
os.environ["RUN"]  = "md_300_r1"
!mkdir -p "$BASE/analysis/$RUN"
!ls -l "$BASE"/$RUN.* "$BASE"/structures/*processed.grov

In [ ]:
%%bash
cd "$BASE/analysis/$RUN"
echo Protein | gmx convert-tpr -s "$BASE/$RUN.tpr" -o protein.tpr 2>&1 | tail -n 3
echo System | gmx mindist -s protein.tpr -f "$BASE/$RUN.xtc" -pi -od mindist_pi.xvg -tu ps 2>&1 | tail -n 5

In [ ]:
BASE = "/content/drive/MyDrive/Plant Protein Extrusion MD"
RUN  = "md_300_r1"          # change to md_300_r2 for replicate 2
os.environ["BASE"], os.environ["RUN"] = BASE, RUN
!mkdir -p "$BASE/analysis/$RUN"
!ls -l "$BASE"/$RUN.* "$BASE"/structures/legumin_hexamer_processed.gro

In [ ]:
%%bash
cd "$BASE/analysis/$RUN"
echo Protein | gmx convert-tpr -s "$BASE/$RUN.tpr" -o protein.tpr > tpr.log 2>&1
echo System | gmx mindist -s protein.tpr -f "$BASE/$RUN.xtc" -pi -od mindist_pi.xvg -tu ps > mindist.log 2>&1
tail -n 4 mindist.log; ls -l protein.tpr mindist_pi.xvg

In [ ]:
%%bash
cd "$BASE/analysis/$RUN"
echo System | gmx trjconv -s protein.tpr -f "$BASE/$RUN.xtc" -o whole.xtc -pbc whole > t1.log 2>&1
printf "System\nSystem\n" | gmx trjconv -s protein.tpr -f whole.xtc -o cluster.xtc -pbc cluster > t2.log 2>&1
printf "System\nSystem\n" | gmx trjconv -s protein.tpr -f cluster.xtc -o centered.xtc -center -pbc mol -ur compact > t3.log 2>&1
printf "Backbone\nSystem\n" | gmx trjconv -s protein.tpr -f centered.xtc -o fit.xtc -fit rot+trans > t4.log 2>&1
ls -l *.xtc

In [ ]:
!gmx --version
!grep -n 'ld-seed' md_300_r1.mdp

In [ ]:
%%bash
cd "/content/drive/MyDrive/Plant Protein Extrusion MD"

gmx grompp \
  -f md_300_r1.mdp \
  -c md_300_r1.gro \
  -t md_300_r1.cpt \
  -p topol.top \
  -o /tmp/test_300_r1.tpr

## 6. Legumin 373 K, replicate 1
NVT, NPT and production at 373 K (r1), including the restart of an interrupted NPT run from its checkpoint.

In [ ]:
%%bash
set -e
T=373; R=1
SEED=$((R*111))
echo "T=$T rep=$R seed=$SEED"

COMMON="integrator = md
dt = 0.002
constraints = h-bonds
constraint-algorithm = lincs
cutoff-scheme = Verlet
coulombtype = PME
rcoulomb = 1.0
rvdw = 1.0
DispCorr = EnerPres
tcoupl = V-rescale
tc-grps = Protein Non-Protein
tau-t = 0.1 0.1
ref-t = ${T} ${T}"

# NVT 100 ps
TAG=nvt_${T}_r${R}
cat > ${TAG}.mdp << EOF
define = -DPOSRES
${COMMON}
nsteps = 50000
continuation = no
pcoupl = no
gen-vel = yes
gen-temp = ${T}
gen-seed = ${SEED}
nstenergy = 500
nstlog = 500
nstxout-compressed = 5000
EOF
gmx grompp -f ${TAG}.mdp -c em.gro -r em.gro -p topol.top -o ${TAG}.tpr
gmx mdrun -deffnm ${TAG} -cpt 5 -ntmpi 1 -ntomp 2 -pin off

# NPT 500 ps
PREV=${TAG}; TAG=npt_${T}_r${R}
cat > ${TAG}.mdp << EOF
define = -DPOSRES
${COMMON}
nsteps = 250000
continuation = yes
pcoupl = C-rescale
pcoupltype = isotropic
tau-p = 2.0
compressibility = 4.5e-5
ref-p = 1.0
refcoord-scaling = com
gen-vel = no
nstenergy = 500
nstlog = 500
nstxout-compressed = 5000
EOF
gmx grompp -f ${TAG}.mdp -c ${PREV}.gro -r ${PREV}.gro -t ${PREV}.cpt -p topol.top -o ${TAG}.tpr
gmx mdrun -deffnm ${TAG} -cpt 5 -ntmpi 1 -ntomp 2 -pin off

# Production 1 ns (NVT, no restraints)
PREV=${TAG}; TAG=md_${T}_r${R}
cat > ${TAG}.mdp << EOF
${COMMON}
nsteps = 500000
continuation = yes
pcoupl = no
gen-vel = no
ld-seed = ${SEED}
nstenergy = 500
nstlog = 5000
nstxout-compressed = 500
compressed-x-grps = Protein
EOF
gmx grompp -f ${TAG}.mdp -c ${PREV}.gro -t ${PREV}.cpt -p topol.top -o ${TAG}.tpr
gmx mdrun -deffnm ${TAG} -cpt 5 -ntmpi 1 -ntomp 2 -pin off

In [ ]:
%cd "/content/drive/MyDrive/Plant Protein Extrusion MD"
!ls -lh npt_373_r1.*
!tail -n 25 npt_373_r1.log

In [ ]:
%cd "/content/drive/MyDrive/Plant Protein Extrusion MD"

!ls -lh npt_373_r1*
!gmx check -f npt_373_r1.edr

In [ ]:
!gmx dump -cp npt_373_r1.cpt 2>/dev/null | grep -E 'step =|time ='

In [ ]:
!gmx mdrun -deffnm npt_373_r1 \
  -cpi npt_373_r1.cpt \
  -noappend \
  -cpt 5 \
  -ntmpi 1 -ntomp 2 -pin off

In [ ]:
!cd "/content/drive/MyDrive/Plant Protein Extrusion MD"

!ls -lh npt_373_r1.gro npt_373_r1.cpt npt_373_r1.tpr

In [ ]:
!cd "/content/drive/MyDrive/Plant Protein Extrusion MD"

!ls -lht npt_373_r1*

In [ ]:
!gmx grompp -f md_373_r1.mdp \
  -c npt_373_r1.gro \
  -t npt_373_r1.cpt \
  -p topol.top \
  -o md_373_r1.tpr

In [ ]:
!cd "/content/drive/MyDrive/Plant Protein Extrusion MD"

!tail -n 5 npt_373_r1.part0002.gro

In [ ]:
!tail -n 35 npt_373_r1.part0002.log

In [ ]:
!cd "/content/drive/MyDrive/Plant Protein Extrusion MD"

!head -n 2 npt_373_r1.part0002.gro

In [ ]:
!head -n 2 em.gro
!head -n 2 npt_373_r1.part0002.gro

In [ ]:
!gmx grompp -f md_373_r1.mdp \
  -c npt_373_r1.part0002.gro \
  -t npt_373_r1.cpt \
  -p topol.top \
  -o md_373_r1.tpr

In [ ]:
!cd "/content/drive/MyDrive/Plant Protein Extrusion MD" && printf '%s\n' \
'integrator = md' \
'dt = 0.002' \
'constraints = h-bonds' \
'constraint-algorithm = lincs' \
'cutoff-scheme = Verlet' \
'coulombtype = PME' \
'rcoulomb = 1.0' \
'rvdw = 1.0' \
'DispCorr = EnerPres' \
'tcoupl = V-rescale' \
'tc-grps = Protein Non-Protein' \
'tau-t = 0.1 0.1' \
'ref-t = 373 373' \
'nsteps = 500000' \
'continuation = yes' \
'pcoupl = no' \
'gen-vel = no' \
'ld-seed = 111' \
'nstenergy = 500' \
'nstlog = 5000' \
'nstxout-compressed = 500' \
'compressed-x-grps = Protein' > md_373_r1.mdp

In [ ]:
!ls -lh md_373_r1.mdp

In [ ]:
!gmx grompp -f md_373_r1.mdp -c npt_373_r1.part0002.gro -t npt_373_r1.cpt -p topol.top -o md_373_r1.tpr

In [ ]:
!gmx mdrun -deffnm md_373_r1 \
  -cpt 5 -ntmpi 1 -ntomp 2 -pin off

In [ ]:
import os

folder = "/content/drive/MyDrive/Plant Protein Extrusion MD"

for name in sorted(os.listdir(folder)):
    path = os.path.join(folder, name)
    if os.path.isfile(path):
        print(f"{os.path.getsize(path)/1024/1024:.2f} MB | {name}")

In [ ]:
import os
import time

os.sync()
time.sleep(10)
os.sync()

print("Filesystem sync requested.")

In [ ]:
import os, time

f = "/content/drive/MyDrive/Plant Protein Extrusion MD/md_373_r1.xtc"

os.sync()
time.sleep(10)
os.sync()

if os.path.exists(f):
    print("Size:", round(os.path.getsize(f) / 1024**2, 2), "MB")
    print("Modified:", time.ctime(os.path.getmtime(f)))
else:
    print("File not found in Drive.")

In [ ]:
import os

roots = [
    "/content",
    "/content/drive/MyDrive/Plant Protein Extrusion MD"
]

target = "md_373_r1.xtc"
found = []

for root in roots:
    if not os.path.exists(root):
        continue

    for current, dirs, files in os.walk(root):
        # Avoid scanning unrelated system directories
        dirs[:] = [
            d for d in dirs
            if d not in ["proc", "sys", "dev"]
        ]

        for name in files:
            if name == target or (
                name.startswith(target + ".")
            ):
                path = os.path.join(current, name)
                try:
                    size = os.path.getsize(path) / (1024**2)
                    modified = os.path.getmtime(path)
                    found.append((path, size, modified))
                except OSError:
                    pass

for path, size, modified in sorted(found, key=lambda x: x[2], reverse=True):
    print(f"{size:.2f} MB | {__import__('time').ctime(modified)}")
    print(path)
    print("-" * 60)

if not found:
    print("No matching files found in the searched locations.")

In [ ]:
!gmx check -f "/content/drive/MyDrive/Plant Protein Extrusion MD/md_373_r1.xtc"

In [ ]:
import os
import shutil

src = "/content/drive/MyDrive/Plant Protein Extrusion MD/md_373_r1.xtc"
dst = "/content/drive/MyDrive/Plant Protein Extrusion MD/md_373_r1_verified_copy.xtc"

shutil.copy2(src, dst)
os.sync()

print("Copy created:", os.path.exists(dst))
print("Copy size:", round(os.path.getsize(dst) / 1024**2, 2), "MB")

In [ ]:
%%bash
cd "/content/drive/MyDrive/Plant Protein Extrusion MD"

echo "=== 373 K starting files ==="
find . -maxdepth 2 -type f \( \
  -name '*373*r1*.gro' -o \
  -name '*373*r1*.cpt' -o \
  -name '*373*r1*.tpr' -o \
  -name '*373*r1*.xtc' \
\) -printf '%f | %s bytes\n' | sort

In [ ]:
%%bash
cd "/content/drive/MyDrive/Plant Protein Extrusion MD"

echo "=== NPT checkpoint time ==="
gmx dump -cp npt_373_r1.cpt 2>/dev/null | grep -E 'step =|time =|box \[' | head -10

echo "=== NPT run input / checkpoint files ==="
ls -lh npt_373_r1.cpt npt_373_r1.tpr npt_373_r1.part0002.gro

echo "=== Existing NPT log files ==="
ls -lh npt_373_r1*.log 2>/dev/null || true

In [ ]:
%%bash
cd "/content/drive/MyDrive/Plant Protein Extrusion MD"
gmx check -f npt_373_r1.part0002.gro

In [ ]:
%%bash
cd "/content/drive/MyDrive/Plant Protein Extrusion MD"

echo "=== Checkpoint details ==="
gmx dump -cp npt_373_r1.cpt 2>/dev/null | grep -E 'step =|time ='

echo "=== NPT log ending ==="
tail -n 35 npt_373_r1.part0002.log

## 7. 373 K shear r1: run and recovery
Starts the shear run from the NPT end state, then looks for the output files and checks the trajectory after the run was interrupted.

In [ ]:
%%bash
set -e
cd "/content/drive/MyDrive/Plant Protein Extrusion MD"

gmx grompp \
  -f md_373_shear_r1.mdp \
  -c npt_373_r1.part0002.gro \
  -t npt_373_r1.cpt \
  -p topol.top \
  -o md_373_shear_r1.tpr

In [ ]:
%cd /content/drive/MyDrive/Plant Protein Extrusion MD
!gmx mdrun -deffnm md_373_shear_r1 -cpt 5 -ntmpi 1 -ntomp 2 -pin off

In [ ]:
!cd "/content/drive/MyDrive/Plant Protein Extrusion MD"

In [ ]:
ls -lh md_373_shear_r1*

In [ ]:
!pwd

In [ ]:
!ls -lah /content/drive/MyDrive/

In [ ]:
!ls -lah "/content/drive/MyDrive/Plant Protein Extrusion MD/"

In [ ]:
!find /content -type f -name "*shear*r1*" 2>/dev/null

In [ ]:
!tail -n 20 "/content/drive/MyDrive/Plant Protein Extrusion MD/md_373_shear_r1.log"

In [ ]:
!cd "/content/drive/MyDrive/Plant Protein Extrusion MD" && ls -lh md_373_shear_r1.cpt md_373_shear_r1.tpr md_373_shear_r1.edr md_373_shear_r1.xtc backups/md_373_shear_r1/20261013_173650/

In [ ]:
!cd "/content/drive/MyDrive/Plant Protein Extrusion MD" && G=/usr/local/envs/gmx_cuda/bin.AVX2_256/gmx && echo "===== FILES =====" && ls -lh md_373_shear_r1.* backups/md_373_shear_r1/20261013_173650/ && echo "===== XTC PROGRESS =====" && $G check -f backups/md_373_shear_r1/20261013_173650/md_373_shear_r1.xtc 2>&1 | tail -20 && echo "===== EDR PROGRESS =====" && $G check -e backups/md_373_shear_r1/20261013_173650/md_373_shear_r1.edr 2>&1 | tail -20 && echo "===== MAIN CPT =====" && $G dump -cp md_373_shear_r1.cpt 2>&1 | grep -Ei 'step|time' | head -10 && echo "===== BACKUP CPT =====" && $G dump -cp backups/md_373_shear_r1/20261013_173650/md_373_shear_r1.cpt 2>&1 | grep -Ei 'step|time' | head -10 && echo "===== TPR SETTINGS =====" && $G dump -s md_373_shear_r1.tpr 2>&1 | grep -E 'nsteps|init-step|dt[[:space:]]*=' | head -10

In [ ]:
!cd "/content/drive/MyDrive/Plant Protein Extrusion MD" && echo "===== LOG END =====" && tail -35 md_373_shear_r1.log && echo "===== CHECKPOINT TIME FIELDS =====" && /usr/local/envs/gmx_cuda/bin.AVX2_256/gmx dump -cp md_373_shear_r1.cpt 2>&1 | grep -Ei 'step =|time =|t[[:space:]]*=' | head -25

In [ ]:
!cd "/content/drive/MyDrive/Plant Protein Extrusion MD" && grep -inE 'checkpoint|Writing checkpoint|step.*22800|step.*35000|Finished mdrun' md_373_shear_r1.log | tail -20

In [ ]:
!cd "/content/drive/MyDrive/Plant Protein Extrusion MD" && cp md_373_shear_r1.cpt md_373_shear_r1.cpt.pre_restart_backup && /usr/local/envs/gmx_cuda/bin.AVX2_256/gmx mdrun -s md_373_shear_r1.tpr -deffnm md_373_shear_r1 -cpi md_373_shear_r1.cpt -noappend -cpt 5 -ntomp 2

In [ ]:
!cd "/content/drive/MyDrive/Plant Protein Extrusion MD" && ls -lh md_373_shear_r1.part0002.*

In [ ]:
!cd "/content/drive/MyDrive/Plant Protein Extrusion MD" && /usr/local/envs/gmx_cuda/bin.AVX2_256/gmx check -f md_373_shear_r1.part0002.xtc

In [ ]:
!cd "/content/drive/MyDrive/Plant Protein Extrusion MD" && /usr/local/envs/gmx_cuda/bin.AVX2_256/gmx trjcat -f backups/md_373_shear_r1/20261013_173650/md_373_shear_r1.xtc md_373_shear_r1.part0002.xtc -o md_373_shear_r1_combined.xtc

In [ ]:
!/usr/local/envs/gmx_cuda/bin.AVX2_256/gmx check -f md_373_shear_r1_combined.xtc

In [ ]:
!cd "/content/drive/MyDrive/Plant Protein Extrusion MD" && /usr/local/envs/gmx_cuda/bin.AVX2_256/gmx check -f md_373_shear_r1_combined.xtc

In [ ]:
!cd "/content/drive/MyDrive/Plant Protein Extrusion MD" && /usr/local/envs/gmx_cuda/bin.AVX2_256/gmx eneconv -f backups/md_373_shear_r1/20261013_173650/md_373_shear_r1.edr md_373_shear_r1.part0002.edr -o md_373_shear_r1_combined.edr

In [ ]:
!cd "/content/drive/MyDrive/Plant Protein Extrusion MD" && /usr/local/envs/gmx_cuda/bin.AVX2_256/gmx check -e md_373_shear_r1_combined.edr

In [ ]:
!cd "/content/drive/MyDrive/Plant Protein Extrusion MD" && ls -lh *.gro *.log *.cpt *.tpr 2>/dev/null

In [ ]:
!cd "/content/drive/MyDrive/Plant Protein Extrusion MD" && /usr/local/envs/gmx_cuda/bin.AVX2_256/gmx check -f md_373_shear_r1_combined.xtc && cp md_373_shear_r1.part0002.gro md_373_shear_r1_final.gro && ls -lh md_373_shear_r1_final.gro

## 8. Per-run analysis (300 K r1)
Trajectory preparation, whole-hexamer and per-chain RMSD, RMSF, Rg, SASA, H-bonds and DSSP. Set `S` (file prefix) and `OUT` (output folder) at the top of the block.

In [ ]:
%%bash
ls -lh $P/$S.*
echo "S=$S  OUT=$OUT"
gmx check -f "$P/$S.xtc" 2>&1 | grep -iE "Natoms|Last frame|Step|Time"
find /content/drive/MyDrive -name "md_300_r1.xtc" -exec ls -lh {} \; 2>/dev/null

In [ ]:
%%bash
cd "$OUT/frames"
printf "Protein\n" | gmx convert-tpr -s "$P/$S.tpr" -o protein.tpr 2>&1 | tail -1
printf "Protein\n" | gmx trjconv -s protein.tpr -f "$P/$S.xtc" -o whole2.xtc -pbc whole 2>&1 | tail -1
printf "Protein\n" | gmx trjconv -s protein.tpr -f whole2.xtc -o first_w.gro -dump 0 2>&1 | tail -1

In [ ]:
import os
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"): drive.mount('/content/drive')
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
os.system('ln -sfn "/content/drive/MyDrive/Plant Protein Extrusion MD" /content/pm')
os.environ.update(P="/content/pm", S="md_300_r1", OUT="/content/pm/analysism/legumin_300_r1")
for d in ["graphs","logs","frames","xvgs"]: os.makedirs(f'{os.environ["OUT"]}/{d}', exist_ok=True)
os.system('pip install -q MDAnalysis scipy; gmx --version | head -1; ls $P/$S.*')

In [ ]:
!pip install -q MDAnalysis scipy

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "pip", "install", "MDAnalysis", "scipy"], capture_output=True, text=True)
print(r.stdout[-400:]); print(r.stderr[-400:])
import MDAnalysis
print("MDAnalysis", MDAnalysis.__version__)

In [ ]:
import os, json, numpy as np
OUT = os.environ["OUT"]
lines = open(f"{OUT}/frames/first_w.gro").read().split("\n")
n = int(lines[1]); body = lines[2:2+n]
resnum = np.array([int(l[0:5]) for l in body]); nm = [l[10:15].strip() for l in body]

# new chain where residue number decreases
starts = [0] + [i for i in range(1, n) if resnum[i] < resnum[i-1]]
print("total atoms:", n, "| chains found:", len(starts))
ends = starts[1:] + [n]
sizes = [e-s for s, e in zip(starts, ends)]
nres = [len(set(resnum[s:e])) for s, e in zip(starts, ends)]
print("atoms per chain:", sizes); print("residues per chain:", nres)
assert len(starts) == 6, "expected 6 chains - send me the two lines above"

L = "ABCDEF"; G = {}; ranges = {}
for k, (s, e) in enumerate(zip(starts, ends)):
    r = range(s+1, e+1); c = "ch"+L[k]; ranges[c] = [s, e]
    G[c] = list(r); G[c+"_bb"] = [i for i in r if nm[i-1] in ("N","CA","C")]; G[c+"_CA"] = [i for i in r if nm[i-1]=="CA"]
r = range(1, n+1)
G["Protein"] = list(r); G["Protein_bb"] = [i for i in r if nm[i-1] in ("N","CA","C")]; G["Protein_CA"] = [i for i in r if nm[i-1]=="CA"]
with open(f"{OUT}/frames/chains.ndx","w") as f:
    for g, idx in G.items():
        f.write(f"[ {g} ]\n")
        for k in range(0, len(idx), 15): f.write(" ".join(f"{i:5d}" for i in idx[k:k+15]) + "\n")
json.dump(dict(ranges=ranges, nres=nres[0], natoms=n), open(f"{OUT}/frames/chains.json","w"))
print({g: len(v) for g, v in G.items()})

In [ ]:
import os, json, numpy as np, MDAnalysis as mda
from MDAnalysis.lib.distances import minimize_vectors
OUT = os.environ["OUT"]
rg = json.load(open(f"{OUT}/frames/chains.json"))["ranges"]
sl = [slice(*rg[c]) for c in ("chA","chB","chC","chD","chE","chF")]
u = mda.Universe(f"{OUT}/frames/first_w.gro", f"{OUT}/frames/whole2.xtc")
with mda.Writer(f"{OUT}/frames/together.xtc", u.atoms.n_atoms) as W:
    for ts in u.trajectory[::5]:
        pos = u.atoms.positions.copy(); box = ts.dimensions.astype(np.float32)
        a0 = pos[sl[0]].mean(0)
        for s in sl[1:]:
            d = pos[s].mean(0) - a0
            dm = minimize_vectors(d[None,:].astype(np.float32), box)[0]
            pos[s] -= (d - dm)
        u.atoms.positions = pos; W.write(u.atoms)
print("done")

In [ ]:
%%bash
cd "$OUT/frames"
printf "Backbone\nProtein\n" | gmx trjconv -s protein.tpr -f together.xtc -o fit_cluster_skip5.xtc -fit rot+trans 2>&1 | tail -1
printf "Protein\n" | gmx trjconv -s protein.tpr -f fit_cluster_skip5.xtc -o first_cl.gro -dump 0 2>&1 | tail -1
rm -f whole2.xtc together.xtc
gmx check -f fit_cluster_skip5.xtc 2>&1 | grep -E "Last frame|Step"

In [ ]:
import itertools
from scipy.spatial import cKDTree
u = mda.Universe(f"{OUT}/frames/first_cl.gro", f"{OUT}/frames/fit_cluster_skip5.xtc")
names = "ABCDEF"; pairs = list(itertools.combinations(range(6), 2))
worst = {p: 0.0 for p in pairs}; lonely = 0
for ts in u.trajectory:
    P = u.atoms.positions/10; ch = [P[s] for s in sl]
    m = {p: cKDTree(ch[p[0]]).query(ch[p[1]])[0].min() for p in pairs}
    for p in pairs: worst[p] = max(worst[p], m[p])
    for i in range(6):
        if min(m[p] for p in pairs if i in p) > 0.5: lonely += 1
print({names[a]+names[b]: round(float(v),2) for (a,b), v in worst.items()})
print("frames x chains with no neighbour within 0.5 nm:", lonely)
assert lonely == 0, "a chain drifted away - send me the two lines above"

In [ ]:
%%bash
cd "$OUT/frames"; X="$OUT/xvgs"; L="$OUT/logs"; T=fit_cluster_skip5.xtc
for c in Protein chA chB chC chD chE chF; do
  printf "${c}_bb\n${c}_bb\n" | gmx rms  -s first_cl.gro -f $T -n chains.ndx -o $X/rmsd_$c.xvg -tu ns > $L/rmsd_$c.log 2>&1
  printf "${c}_CA\n${c}_CA\n" | gmx rmsf -s first_cl.gro -f $T -n chains.ndx -o $X/rmsf_$c.xvg -res   > $L/rmsf_$c.log 2>&1
  printf "$c\n"               | gmx gyrate -s protein.tpr -f $T -n chains.ndx -o $X/rg_$c.xvg          > $L/rg_$c.log 2>&1
  gmx sasa -s protein.tpr -f $T -n chains.ndx -surface "group Protein" -output "group $c" -o $X/sasa_$c.xvg -or $X/tmp_$c.xvg -tu ns > $L/sasa_$c.log 2>&1
  gmx hbond -s protein.tpr -f $T -n chains.ndx -r "group $c" -t "group $c" -num $X/hbnum_$c.xvg -tu ns > $L/hbond_$c.log 2>&1
done
for p in "chA chB" "chA chC" "chA chD" "chA chE" "chA chF" "chB chC" "chB chD" "chB chE" "chB chF" "chC chD" "chC chE" "chC chF" "chD chE" "chD chF" "chE chF"; do set -- $p
  gmx hbond -s protein.tpr -f $T -n chains.ndx -r "group $1" -t "group $2" -num $X/hbinter_$1_$2.xvg -tu ns > $L/hbinter_$1_$2.log 2>&1
done
mv $X/tmp_Protein.xvg $X/sasa_perres_Protein.xvg; rm -f $X/tmp_ch*.xvg
for f in rmsd_Protein rmsd_chA rmsf_chA rg_Protein sasa_chA hbnum_Protein hbinter_chA_chB; do echo $f $(tail -1 $X/$f.xvg); done

In [ ]:
import os, numpy as np, matplotlib.pyplot as plt
OUT = os.environ["OUT"]; X = f"{OUT}/xvgs"
CH = ["chA","chB","chC","chD","chE","chF"]; NR = 496
cols = ["#d62728","#ff9896","#1f77b4","#7fbfff","#2ca02c","#98df8a"]
sty = {c: dict(color=cols[i], ls="-" if i%2==0 else ":", label="Chain "+c[-1]) for i, c in enumerate(CH)}
def xvg(f): return np.loadtxt(f, comments=('#','@'))
def draw(out, title, xl, yl, items):
    fig, ax = plt.subplots(figsize=(12,5), dpi=150)
    for x, y, s in items: ax.plot(x, y, lw=1.6, **s)
    ax.set(xlabel=xl, ylabel=yl, title=title); ax.legend(ncol=2); fig.tight_layout()
    fig.savefig(f"{OUT}/graphs/{out}.png"); plt.show()
def chains(pre, col=1, sc=1):
    return [(xvg(f"{X}/{pre}_{c}.xvg")[:,0]/sc, xvg(f"{X}/{pre}_{c}.xvg")[:,col], s) for c, s in sty.items()]
def whole(pre, col=1, sc=1):
    a = xvg(f"{X}/{pre}_Protein.xvg"); return [(a[:,0]/sc, a[:,col], dict(color="grey", label="Whole hexamer"))]

for pre, yl, ttl, sc in (("rmsd","RMSD (nm)","Backbone RMSD",1), ("rg","Rg (nm)","Radius of gyration",1000),
                         ("sasa","SASA (nm²)","SASA",1), ("hbnum","H-bonds","Intra-chain H-bonds",1)):
    cc = 2 if pre == "sasa" else 1
    draw(f"{pre}_per_chain", f"{ttl} per chain, 300 K (r1)", "Time (ns)", yl, chains(pre, cc, sc))
    draw(f"{pre}_protein", f"{ttl}, whole hexamer, 300 K (r1)", "Time (ns)", yl, whole(pre, 1, sc))

draw("rmsf_per_chain", "C-alpha RMSF per residue, 300 K (r1)", "Residue", "RMSF (nm)", chains("rmsf"))
d = xvg(f"{X}/sasa_perres_Protein.xvg"); assert len(d) == 6*NR, len(d)
draw("sasa_per_residue", "SASA per residue (time-averaged), 300 K (r1)", "Residue", "SASA (nm²)",
     [(np.arange(1,NR+1), d[i*NR:(i+1)*NR,1], s) for i, s in enumerate(sty.values())])

import itertools
pairs = list(itertools.combinations(CH, 2))
fig, ax = plt.subplots(figsize=(12,6), dpi=150)
for k, (a, b) in enumerate(pairs):
    h = xvg(f"{X}/hbinter_{a}_{b}.xvg"); ax.plot(h[:,0], h[:,1], lw=1.2, color=plt.cm.tab20(k), label=f"{a[-1]}–{b[-1]}")
ax.set(xlabel="Time (ns)", ylabel="H-bonds", title="Inter-chain H-bonds, 300 K (r1)"); ax.legend(ncol=3, fontsize=8)
fig.tight_layout(); fig.savefig(f"{OUT}/graphs/hbonds_inter_chain.png"); plt.show()

In [ ]:
import numpy as np, MDAnalysis as mda
from scipy.spatial import cKDTree
from MDAnalysis.analysis.align import rotation_matrix
u = mda.Universe(f"{OUT}/frames/first_cl.gro", f"{OUT}/frames/fit_cluster_skip5.xtc")
NA = u.atoms.n_atoms // 6
cs = {c: slice(i*NA, (i+1)*NA) for i, c in enumerate(CH)}

# per-residue C-alpha RMSD vs first frame (each chain fitted on its own backbone)
resR = {}
for name, s in cs.items():
    chn = u.atoms[s]; bb = chn.select_atoms("name N CA C"); ca = chn.select_atoms("name CA")
    u.trajectory[0]; c0 = bb.positions.mean(0); bb0 = bb.positions - c0; ca0 = ca.positions - c0
    d2 = np.zeros(len(ca)); n = 0
    for ts in u.trajectory:
        c = bb.positions.mean(0); R, _ = rotation_matrix(bb.positions - c, bb0)
        d2 += (((ca.positions - c) @ R.T - ca0)**2).sum(1); n += 1
    resR[name] = np.sqrt(d2/n)/10
    np.savetxt(f"{X}/rmsd_perres_{name}.xvg", np.c_[np.arange(1,NR+1), resR[name]], fmt="%d %.6f")
draw("rmsd_per_residue", "C-alpha RMSD per residue vs first frame, 300 K (r1)", "Residue", "RMSD (nm)",
     [(np.arange(1,NR+1), resR[c], s) for c, s in sty.items()])
m = np.mean(list(resR.values()), axis=0)
draw("rmsd_per_residue_mean", "Mean C-alpha RMSD per residue (6 chains), 300 K (r1)", "Residue", "RMSD (nm)",
     [(np.arange(1,NR+1), m, dict(color="grey", label="Mean of chains"))])

# per-residue H-bonds (same geometry criteria as the vicilin run: H...A < 0.25 nm, D-A < 0.35 nm, angle < 30 deg)
nmA = np.array([n.strip() for n in u.atoms.names]); rn = u.atoms.resnames
chain = np.arange(len(nmA)) // NA
ridx = u.atoms.resindices - chain*NR            # residue position within its chain, 0..495
assert ridx.min() == 0 and ridx.max() == NR-1
isH = np.array([n.startswith("H") for n in nmA])
P0 = u.atoms.positions/10; heavy = np.where(~isH)[0]; Hs = np.where(isH)[0]
don = heavy[cKDTree(P0[heavy]).query(P0[Hs])[1]]
k = np.array([nmA[i][0] in "NO" for i in don]); Hs, don = Hs[k], don[k]
acc = np.array([i for i in range(len(nmA)) if nmA[i][0]=="O" or (nmA[i] in ("ND1","NE2") and rn[i].startswith("HI"))])
tot = np.zeros((6,NR)); inter = np.zeros((6,NR)); nb = []
for ts in u.trajectory:
    Pf = u.atoms.positions/10; n = 0
    for hi, lst in enumerate(cKDTree(Pf[Hs]).query_ball_tree(cKDTree(Pf[acc]), 0.25)):
        D = don[hi]
        for aj in lst:
            A = acc[aj]
            if A == D or (chain[A]==chain[D] and ridx[A]==ridx[D]): continue
            v1, v2 = Pf[Hs[hi]]-Pf[D], Pf[A]-Pf[D]; dist = np.linalg.norm(v2)
            if dist > 0.35 or v1@v2/(np.linalg.norm(v1)*dist) < np.cos(np.radians(30)): continue
            n += 1
            for Xa in (D, A):
                tot[chain[Xa], ridx[Xa]] += 1
                if chain[D] != chain[A]: inter[chain[Xa], ridx[Xa]] += 1
    nb.append(n)
nf = len(nb); print("mean H-bonds/frame:", round(float(np.mean(nb)),1))
for nmx, arr, ttl in (("hbperres",tot,"All H-bonds per residue"), ("hbperres_inter",inter,"Inter-chain H-bonds per residue")):
    for i, c in enumerate(CH): np.savetxt(f"{X}/{nmx}_{c}.xvg", np.c_[np.arange(1,NR+1), arr[i]/nf], fmt="%d %.5f")
    draw(nmx, f"{ttl}, 300 K (r1)", "Residue", "H-bonds per frame",
         [(np.arange(1,NR+1), arr[i]/nf, s) for i, s in enumerate(sty.values())])

In [ ]:
%%bash
cd "$OUT/frames"
printf "Protein\n" | gmx dssp -s protein.tpr -f fit_cluster_skip5.xtc -n chains.ndx -o "$OUT/xvgs/dssp_Protein.dat" > "$OUT/logs/dssp_Protein.log" 2>&1
cp "$OUT/xvgs/dssp_Protein.dat" "$OUT/xvgs/dssp_chA.dat"
head -c 80 "$OUT/xvgs/dssp_Protein.dat"; echo; tail -3 "$OUT/logs/dssp_Protein.log"

In [ ]:
import numpy as np, matplotlib.pyplot as plt
a = np.array([list(l.rstrip("\n")) for l in open(f"{X}/dssp_Protein.dat") if l.strip()])
brk = np.where((a == "=").all(0))[0]
print("shape:", a.shape, "| chain breaks at columns:", brk)
a = a[:, ~(a == "=").all(0)]
assert a.shape[1] == 6*NR, a.shape
D = {c: a[:, i*NR:(i+1)*NR] for i, c in enumerate(CH)}
t = np.arange(len(a))*0.005          # 201 frames x 5 ps -> ns
for name, ss, yl in (("helix","HGI","Helix residues (%)"), ("strand","EB","Strand residues (%)")):
    fig, ax = plt.subplots(figsize=(12,5), dpi=150)
    for c, s in sty.items():
        ax.plot(t, 100*np.isin(D[c], list(ss)).mean(1), lw=1.6, **s)
        print(name, c, "mean %:", round(100*np.isin(D[c], list(ss)).mean(), 1))
    ax.set(xlabel="Time (ns)", ylabel=yl, title=f"{name.capitalize()} content vs time, 300 K (r1)")
    ax.legend(ncol=2); fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_{name}_time.png"); plt.show()

    fig, ax = plt.subplots(figsize=(12,5), dpi=150)
    for c, s in sty.items():
        f = np.isin(D[c], list(ss)).mean(0); ax.plot(np.arange(1,NR+1), f, lw=1.4, **s)
        np.savetxt(f"{X}/dssp_{name}_perres_{c}.xvg", np.c_[np.arange(1,NR+1), f], fmt="%d %.4f")
    ax.set(xlabel="Residue", ylabel=f"Fraction of frames in {name}", title=f"{name.capitalize()} per residue, 300 K (r1)")
    ax.legend(ncol=2); fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_{name}_perres.png"); plt.show()

In [ ]:
from matplotlib.colors import ListedColormap
fig, ax = plt.subplots(figsize=(12,5), dpi=150)
for c, s in sty.items():
    ax.plot(t, 100*(1-np.isin(D[c], list("HGIEB")).mean(1)), lw=1.6, **s)
ax.set(xlabel="Time (ns)", ylabel="Coil/turn residues (%)", title="Coil/turn content vs time, 300 K (r1)")
ax.legend(ncol=2); fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_coil_time.png"); plt.show()

cmap = ListedColormap(["#eeeeee", "#d62728", "#1f77b4"])
code = lambda m: np.where(np.isin(m, list("HGI")), 1, np.where(np.isin(m, list("EB")), 2, 0))
fig, axs = plt.subplots(6, 1, figsize=(12,13), dpi=150, sharex=True)
for ax, (c, s) in zip(axs, sty.items()):
    ax.imshow(code(D[c]), aspect="auto", cmap=cmap, vmin=0, vmax=2, origin="lower", extent=[1,NR,0,1])
    ax.set_ylabel("Time (ns)"); ax.set_title(s["label"], loc="left", fontsize=10)
axs[-1].set_xlabel("Residue"); fig.suptitle("DSSP map: red = helix, blue = strand, grey = coil/turn, 300 K (r1)")
fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_map.png"); plt.show()

In [ ]:
import pandas as pd
def last(f, col=1): a = np.loadtxt(f, comments=('#','@')); return a[len(a)//2:, col].mean()
h = len(t)//2; rows = []
for c in CH:
    rows.append(dict(chain=c, RMSD_nm=last(f"{X}/rmsd_{c}.xvg"), Rg_nm=last(f"{X}/rg_{c}.xvg"),
      SASA_nm2=last(f"{X}/sasa_{c}.xvg", 2), intra_Hbonds=last(f"{X}/hbnum_{c}.xvg"),
      helix_pct=100*np.isin(D[c][h:], list("HGI")).mean(), strand_pct=100*np.isin(D[c][h:], list("EB")).mean()))
rows.append(dict(chain="hexamer", RMSD_nm=last(f"{X}/rmsd_Protein.xvg"), Rg_nm=last(f"{X}/rg_Protein.xvg"),
                 SASA_nm2=last(f"{X}/sasa_Protein.xvg"), intra_Hbonds=last(f"{X}/hbnum_Protein.xvg")))
df = pd.DataFrame(rows).round(3); df.to_csv(f"{X}/summary_last0.5ns.csv", index=False); print(df)

STATES = [("Alpha helix","H"),("3-10 helix","G"),("Pi helix","I"),("Beta strand","E"),("Beta bridge","B"),
          ("Turn","T"),("Bend","S"),("Polyproline II helix","P")]
P = {n: 100*(a == c).mean(1) for n, c in STATES}; P["Coil / Unstructured"] = 100 - sum(P.values())
tp = np.arange(len(a))*5.0
fig, ax = plt.subplots(figsize=(12,6.2), dpi=150)
for i, (n, y) in enumerate(P.items()): ax.plot(tp, y, color=plt.cm.tab10(i), lw=1.6, label=n)
ax.set(xlabel="Time (ps)", ylabel="Percentage (%)", title="Legumin All Secondary Structure States — 300 K (r1)")
ax.grid(alpha=.3); ax.legend(title="Secondary structure", ncol=2, loc="upper right"); fig.tight_layout()
fig.savefig(f"{OUT}/graphs/dssp_all_states_300K_r1.png"); plt.show()

G = {"Alpha helix": P["Alpha helix"], "Beta structure": P["Beta strand"]+P["Beta bridge"], "Turns": P["Turn"]}
G["Loops / Other"] = 100 - sum(G.values())
fig, ax = plt.subplots(figsize=(10,6), dpi=150)
for i, (n, y) in enumerate(G.items()): ax.plot(tp, y, color=plt.cm.tab10(i), lw=1.8, label=n)
ax.set(xlabel="Time (ps)", ylabel="Percentage (%)", title="Legumin Secondary Structure — 300 K (r1)")
ax.grid(alpha=.3); ax.legend(); fig.tight_layout()
fig.savefig(f"{OUT}/graphs/dssp_groups_300K_r1.png"); plt.show()

## 9. Single-cell analysis pipeline
One cell per run (set `TEMP`, `RUN`, `TPR_NAME`, `TRAJ_NAME` at the top): 300 K r2, 373 K r1 and 373 K shear r1, with file checks between runs.

In [ ]:
import os, subprocess, sys
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"): drive.mount('/content/drive')
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
os.system('ln -sfn "/content/drive/MyDrive/Plant Protein Extrusion MD" /content/pm')
os.environ.update(P="/content/pm", S="md_300_r2", OUT="/content/pm/analysism/legumin_300_r2")
for d in ["graphs","logs","frames","xvgs"]: os.makedirs(f'{os.environ["OUT"]}/{d}', exist_ok=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "MDAnalysis", "scipy"])
import MDAnalysis; print("MDAnalysis", MDAnalysis.__version__)
os.system('gmx --version | head -1; ls $P/$S.*')

In [ ]:
%%bash
ls -lh $P/$S.*
echo "S=$S  OUT=$OUT"
gmx check -f "$P/$S.xtc" 2>&1 | grep -iE "Natoms|Last frame|Step|Time"

In [ ]:
import os, sys, json, itertools, subprocess
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from google.colab import drive
RUN = "r2"                                   # <- only this changes per run
if not os.path.exists("/content/drive/MyDrive"): drive.mount('/content/drive')
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
os.system('ln -sfn "/content/drive/MyDrive/Plant Protein Extrusion MD" /content/pm')
S = f"md_300_{RUN}"; OUT = f"/content/pm/analysism/legumin_300_{RUN}"; T = f"300 K ({RUN})"
os.environ.update(P="/content/pm", S=S, OUT=OUT)
for d in ["graphs","logs","frames","xvgs"]: os.makedirs(f"{OUT}/{d}", exist_ok=True)
subprocess.run([sys.executable,"-m","pip","install","-q","MDAnalysis","scipy"])
import MDAnalysis as mda
from MDAnalysis.lib.distances import minimize_vectors
from MDAnalysis.analysis.align import rotation_matrix
from scipy.spatial import cKDTree
from matplotlib.colors import ListedColormap
def sh(cmd, tail=0):
    r = subprocess.run(["bash","-c",cmd], capture_output=True, text=True)
    o = (r.stdout + r.stderr).strip().split("\n")
    print("\n".join(o[-tail:] if tail else o))

# ---- check files
print("== files"); sh('ls -lh $P/$S.*; gmx check -f "$P/$S.xtc" 2>&1 | grep -iE "Natoms|Last frame|Step"')

# ---- protein tpr, whole, first frame
print("== whole")
sh(r'''cd "$OUT/frames"
printf "Protein\n" | gmx convert-tpr -s "$P/$S.tpr" -o protein.tpr 2>&1 | tail -1
printf "Protein\n" | gmx trjconv -s protein.tpr -f "$P/$S.xtc" -o whole2.xtc -pbc whole 2>&1 | tail -1
printf "Protein\n" | gmx trjconv -s protein.tpr -f whole2.xtc -o first_w.gro -dump 0 2>&1 | tail -1''')

# ---- chain detection from .gro
lines = open(f"{OUT}/frames/first_w.gro").read().split("\n")
n = int(lines[1]); body = lines[2:2+n]
resnum = np.array([int(l[0:5]) for l in body]); nm = [l[10:15].strip() for l in body]
starts = [0] + [i for i in range(1, n) if resnum[i] < resnum[i-1]]; ends = starts[1:] + [n]
nres = [len(set(resnum[s:e])) for s, e in zip(starts, ends)]
print("atoms:", n, "| chains:", len(starts), "| atoms/chain:", [e-s for s,e in zip(starts,ends)], "| res/chain:", nres)
assert len(starts) == 6, "expected 6 chains"
NR = nres[0]; CH = ["chA","chB","chC","chD","chE","chF"]; G = {}; ranges = {}
for k, (s, e) in enumerate(zip(starts, ends)):
    r = range(s+1, e+1); c = CH[k]; ranges[c] = [s, e]
    G[c] = list(r); G[c+"_bb"] = [i for i in r if nm[i-1] in ("N","CA","C")]; G[c+"_CA"] = [i for i in r if nm[i-1]=="CA"]
r = range(1, n+1)
G["Protein"] = list(r); G["Protein_bb"] = [i for i in r if nm[i-1] in ("N","CA","C")]; G["Protein_CA"] = [i for i in r if nm[i-1]=="CA"]
with open(f"{OUT}/frames/chains.ndx","w") as f:
    for g, idx in G.items():
        f.write(f"[ {g} ]\n")
        for k in range(0, len(idx), 15): f.write(" ".join(f"{i:5d}" for i in idx[k:k+15]) + "\n")
json.dump(dict(ranges=ranges, nres=NR, natoms=n), open(f"{OUT}/frames/chains.json","w"))

# ---- keep 6 chains together
sl = [slice(*ranges[c]) for c in CH]
u = mda.Universe(f"{OUT}/frames/first_w.gro", f"{OUT}/frames/whole2.xtc")
with mda.Writer(f"{OUT}/frames/together.xtc", u.atoms.n_atoms) as W:
    for ts in u.trajectory[::5]:
        pos = u.atoms.positions.copy(); box = ts.dimensions.astype(np.float32); a0 = pos[sl[0]].mean(0)
        for s in sl[1:]:
            d = pos[s].mean(0) - a0
            dm = minimize_vectors(d[None,:].astype(np.float32), box)[0]; pos[s] -= (d - dm)
        u.atoms.positions = pos; W.write(u.atoms)

# ---- fit
print("== fit")
sh(r'''cd "$OUT/frames"
printf "Backbone\nProtein\n" | gmx trjconv -s protein.tpr -f together.xtc -o fit_cluster_skip5.xtc -fit rot+trans 2>&1 | tail -1
printf "Protein\n" | gmx trjconv -s protein.tpr -f fit_cluster_skip5.xtc -o first_cl.gro -dump 0 2>&1 | tail -1
rm -f whole2.xtc together.xtc
gmx check -f fit_cluster_skip5.xtc 2>&1 | grep -E "Last frame|Step"''')

# ---- hexamer integrity check
u = mda.Universe(f"{OUT}/frames/first_cl.gro", f"{OUT}/frames/fit_cluster_skip5.xtc")
pairs = list(itertools.combinations(range(6), 2)); worst = {p: 0.0 for p in pairs}; lonely = 0
for ts in u.trajectory:
    Pp = u.atoms.positions/10; ch = [Pp[s] for s in sl]
    m = {p: cKDTree(ch[p[0]]).query(ch[p[1]])[0].min() for p in pairs}
    for p in pairs: worst[p] = max(worst[p], m[p])
    for i in range(6):
        if min(m[p] for p in pairs if i in p) > 0.5: lonely += 1
print({"ABCDEF"[a]+"ABCDEF"[b]: round(float(v),2) for (a,b), v in worst.items()})
print("chains with no neighbour within 0.5 nm:", lonely)
assert lonely == 0, "a chain drifted away"

# ---- GROMACS analysis
print("== gmx analysis (slow)")
sh(r'''cd "$OUT/frames"; X="$OUT/xvgs"; L="$OUT/logs"; T=fit_cluster_skip5.xtc
for c in Protein chA chB chC chD chE chF; do
  printf "${c}_bb\n${c}_bb\n" | gmx rms  -s first_cl.gro -f $T -n chains.ndx -o $X/rmsd_$c.xvg -tu ns > $L/rmsd_$c.log 2>&1
  printf "${c}_CA\n${c}_CA\n" | gmx rmsf -s first_cl.gro -f $T -n chains.ndx -o $X/rmsf_$c.xvg -res   > $L/rmsf_$c.log 2>&1
  printf "$c\n"               | gmx gyrate -s protein.tpr -f $T -n chains.ndx -o $X/rg_$c.xvg          > $L/rg_$c.log 2>&1
  gmx sasa -s protein.tpr -f $T -n chains.ndx -surface "group Protein" -output "group $c" -o $X/sasa_$c.xvg -or $X/tmp_$c.xvg -tu ns > $L/sasa_$c.log 2>&1
  gmx hbond -s protein.tpr -f $T -n chains.ndx -r "group $c" -t "group $c" -num $X/hbnum_$c.xvg -tu ns > $L/hbond_$c.log 2>&1
done
for p in "chA chB" "chA chC" "chA chD" "chA chE" "chA chF" "chB chC" "chB chD" "chB chE" "chB chF" "chC chD" "chC chE" "chC chF" "chD chE" "chD chF" "chE chF"; do set -- $p
  gmx hbond -s protein.tpr -f $T -n chains.ndx -r "group $1" -t "group $2" -num $X/hbinter_$1_$2.xvg -tu ns > $L/hbinter_$1_$2.log 2>&1
done
mv $X/tmp_Protein.xvg $X/sasa_perres_Protein.xvg; rm -f $X/tmp_ch*.xvg
for f in rmsd_Protein rmsd_chA rmsf_chA rg_Protein sasa_chA hbnum_Protein hbinter_chA_chB; do echo $f $(tail -1 $X/$f.xvg); done''')

# ---- plots
X = f"{OUT}/xvgs"
cols = ["#d62728","#ff9896","#1f77b4","#7fbfff","#2ca02c","#98df8a"]
sty = {c: dict(color=cols[i], ls="-" if i%2==0 else ":", label="Chain "+c[-1]) for i, c in enumerate(CH)}
xvg = lambda f: np.loadtxt(f, comments=('#','@'))
def draw(out, title, xl, yl, items):
    fig, ax = plt.subplots(figsize=(12,5), dpi=150)
    for x, y, s in items: ax.plot(x, y, lw=1.6, **s)
    ax.set(xlabel=xl, ylabel=yl, title=title); ax.legend(ncol=2); fig.tight_layout()
    fig.savefig(f"{OUT}/graphs/{out}.png"); plt.show()
chains = lambda pre, col=1, sc=1: [(xvg(f"{X}/{pre}_{c}.xvg")[:,0]/sc, xvg(f"{X}/{pre}_{c}.xvg")[:,col], s) for c, s in sty.items()]
def whole(pre, col=1, sc=1):
    a = xvg(f"{X}/{pre}_Protein.xvg"); return [(a[:,0]/sc, a[:,col], dict(color="grey", label="Whole hexamer"))]
for pre, yl, ttl, sc in (("rmsd","RMSD (nm)","Backbone RMSD",1), ("rg","Rg (nm)","Radius of gyration",1000),
                         ("sasa","SASA (nm²)","SASA",1), ("hbnum","H-bonds","Intra-chain H-bonds",1)):
    cc = 2 if pre == "sasa" else 1
    draw(f"{pre}_per_chain", f"{ttl} per chain, {T}", "Time (ns)", yl, chains(pre, cc, sc))
    draw(f"{pre}_protein", f"{ttl}, whole hexamer, {T}", "Time (ns)", yl, whole(pre, 1, sc))
draw("rmsf_per_chain", f"C-alpha RMSF per residue, {T}", "Residue", "RMSF (nm)", chains("rmsf"))
d = xvg(f"{X}/sasa_perres_Protein.xvg"); assert len(d) == 6*NR, len(d)
draw("sasa_per_residue", f"SASA per residue, {T}", "Residue", "SASA (nm²)",
     [(np.arange(1,NR+1), d[i*NR:(i+1)*NR,1], s) for i, s in enumerate(sty.values())])
fig, ax = plt.subplots(figsize=(12,6), dpi=150)
for k, (a_, b_) in enumerate(itertools.combinations(CH, 2)):
    h = xvg(f"{X}/hbinter_{a_}_{b_}.xvg"); ax.plot(h[:,0], h[:,1], lw=1.2, color=plt.cm.tab20(k), label=f"{a_[-1]}–{b_[-1]}")
ax.set(xlabel="Time (ns)", ylabel="H-bonds", title=f"Inter-chain H-bonds, {T}"); ax.legend(ncol=3, fontsize=8)
fig.tight_layout(); fig.savefig(f"{OUT}/graphs/hbonds_inter_chain.png"); plt.show()

# ---- per-residue RMSD and H-bonds
u = mda.Universe(f"{OUT}/frames/first_cl.gro", f"{OUT}/frames/fit_cluster_skip5.xtc")
NA = u.atoms.n_atoms // 6; cs = {c: slice(i*NA, (i+1)*NA) for i, c in enumerate(CH)}; resR = {}
for name, s in cs.items():
    chn = u.atoms[s]; bb = chn.select_atoms("name N CA C"); ca = chn.select_atoms("name CA")
    u.trajectory[0]; c0 = bb.positions.mean(0); bb0 = bb.positions - c0; ca0 = ca.positions - c0
    d2 = np.zeros(len(ca)); k = 0
    for ts in u.trajectory:
        c = bb.positions.mean(0); R, _ = rotation_matrix(bb.positions - c, bb0)
        d2 += (((ca.positions - c) @ R.T - ca0)**2).sum(1); k += 1
    resR[name] = np.sqrt(d2/k)/10
    np.savetxt(f"{X}/rmsd_perres_{name}.xvg", np.c_[np.arange(1,NR+1), resR[name]], fmt="%d %.6f")
draw("rmsd_per_residue", f"C-alpha RMSD per residue vs first frame, {T}", "Residue", "RMSD (nm)",
     [(np.arange(1,NR+1), resR[c], s) for c, s in sty.items()])
draw("rmsd_per_residue_mean", f"Mean C-alpha RMSD per residue (6 chains), {T}", "Residue", "RMSD (nm)",
     [(np.arange(1,NR+1), np.mean(list(resR.values()), axis=0), dict(color="grey", label="Mean of chains"))])

nmA = np.array([x.strip() for x in u.atoms.names]); rn = u.atoms.resnames
chain = np.arange(len(nmA)) // NA; ridx = u.atoms.resindices - chain*NR
assert ridx.min() == 0 and ridx.max() == NR-1
isH = np.array([x.startswith("H") for x in nmA])
P0 = u.atoms.positions/10; heavy = np.where(~isH)[0]; Hs = np.where(isH)[0]
don = heavy[cKDTree(P0[heavy]).query(P0[Hs])[1]]
kk = np.array([nmA[i][0] in "NO" for i in don]); Hs, don = Hs[kk], don[kk]
acc = np.array([i for i in range(len(nmA)) if nmA[i][0]=="O" or (nmA[i] in ("ND1","NE2") and rn[i].startswith("HI"))])
tot = np.zeros((6,NR)); inter = np.zeros((6,NR)); nb = []
for ts in u.trajectory:
    Pf = u.atoms.positions/10; k = 0
    for hi, lst in enumerate(cKDTree(Pf[Hs]).query_ball_tree(cKDTree(Pf[acc]), 0.25)):
        D_ = don[hi]
        for aj in lst:
            A = acc[aj]
            if A == D_ or (chain[A]==chain[D_] and ridx[A]==ridx[D_]): continue
            v1, v2 = Pf[Hs[hi]]-Pf[D_], Pf[A]-Pf[D_]; dist = np.linalg.norm(v2)
            if dist > 0.35 or v1@v2/(np.linalg.norm(v1)*dist) < np.cos(np.radians(30)): continue
            k += 1
            for Xa in (D_, A):
                tot[chain[Xa], ridx[Xa]] += 1
                if chain[D_] != chain[A]: inter[chain[Xa], ridx[Xa]] += 1
    nb.append(k)
nf = len(nb); print("mean H-bonds/frame:", round(float(np.mean(nb)),1))
for nmx, arr, ttl in (("hbperres",tot,"All H-bonds per residue"), ("hbperres_inter",inter,"Inter-chain H-bonds per residue")):
    for i, c in enumerate(CH): np.savetxt(f"{X}/{nmx}_{c}.xvg", np.c_[np.arange(1,NR+1), arr[i]/nf], fmt="%d %.5f")
    draw(nmx, f"{ttl}, {T}", "Residue", "H-bonds per frame", [(np.arange(1,NR+1), arr[i]/nf, s) for i, s in enumerate(sty.values())])

# ---- DSSP
print("== dssp")
sh(r'''cd "$OUT/frames"
printf "Protein\n" | gmx dssp -s protein.tpr -f fit_cluster_skip5.xtc -n chains.ndx -o "$OUT/xvgs/dssp_Protein.dat" > "$OUT/logs/dssp_Protein.log" 2>&1
cp "$OUT/xvgs/dssp_Protein.dat" "$OUT/xvgs/dssp_chA.dat"
head -c 80 "$OUT/xvgs/dssp_Protein.dat"; echo''')
a = np.array([list(l.rstrip("\n")) for l in open(f"{X}/dssp_Protein.dat") if l.strip()])
a = a[:, ~(a == "=").all(0)]; print("dssp shape:", a.shape); assert a.shape[1] == 6*NR, a.shape
Dd = {c: a[:, i*NR:(i+1)*NR] for i, c in enumerate(CH)}; t = np.arange(len(a))*0.005
for name, ss, yl in (("helix","HGI","Helix residues (%)"), ("strand","EB","Strand residues (%)")):
    fig, ax = plt.subplots(figsize=(12,5), dpi=150)
    for c, s in sty.items():
        ax.plot(t, 100*np.isin(Dd[c], list(ss)).mean(1), lw=1.6, **s)
        print(name, c, "mean %:", round(100*np.isin(Dd[c], list(ss)).mean(), 1))
    ax.set(xlabel="Time (ns)", ylabel=yl, title=f"{name.capitalize()} content vs time, {T}")
    ax.legend(ncol=2); fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_{name}_time.png"); plt.show()
    fig, ax = plt.subplots(figsize=(12,5), dpi=150)
    for c, s in sty.items():
        f = np.isin(Dd[c], list(ss)).mean(0); ax.plot(np.arange(1,NR+1), f, lw=1.4, **s)
        np.savetxt(f"{X}/dssp_{name}_perres_{c}.xvg", np.c_[np.arange(1,NR+1), f], fmt="%d %.4f")
    ax.set(xlabel="Residue", ylabel=f"Fraction of frames in {name}", title=f"{name.capitalize()} per residue, {T}")
    ax.legend(ncol=2); fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_{name}_perres.png"); plt.show()
fig, ax = plt.subplots(figsize=(12,5), dpi=150)
for c, s in sty.items(): ax.plot(t, 100*(1-np.isin(Dd[c], list("HGIEB")).mean(1)), lw=1.6, **s)
ax.set(xlabel="Time (ns)", ylabel="Coil/turn residues (%)", title=f"Coil/turn content vs time, {T}")
ax.legend(ncol=2); fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_coil_time.png"); plt.show()
cmap = ListedColormap(["#eeeeee", "#d62728", "#1f77b4"])
code = lambda m: np.where(np.isin(m, list("HGI")), 1, np.where(np.isin(m, list("EB")), 2, 0))
fig, axs = plt.subplots(6, 1, figsize=(12,13), dpi=150, sharex=True)
for ax, (c, s) in zip(axs, sty.items()):
    ax.imshow(code(Dd[c]), aspect="auto", cmap=cmap, vmin=0, vmax=2, origin="lower", extent=[1,NR,0,1])
    ax.set_ylabel("Time (ns)"); ax.set_title(s["label"], loc="left", fontsize=10)
axs[-1].set_xlabel("Residue"); fig.suptitle(f"DSSP map: red = helix, blue = strand, grey = coil/turn, {T}")
fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_map.png"); plt.show()

# ---- summary + all-states plots
def last(f, col=1): b = xvg(f); return b[len(b)//2:, col].mean()
h = len(t)//2; rows = []
for c in CH:
    rows.append(dict(chain=c, RMSD_nm=last(f"{X}/rmsd_{c}.xvg"), Rg_nm=last(f"{X}/rg_{c}.xvg"),
      SASA_nm2=last(f"{X}/sasa_{c}.xvg", 2), intra_Hbonds=last(f"{X}/hbnum_{c}.xvg"),
      helix_pct=100*np.isin(Dd[c][h:], list("HGI")).mean(), strand_pct=100*np.isin(Dd[c][h:], list("EB")).mean()))
rows.append(dict(chain="hexamer", RMSD_nm=last(f"{X}/rmsd_Protein.xvg"), Rg_nm=last(f"{X}/rg_Protein.xvg"),
                 SASA_nm2=last(f"{X}/sasa_Protein.xvg"), intra_Hbonds=last(f"{X}/hbnum_Protein.xvg")))
df = pd.DataFrame(rows).round(3); df.to_csv(f"{X}/summary_last0.5ns.csv", index=False); print(df)
STATES = [("Alpha helix","H"),("3-10 helix","G"),("Pi helix","I"),("Beta strand","E"),("Beta bridge","B"),
          ("Turn","T"),("Bend","S"),("Polyproline II helix","P")]
Pc = {k_: 100*(a == c_).mean(1) for k_, c_ in STATES}; Pc["Coil / Unstructured"] = 100 - sum(Pc.values())
tp = np.arange(len(a))*5.0
fig, ax = plt.subplots(figsize=(12,6.2), dpi=150)
for i, (k_, y) in enumerate(Pc.items()): ax.plot(tp, y, color=plt.cm.tab10(i), lw=1.6, label=k_)
ax.set(xlabel="Time (ps)", ylabel="Percentage (%)", title=f"Legumin All Secondary Structure States — {T}")
ax.grid(alpha=.3); ax.legend(title="Secondary structure", ncol=2, loc="upper right"); fig.tight_layout()
fig.savefig(f"{OUT}/graphs/dssp_all_states_300K_{RUN}.png"); plt.show()
Gp = {"Alpha helix": Pc["Alpha helix"], "Beta structure": Pc["Beta strand"]+Pc["Beta bridge"], "Turns": Pc["Turn"]}
Gp["Loops / Other"] = 100 - sum(Gp.values())
fig, ax = plt.subplots(figsize=(10,6), dpi=150)
for i, (k_, y) in enumerate(Gp.items()): ax.plot(tp, y, color=plt.cm.tab10(i), lw=1.8, label=k_)
ax.set(xlabel="Time (ps)", ylabel="Percentage (%)", title=f"Legumin Secondary Structure — {T}")
ax.grid(alpha=.3); ax.legend(); fig.tight_layout()
fig.savefig(f"{OUT}/graphs/dssp_groups_300K_{RUN}.png"); plt.show()
print("DONE ->", OUT)

In [ ]:
import os
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"): drive.mount('/content/drive')
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
os.system('ln -sfn "/content/drive/MyDrive/Plant Protein Extrusion MD" /content/pm')
os.system('''cd /content/pm
echo "== files"; ls -lh md_373_r1* npt_373_r1* 2>/dev/null
echo; echo "== md_373_r1.xtc"; gmx check -f md_373_r1.xtc 2>&1 | grep -iE "Natoms|Last frame|Step|Time"
echo; echo "== verified_copy.xtc"; gmx check -f md_373_r1_verified_copy.xtc 2>&1 | grep -iE "Natoms|Last frame|Step|Time"
echo; echo "== same content?"; cmp md_373_r1.xtc md_373_r1_verified_copy.xtc && echo IDENTICAL
echo; echo "== tpr / gro"; head -2 md_373_r1.gro | tail -1
grep -iE "^ *(tinit|dt|nsteps|ref_t|gen_temp|nstxout-compressed)" md_373_r1.mdp
echo; echo "== log tail"; tail -5 md_373_r1.log''')

In [ ]:
import os, subprocess
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"): drive.mount('/content/drive')
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
os.system('ln -sfn "/content/drive/MyDrive/Plant Protein Extrusion MD" /content/pm')
cmd = '''cd /content/pm
echo "== files"; ls -lh md_373_r1* npt_373_r1* 2>/dev/null
echo; echo "== md_373_r1.xtc"; gmx check -f md_373_r1.xtc 2>&1 | grep -iE "Natoms|Last frame|Step|Time"
echo; echo "== verified_copy.xtc"; gmx check -f md_373_r1_verified_copy.xtc 2>&1 | grep -iE "Natoms|Last frame|Step|Time"
echo; echo "== same content?"; cmp md_373_r1.xtc md_373_r1_verified_copy.xtc && echo IDENTICAL
echo; echo "== gro atoms"; head -2 md_373_r1.gro | tail -1
echo; echo "== mdp"; grep -iE "^ *(tinit|dt|nsteps|ref_t|gen_temp|nstxout-compressed)" md_373_r1.mdp
echo; echo "== log tail"; tail -5 md_373_r1.log'''
r = subprocess.run(["bash","-c",cmd], capture_output=True, text=True)
print(r.stdout); print(r.stderr[-500:])

In [ ]:
import os, sys, json, itertools, subprocess
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from google.colab import drive
TEMP, RUN = "373", "r1"                      # <- only this line changes per run
if not os.path.exists("/content/drive/MyDrive"): drive.mount('/content/drive')
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
os.system('ln -sfn "/content/drive/MyDrive/Plant Protein Extrusion MD" /content/pm')
S = f"md_{TEMP}_{RUN}"; OUT = f"/content/pm/analysism/legumin_{TEMP}_{RUN}"; T = f"{TEMP} K ({RUN})"
os.environ.update(P="/content/pm", S=S, OUT=OUT)
for d in ["graphs","logs","frames","xvgs"]: os.makedirs(f"{OUT}/{d}", exist_ok=True)
subprocess.run([sys.executable,"-m","pip","install","-q","MDAnalysis","scipy"])
import MDAnalysis as mda
from MDAnalysis.lib.distances import minimize_vectors
from MDAnalysis.analysis.align import rotation_matrix
from scipy.spatial import cKDTree
from matplotlib.colors import ListedColormap
def sh(cmd, tail=0):
    r = subprocess.run(["bash","-c",cmd], capture_output=True, text=True)
    o = (r.stdout + r.stderr).strip().split("\n")
    print("\n".join(o[-tail:] if tail else o))

# ---- check files + temperature
print("== files"); sh('ls -lh $P/$S.*; grep -iE "ref[-_]t|tcoupl|gen[-_]temp" $P/$S.mdp; gmx check -f "$P/$S.xtc" 2>&1 | grep -E "Last frame|^Step"')

# ---- protein tpr, whole, first frame
print("== whole")
sh(r'''cd "$OUT/frames"
printf "Protein\n" | gmx convert-tpr -s "$P/$S.tpr" -o protein.tpr 2>&1 | tail -1
printf "Protein\n" | gmx trjconv -s protein.tpr -f "$P/$S.xtc" -o whole2.xtc -pbc whole 2>&1 | tail -1
printf "Protein\n" | gmx trjconv -s protein.tpr -f whole2.xtc -o first_w.gro -dump 0 2>&1 | tail -1''')

# ---- chain detection from .gro
lines = open(f"{OUT}/frames/first_w.gro").read().split("\n")
n = int(lines[1]); body = lines[2:2+n]
resnum = np.array([int(l[0:5]) for l in body]); nm = [l[10:15].strip() for l in body]
starts = [0] + [i for i in range(1, n) if resnum[i] < resnum[i-1]]; ends = starts[1:] + [n]
nres = [len(set(resnum[s:e])) for s, e in zip(starts, ends)]
print("atoms:", n, "| chains:", len(starts), "| atoms/chain:", [e-s for s,e in zip(starts,ends)], "| res/chain:", nres)
assert len(starts) == 6, "expected 6 chains"
NR = nres[0]; CH = ["chA","chB","chC","chD","chE","chF"]; G = {}; ranges = {}
for k, (s, e) in enumerate(zip(starts, ends)):
    r = range(s+1, e+1); c = CH[k]; ranges[c] = [s, e]
    G[c] = list(r); G[c+"_bb"] = [i for i in r if nm[i-1] in ("N","CA","C")]; G[c+"_CA"] = [i for i in r if nm[i-1]=="CA"]
r = range(1, n+1)
G["Protein"] = list(r); G["Protein_bb"] = [i for i in r if nm[i-1] in ("N","CA","C")]; G["Protein_CA"] = [i for i in r if nm[i-1]=="CA"]
with open(f"{OUT}/frames/chains.ndx","w") as f:
    for g, idx in G.items():
        f.write(f"[ {g} ]\n")
        for k in range(0, len(idx), 15): f.write(" ".join(f"{i:5d}" for i in idx[k:k+15]) + "\n")
json.dump(dict(ranges=ranges, nres=NR, natoms=n), open(f"{OUT}/frames/chains.json","w"))

# ---- keep 6 chains together
sl = [slice(*ranges[c]) for c in CH]
u = mda.Universe(f"{OUT}/frames/first_w.gro", f"{OUT}/frames/whole2.xtc")
with mda.Writer(f"{OUT}/frames/together.xtc", u.atoms.n_atoms) as W:
    for ts in u.trajectory[::5]:
        pos = u.atoms.positions.copy(); box = ts.dimensions.astype(np.float32); a0 = pos[sl[0]].mean(0)
        for s in sl[1:]:
            d = pos[s].mean(0) - a0
            dm = minimize_vectors(d[None,:].astype(np.float32), box)[0]; pos[s] -= (d - dm)
        u.atoms.positions = pos; W.write(u.atoms)

# ---- fit
print("== fit")
sh(r'''cd "$OUT/frames"
printf "Backbone\nProtein\n" | gmx trjconv -s protein.tpr -f together.xtc -o fit_cluster_skip5.xtc -fit rot+trans 2>&1 | tail -1
printf "Protein\n" | gmx trjconv -s protein.tpr -f fit_cluster_skip5.xtc -o first_cl.gro -dump 0 2>&1 | tail -1
rm -f whole2.xtc together.xtc
gmx check -f fit_cluster_skip5.xtc 2>&1 | grep -E "Last frame|^Step"''')

# ---- hexamer integrity check
u = mda.Universe(f"{OUT}/frames/first_cl.gro", f"{OUT}/frames/fit_cluster_skip5.xtc")
pairs = list(itertools.combinations(range(6), 2)); worst = {p: 0.0 for p in pairs}; lonely = 0
for ts in u.trajectory:
    Pp = u.atoms.positions/10; ch = [Pp[s] for s in sl]
    m = {p: cKDTree(ch[p[0]]).query(ch[p[1]])[0].min() for p in pairs}
    for p in pairs: worst[p] = max(worst[p], m[p])
    for i in range(6):
        if min(m[p] for p in pairs if i in p) > 0.5: lonely += 1
print({"ABCDEF"[a]+"ABCDEF"[b]: round(float(v),2) for (a,b), v in worst.items()})
print("chains with no neighbour within 0.5 nm:", lonely)
assert lonely == 0, "a chain drifted away"

# ---- GROMACS analysis
print("== gmx analysis (slow)")
sh(r'''cd "$OUT/frames"; X="$OUT/xvgs"; L="$OUT/logs"; T=fit_cluster_skip5.xtc
for c in Protein chA chB chC chD chE chF; do
  printf "${c}_bb\n${c}_bb\n" | gmx rms  -s first_cl.gro -f $T -n chains.ndx -o $X/rmsd_$c.xvg -tu ns > $L/rmsd_$c.log 2>&1
  printf "${c}_CA\n${c}_CA\n" | gmx rmsf -s first_cl.gro -f $T -n chains.ndx -o $X/rmsf_$c.xvg -res   > $L/rmsf_$c.log 2>&1
  printf "$c\n"               | gmx gyrate -s protein.tpr -f $T -n chains.ndx -o $X/rg_$c.xvg          > $L/rg_$c.log 2>&1
  gmx sasa -s protein.tpr -f $T -n chains.ndx -surface "group Protein" -output "group $c" -o $X/sasa_$c.xvg -or $X/tmp_$c.xvg -tu ns > $L/sasa_$c.log 2>&1
  gmx hbond -s protein.tpr -f $T -n chains.ndx -r "group $c" -t "group $c" -num $X/hbnum_$c.xvg -tu ns > $L/hbond_$c.log 2>&1
done
for p in "chA chB" "chA chC" "chA chD" "chA chE" "chA chF" "chB chC" "chB chD" "chB chE" "chB chF" "chC chD" "chC chE" "chC chF" "chD chE" "chD chF" "chE chF"; do set -- $p
  gmx hbond -s protein.tpr -f $T -n chains.ndx -r "group $1" -t "group $2" -num $X/hbinter_$1_$2.xvg -tu ns > $L/hbinter_$1_$2.log 2>&1
done
mv $X/tmp_Protein.xvg $X/sasa_perres_Protein.xvg; rm -f $X/tmp_ch*.xvg
for f in rmsd_Protein rmsd_chA rmsf_chA rg_Protein sasa_chA hbnum_Protein hbinter_chA_chB; do echo $f $(tail -1 $X/$f.xvg); done''')

# ---- plots
X = f"{OUT}/xvgs"
cols = ["#d62728","#ff9896","#1f77b4","#7fbfff","#2ca02c","#98df8a"]
sty = {c: dict(color=cols[i], ls="-" if i%2==0 else ":", label="Chain "+c[-1]) for i, c in enumerate(CH)}
xvg = lambda f: np.loadtxt(f, comments=('#','@'))
def draw(out, title, xl, yl, items):
    fig, ax = plt.subplots(figsize=(12,5), dpi=150)
    for x, y, s in items: ax.plot(x, y, lw=1.6, **s)
    ax.set(xlabel=xl, ylabel=yl, title=title); ax.legend(ncol=2); fig.tight_layout()
    fig.savefig(f"{OUT}/graphs/{out}.png"); plt.show()
chains = lambda pre, col=1, sc=1: [(xvg(f"{X}/{pre}_{c}.xvg")[:,0]/sc, xvg(f"{X}/{pre}_{c}.xvg")[:,col], s) for c, s in sty.items()]
def whole(pre, col=1, sc=1):
    a = xvg(f"{X}/{pre}_Protein.xvg"); return [(a[:,0]/sc, a[:,col], dict(color="grey", label="Whole hexamer"))]
for pre, yl, ttl, sc in (("rmsd","RMSD (nm)","Backbone RMSD",1), ("rg","Rg (nm)","Radius of gyration",1000),
                         ("sasa","SASA (nm²)","SASA",1), ("hbnum","H-bonds","Intra-chain H-bonds",1)):
    cc = 2 if pre == "sasa" else 1
    draw(f"{pre}_per_chain", f"{ttl} per chain, {T}", "Time (ns)", yl, chains(pre, cc, sc))
    draw(f"{pre}_protein", f"{ttl}, whole hexamer, {T}", "Time (ns)", yl, whole(pre, 1, sc))
draw("rmsf_per_chain", f"C-alpha RMSF per residue, {T}", "Residue", "RMSF (nm)", chains("rmsf"))
d = xvg(f"{X}/sasa_perres_Protein.xvg"); assert len(d) == 6*NR, len(d)
draw("sasa_per_residue", f"SASA per residue, {T}", "Residue", "SASA (nm²)",
     [(np.arange(1,NR+1), d[i*NR:(i+1)*NR,1], s) for i, s in enumerate(sty.values())])
fig, ax = plt.subplots(figsize=(12,6), dpi=150)
for k, (a_, b_) in enumerate(itertools.combinations(CH, 2)):
    h = xvg(f"{X}/hbinter_{a_}_{b_}.xvg"); ax.plot(h[:,0], h[:,1], lw=1.2, color=plt.cm.tab20(k), label=f"{a_[-1]}–{b_[-1]}")
ax.set(xlabel="Time (ns)", ylabel="H-bonds", title=f"Inter-chain H-bonds, {T}"); ax.legend(ncol=3, fontsize=8)
fig.tight_layout(); fig.savefig(f"{OUT}/graphs/hbonds_inter_chain.png"); plt.show()

# ---- per-residue RMSD and H-bonds
u = mda.Universe(f"{OUT}/frames/first_cl.gro", f"{OUT}/frames/fit_cluster_skip5.xtc")
NA = u.atoms.n_atoms // 6; cs = {c: slice(i*NA, (i+1)*NA) for i, c in enumerate(CH)}; resR = {}
for name, s in cs.items():
    chn = u.atoms[s]; bb = chn.select_atoms("name N CA C"); ca = chn.select_atoms("name CA")
    u.trajectory[0]; c0 = bb.positions.mean(0); bb0 = bb.positions - c0; ca0 = ca.positions - c0
    d2 = np.zeros(len(ca)); k = 0
    for ts in u.trajectory:
        c = bb.positions.mean(0); R, _ = rotation_matrix(bb.positions - c, bb0)
        d2 += (((ca.positions - c) @ R.T - ca0)**2).sum(1); k += 1
    resR[name] = np.sqrt(d2/k)/10
    np.savetxt(f"{X}/rmsd_perres_{name}.xvg", np.c_[np.arange(1,NR+1), resR[name]], fmt="%d %.6f")
draw("rmsd_per_residue", f"C-alpha RMSD per residue vs first frame, {T}", "Residue", "RMSD (nm)",
     [(np.arange(1,NR+1), resR[c], s) for c, s in sty.items()])
draw("rmsd_per_residue_mean", f"Mean C-alpha RMSD per residue (6 chains), {T}", "Residue", "RMSD (nm)",
     [(np.arange(1,NR+1), np.mean(list(resR.values()), axis=0), dict(color="grey", label="Mean of chains"))])

nmA = np.array([x.strip() for x in u.atoms.names]); rn = u.atoms.resnames
chain = np.arange(len(nmA)) // NA; ridx = u.atoms.resindices - chain*NR
assert ridx.min() == 0 and ridx.max() == NR-1
isH = np.array([x.startswith("H") for x in nmA])
P0 = u.atoms.positions/10; heavy = np.where(~isH)[0]; Hs = np.where(isH)[0]
don = heavy[cKDTree(P0[heavy]).query(P0[Hs])[1]]
kk = np.array([nmA[i][0] in "NO" for i in don]); Hs, don = Hs[kk], don[kk]
acc = np.array([i for i in range(len(nmA)) if nmA[i][0]=="O" or (nmA[i] in ("ND1","NE2") and rn[i].startswith("HI"))])
tot = np.zeros((6,NR)); inter = np.zeros((6,NR)); nb = []
for ts in u.trajectory:
    Pf = u.atoms.positions/10; k = 0
    for hi, lst in enumerate(cKDTree(Pf[Hs]).query_ball_tree(cKDTree(Pf[acc]), 0.25)):
        D_ = don[hi]
        for aj in lst:
            A = acc[aj]
            if A == D_ or (chain[A]==chain[D_] and ridx[A]==ridx[D_]): continue
            v1, v2 = Pf[Hs[hi]]-Pf[D_], Pf[A]-Pf[D_]; dist = np.linalg.norm(v2)
            if dist > 0.35 or v1@v2/(np.linalg.norm(v1)*dist) < np.cos(np.radians(30)): continue
            k += 1
            for Xa in (D_, A):
                tot[chain[Xa], ridx[Xa]] += 1
                if chain[D_] != chain[A]: inter[chain[Xa], ridx[Xa]] += 1
    nb.append(k)
nf = len(nb); print("mean H-bonds/frame:", round(float(np.mean(nb)),1))
for nmx, arr, ttl in (("hbperres",tot,"All H-bonds per residue"), ("hbperres_inter",inter,"Inter-chain H-bonds per residue")):
    for i, c in enumerate(CH): np.savetxt(f"{X}/{nmx}_{c}.xvg", np.c_[np.arange(1,NR+1), arr[i]/nf], fmt="%d %.5f")
    draw(nmx, f"{ttl}, {T}", "Residue", "H-bonds per frame", [(np.arange(1,NR+1), arr[i]/nf, s) for i, s in enumerate(sty.values())])

# ---- DSSP
print("== dssp")
sh(r'''cd "$OUT/frames"
printf "Protein\n" | gmx dssp -s protein.tpr -f fit_cluster_skip5.xtc -n chains.ndx -o "$OUT/xvgs/dssp_Protein.dat" > "$OUT/logs/dssp_Protein.log" 2>&1
cp "$OUT/xvgs/dssp_Protein.dat" "$OUT/xvgs/dssp_chA.dat"
head -c 80 "$OUT/xvgs/dssp_Protein.dat"; echo''')
a = np.array([list(l.rstrip("\n")) for l in open(f"{X}/dssp_Protein.dat") if l.strip()])
a = a[:, ~(a == "=").all(0)]; print("dssp shape:", a.shape); assert a.shape[1] == 6*NR, a.shape
Dd = {c: a[:, i*NR:(i+1)*NR] for i, c in enumerate(CH)}; t = np.arange(len(a))*0.005
for name, ss, yl in (("helix","HGI","Helix residues (%)"), ("strand","EB","Strand residues (%)")):
    fig, ax = plt.subplots(figsize=(12,5), dpi=150)
    for c, s in sty.items():
        ax.plot(t, 100*np.isin(Dd[c], list(ss)).mean(1), lw=1.6, **s)
        print(name, c, "mean %:", round(100*np.isin(Dd[c], list(ss)).mean(), 1))
    ax.set(xlabel="Time (ns)", ylabel=yl, title=f"{name.capitalize()} content vs time, {T}")
    ax.legend(ncol=2); fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_{name}_time.png"); plt.show()
    fig, ax = plt.subplots(figsize=(12,5), dpi=150)
    for c, s in sty.items():
        f = np.isin(Dd[c], list(ss)).mean(0); ax.plot(np.arange(1,NR+1), f, lw=1.4, **s)
        np.savetxt(f"{X}/dssp_{name}_perres_{c}.xvg", np.c_[np.arange(1,NR+1), f], fmt="%d %.4f")
    ax.set(xlabel="Residue", ylabel=f"Fraction of frames in {name}", title=f"{name.capitalize()} per residue, {T}")
    ax.legend(ncol=2); fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_{name}_perres.png"); plt.show()
fig, ax = plt.subplots(figsize=(12,5), dpi=150)
for c, s in sty.items(): ax.plot(t, 100*(1-np.isin(Dd[c], list("HGIEB")).mean(1)), lw=1.6, **s)
ax.set(xlabel="Time (ns)", ylabel="Coil/turn residues (%)", title=f"Coil/turn content vs time, {T}")
ax.legend(ncol=2); fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_coil_time.png"); plt.show()
cmap = ListedColormap(["#eeeeee", "#d62728", "#1f77b4"])
code = lambda m: np.where(np.isin(m, list("HGI")), 1, np.where(np.isin(m, list("EB")), 2, 0))
fig, axs = plt.subplots(6, 1, figsize=(12,13), dpi=150, sharex=True)
for ax, (c, s) in zip(axs, sty.items()):
    ax.imshow(code(Dd[c]), aspect="auto", cmap=cmap, vmin=0, vmax=2, origin="lower", extent=[1,NR,0,1])
    ax.set_ylabel("Time (ns)"); ax.set_title(s["label"], loc="left", fontsize=10)
axs[-1].set_xlabel("Residue"); fig.suptitle(f"DSSP map: red = helix, blue = strand, grey = coil/turn, {T}")
fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_map.png"); plt.show()

# ---- summary + all-states plots
def last(f, col=1): b = xvg(f); return b[len(b)//2:, col].mean()
h = len(t)//2; rows = []
for c in CH:
    rows.append(dict(chain=c, RMSD_nm=last(f"{X}/rmsd_{c}.xvg"), Rg_nm=last(f"{X}/rg_{c}.xvg"),
      SASA_nm2=last(f"{X}/sasa_{c}.xvg", 2), intra_Hbonds=last(f"{X}/hbnum_{c}.xvg"),
      helix_pct=100*np.isin(Dd[c][h:], list("HGI")).mean(), strand_pct=100*np.isin(Dd[c][h:], list("EB")).mean()))
rows.append(dict(chain="hexamer", RMSD_nm=last(f"{X}/rmsd_Protein.xvg"), Rg_nm=last(f"{X}/rg_Protein.xvg"),
                 SASA_nm2=last(f"{X}/sasa_Protein.xvg"), intra_Hbonds=last(f"{X}/hbnum_Protein.xvg")))
df = pd.DataFrame(rows).round(3); df.to_csv(f"{X}/summary_last0.5ns.csv", index=False); print(df)
STATES = [("Alpha helix","H"),("3-10 helix","G"),("Pi helix","I"),("Beta strand","E"),("Beta bridge","B"),
          ("Turn","T"),("Bend","S"),("Polyproline II helix","P")]
Pc = {k_: 100*(a == c_).mean(1) for k_, c_ in STATES}; Pc["Coil / Unstructured"] = 100 - sum(Pc.values())
tp = np.arange(len(a))*5.0
fig, ax = plt.subplots(figsize=(12,6.2), dpi=150)
for i, (k_, y) in enumerate(Pc.items()): ax.plot(tp, y, color=plt.cm.tab10(i), lw=1.6, label=k_)
ax.set(xlabel="Time (ps)", ylabel="Percentage (%)", title=f"Legumin All Secondary Structure States — {T}")
ax.grid(alpha=.3); ax.legend(title="Secondary structure", ncol=2, loc="upper right"); fig.tight_layout()
fig.savefig(f"{OUT}/graphs/dssp_all_states_{TEMP}K_{RUN}.png"); plt.show()
Gp = {"Alpha helix": Pc["Alpha helix"], "Beta structure": Pc["Beta strand"]+Pc["Beta bridge"], "Turns": Pc["Turn"]}
Gp["Loops / Other"] = 100 - sum(Gp.values())
fig, ax = plt.subplots(figsize=(10,6), dpi=150)
for i, (k_, y) in enumerate(Gp.items()): ax.plot(tp, y, color=plt.cm.tab10(i), lw=1.8, label=k_)
ax.set(xlabel="Time (ps)", ylabel="Percentage (%)", title=f"Legumin Secondary Structure — {T}")
ax.grid(alpha=.3); ax.legend(); fig.tight_layout()
fig.savefig(f"{OUT}/graphs/dssp_groups_{TEMP}K_{RUN}.png"); plt.show()
print("DONE ->", OUT)

In [ ]:
import os, subprocess
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"): drive.mount('/content/drive')
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
cmd = r'''
echo "== where are the shear files"
find /content/drive/MyDrive -name "md_373_shear_r1*" -not -name "*.cpt*" 2>/dev/null | while read f; do ls -lh "$f"; done
D=$(dirname "$(find /content/drive/MyDrive -name md_373_shear_r1_combined.xtc 2>/dev/null | head -1)")
echo; echo "FOLDER: $D"; cd "$D"
for x in md_373_shear_r1_combined.xtc md_373_shear_r1.part0002.xtc md_373_shear_r1.xtc; do
  [ -f "$x" ] && { echo; echo "== $x"; gmx check -f "$x" 2>&1 | grep -E "Natoms|^Last frame|^Step|^Time"; }
done
echo; echo "== mdp (shear / temperature)"
grep -iE "deform|ref[-_]t|tcoupl|nsteps|^ *dt|nstxout-compressed|pbc" md_373_shear_r1.mdp
echo; echo "== atoms in tpr/gro"; head -2 md_373_shear_r1_final.gro | tail -1
echo; echo "== log tails"; tail -4 md_373_shear_r1.log; echo ---; tail -4 md_373_shear_r1.part0002.log
'''
r = subprocess.run(["bash","-c",cmd], capture_output=True, text=True)
print(r.stdout); print(r.stderr[-500:])

In [ ]:
import os, sys, json, itertools, subprocess
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from google.colab import drive
TEMP, RUN = "373", "shear_r1"
TPR_NAME  = "md_373_shear_r1"                 # tpr
TRAJ_NAME = "md_373_shear_r1_combined"        # full trajectory
if not os.path.exists("/content/drive/MyDrive"): drive.mount('/content/drive')
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
os.system('ln -sfn "/content/drive/MyDrive/Plant Protein Extrusion MD" /content/pm')
OUT = f"/content/pm/analysism/legumin_{TEMP}_{RUN}"; T = "373 K shear (r1)"
os.environ.update(P="/content/pm", S=TPR_NAME, TRAJ=TRAJ_NAME, OUT=OUT)
for d in ["graphs","logs","frames","xvgs"]: os.makedirs(f"{OUT}/{d}", exist_ok=True)
subprocess.run([sys.executable,"-m","pip","install","-q","MDAnalysis","scipy"])
import MDAnalysis as mda
from MDAnalysis.coordinates.XTC import XTCReader
from MDAnalysis.lib.distances import minimize_vectors
from MDAnalysis.analysis.align import rotation_matrix
from scipy.spatial import cKDTree
from matplotlib.colors import ListedColormap
def sh(cmd, tail=0):
    r = subprocess.run(["bash","-c",cmd], capture_output=True, text=True)
    o = (r.stdout + r.stderr).strip().split("\n")
    print("\n".join(o[-tail:] if tail else o))

# ---- check files, mdp, and trajectory continuity
print("== files"); sh('ls -lh $P/$S.tpr $P/$TRAJ.xtc; grep -iE "ref[-_]t|deform" $P/$S.mdp')
rd = XTCReader(f"{OUT.rsplit('/analysism',1)[0]}/{TRAJ_NAME}.xtc")
times = np.array([ts.time for ts in rd]); dims = np.array([ts.dimensions.copy() for ts in rd])
print("frames:", len(times), "| time", times[0], "->", times[-1], "| natoms:", rd.n_atoms)
assert times[0] == 0 and abs(times[-1]-1000) < 1e-3 and len(times) == 1001 and np.all(np.diff(times) == 1), "trajectory not continuous 0-1000 ps"
print("box a,b,c start:", dims[0,:3].round(2), "end:", dims[-1,:3].round(2))
print("angles start:", dims[0,3:].round(2), "end:", dims[-1,3:].round(2))
gd = 3.644/dims[0,1]; print(f"shear rate {gd:.4f} /ns -> expected rigid rotation {-0.5*gd*180/np.pi:.2f} deg/ns about z")

# ---- protein tpr, whole, first frame
print("== whole")
sh(r'''cd "$OUT/frames"
printf "Protein\n" | gmx convert-tpr -s "$P/$S.tpr" -o protein.tpr 2>&1 | tail -1
printf "Protein\n" | gmx trjconv -s protein.tpr -f "$P/$TRAJ.xtc" -o whole2.xtc -pbc whole 2>&1 | tail -1
printf "Protein\n" | gmx trjconv -s protein.tpr -f whole2.xtc -o first_w.gro -dump 0 2>&1 | tail -1''')

# ---- chain detection from .gro
lines = open(f"{OUT}/frames/first_w.gro").read().split("\n")
n = int(lines[1]); body = lines[2:2+n]
resnum = np.array([int(l[0:5]) for l in body]); nm = [l[10:15].strip() for l in body]
starts = [0] + [i for i in range(1, n) if resnum[i] < resnum[i-1]]; ends = starts[1:] + [n]
nres = [len(set(resnum[s:e])) for s, e in zip(starts, ends)]
print("atoms:", n, "| chains:", len(starts), "| atoms/chain:", [e-s for s,e in zip(starts,ends)], "| res/chain:", nres)
assert len(starts) == 6, "expected 6 chains"
NR = nres[0]; CH = ["chA","chB","chC","chD","chE","chF"]; G = {}; ranges = {}
for k, (s, e) in enumerate(zip(starts, ends)):
    r = range(s+1, e+1); c = CH[k]; ranges[c] = [s, e]
    G[c] = list(r); G[c+"_bb"] = [i for i in r if nm[i-1] in ("N","CA","C")]; G[c+"_CA"] = [i for i in r if nm[i-1]=="CA"]
r = range(1, n+1)
G["Protein"] = list(r); G["Protein_bb"] = [i for i in r if nm[i-1] in ("N","CA","C")]; G["Protein_CA"] = [i for i in r if nm[i-1]=="CA"]
with open(f"{OUT}/frames/chains.ndx","w") as f:
    for g, idx in G.items():
        f.write(f"[ {g} ]\n")
        for k in range(0, len(idx), 15): f.write(" ".join(f"{i:5d}" for i in idx[k:k+15]) + "\n")
json.dump(dict(ranges=ranges, nres=NR, natoms=n), open(f"{OUT}/frames/chains.json","w"))

# ---- keep 6 chains together (triclinic-safe)
sl = [slice(*ranges[c]) for c in CH]
u = mda.Universe(f"{OUT}/frames/first_w.gro", f"{OUT}/frames/whole2.xtc")
with mda.Writer(f"{OUT}/frames/together.xtc", u.atoms.n_atoms) as W:
    for ts in u.trajectory[::5]:
        pos = u.atoms.positions.copy(); box = ts.dimensions.astype(np.float32); a0 = pos[sl[0]].mean(0)
        for s in sl[1:]:
            d = pos[s].mean(0) - a0
            dm = minimize_vectors(d[None,:].astype(np.float32), box)[0]; pos[s] -= (d - dm)
        u.atoms.positions = pos; W.write(u.atoms)

# ---- fit
print("== fit")
sh(r'''cd "$OUT/frames"
printf "Backbone\nProtein\n" | gmx trjconv -s protein.tpr -f together.xtc -o fit_cluster_skip5.xtc -fit rot+trans 2>&1 | tail -1
printf "Protein\n" | gmx trjconv -s protein.tpr -f fit_cluster_skip5.xtc -o first_cl.gro -dump 0 2>&1 | tail -1
rm -f whole2.xtc together.xtc
gmx check -f fit_cluster_skip5.xtc 2>&1 | grep -E "Last frame|^Step"''')

# ---- hexamer integrity check
u = mda.Universe(f"{OUT}/frames/first_cl.gro", f"{OUT}/frames/fit_cluster_skip5.xtc")
pairs = list(itertools.combinations(range(6), 2)); worst = {p: 0.0 for p in pairs}; lonely = 0
for ts in u.trajectory:
    Pp = u.atoms.positions/10; ch = [Pp[s] for s in sl]
    m = {p: cKDTree(ch[p[0]]).query(ch[p[1]])[0].min() for p in pairs}
    for p in pairs: worst[p] = max(worst[p], m[p])
    for i in range(6):
        if min(m[p] for p in pairs if i in p) > 0.5: lonely += 1
print({"ABCDEF"[a]+"ABCDEF"[b]: round(float(v),2) for (a,b), v in worst.items()})
print("chains with no neighbour within 0.5 nm:", lonely)
assert lonely == 0, "a chain drifted away"

# ---- GROMACS analysis
print("== gmx analysis (slow)")
sh(r'''cd "$OUT/frames"; X="$OUT/xvgs"; L="$OUT/logs"; T=fit_cluster_skip5.xtc
for c in Protein chA chB chC chD chE chF; do
  printf "${c}_bb\n${c}_bb\n" | gmx rms  -s first_cl.gro -f $T -n chains.ndx -o $X/rmsd_$c.xvg -tu ns > $L/rmsd_$c.log 2>&1
  printf "${c}_CA\n${c}_CA\n" | gmx rmsf -s first_cl.gro -f $T -n chains.ndx -o $X/rmsf_$c.xvg -res   > $L/rmsf_$c.log 2>&1
  printf "$c\n"               | gmx gyrate -s protein.tpr -f $T -n chains.ndx -o $X/rg_$c.xvg          > $L/rg_$c.log 2>&1
  gmx sasa -s protein.tpr -f $T -n chains.ndx -surface "group Protein" -output "group $c" -o $X/sasa_$c.xvg -or $X/tmp_$c.xvg -tu ns > $L/sasa_$c.log 2>&1
  gmx hbond -s protein.tpr -f $T -n chains.ndx -r "group $c" -t "group $c" -num $X/hbnum_$c.xvg -tu ns > $L/hbond_$c.log 2>&1
done
for p in "chA chB" "chA chC" "chA chD" "chA chE" "chA chF" "chB chC" "chB chD" "chB chE" "chB chF" "chC chD" "chC chE" "chC chF" "chD chE" "chD chF" "chE chF"; do set -- $p
  gmx hbond -s protein.tpr -f $T -n chains.ndx -r "group $1" -t "group $2" -num $X/hbinter_$1_$2.xvg -tu ns > $L/hbinter_$1_$2.log 2>&1
done
mv $X/tmp_Protein.xvg $X/sasa_perres_Protein.xvg; rm -f $X/tmp_ch*.xvg
for f in rmsd_Protein rmsd_chA rmsf_chA rg_Protein sasa_chA hbnum_Protein hbinter_chA_chB; do echo $f $(tail -1 $X/$f.xvg); done''')

# ---- plots
X = f"{OUT}/xvgs"
cols = ["#d62728","#ff9896","#1f77b4","#7fbfff","#2ca02c","#98df8a"]
sty = {c: dict(color=cols[i], ls="-" if i%2==0 else ":", label="Chain "+c[-1]) for i, c in enumerate(CH)}
xvg = lambda f: np.loadtxt(f, comments=('#','@'))
def draw(out, title, xl, yl, items):
    fig, ax = plt.subplots(figsize=(12,5), dpi=150)
    for x, y, s in items: ax.plot(x, y, lw=1.6, **s)
    ax.set(xlabel=xl, ylabel=yl, title=title); ax.legend(ncol=2); fig.tight_layout()
    fig.savefig(f"{OUT}/graphs/{out}.png"); plt.show()
chains = lambda pre, col=1, sc=1: [(xvg(f"{X}/{pre}_{c}.xvg")[:,0]/sc, xvg(f"{X}/{pre}_{c}.xvg")[:,col], s) for c, s in sty.items()]
def whole(pre, col=1, sc=1):
    a = xvg(f"{X}/{pre}_Protein.xvg"); return [(a[:,0]/sc, a[:,col], dict(color="grey", label="Whole hexamer"))]
for pre, yl, ttl, sc in (("rmsd","RMSD (nm)","Backbone RMSD",1), ("rg","Rg (nm)","Radius of gyration",1000),
                         ("sasa","SASA (nm²)","SASA",1), ("hbnum","H-bonds","Intra-chain H-bonds",1)):
    cc = 2 if pre == "sasa" else 1
    draw(f"{pre}_per_chain", f"{ttl} per chain, {T}", "Time (ns)", yl, chains(pre, cc, sc))
    draw(f"{pre}_protein", f"{ttl}, whole hexamer, {T}", "Time (ns)", yl, whole(pre, 1, sc))
draw("rmsf_per_chain", f"C-alpha RMSF per residue, {T}", "Residue", "RMSF (nm)", chains("rmsf"))
d = xvg(f"{X}/sasa_perres_Protein.xvg"); assert len(d) == 6*NR, len(d)
draw("sasa_per_residue", f"SASA per residue, {T}", "Residue", "SASA (nm²)",
     [(np.arange(1,NR+1), d[i*NR:(i+1)*NR,1], s) for i, s in enumerate(sty.values())])
fig, ax = plt.subplots(figsize=(12,6), dpi=150)
for k, (a_, b_) in enumerate(itertools.combinations(CH, 2)):
    h = xvg(f"{X}/hbinter_{a_}_{b_}.xvg"); ax.plot(h[:,0], h[:,1], lw=1.2, color=plt.cm.tab20(k), label=f"{a_[-1]}–{b_[-1]}")
ax.set(xlabel="Time (ns)", ylabel="H-bonds", title=f"Inter-chain H-bonds, {T}"); ax.legend(ncol=3, fontsize=8)
fig.tight_layout(); fig.savefig(f"{OUT}/graphs/hbonds_inter_chain.png"); plt.show()

# ---- per-residue RMSD and H-bonds
u = mda.Universe(f"{OUT}/frames/first_cl.gro", f"{OUT}/frames/fit_cluster_skip5.xtc")
NA = u.atoms.n_atoms // 6; cs = {c: slice(i*NA, (i+1)*NA) for i, c in enumerate(CH)}; resR = {}
for name, s in cs.items():
    chn = u.atoms[s]; bb = chn.select_atoms("name N CA C"); ca = chn.select_atoms("name CA")
    u.trajectory[0]; c0 = bb.positions.mean(0); bb0 = bb.positions - c0; ca0 = ca.positions - c0
    d2 = np.zeros(len(ca)); k = 0
    for ts in u.trajectory:
        c = bb.positions.mean(0); R, _ = rotation_matrix(bb.positions - c, bb0)
        d2 += (((ca.positions - c) @ R.T - ca0)**2).sum(1); k += 1
    resR[name] = np.sqrt(d2/k)/10
    np.savetxt(f"{X}/rmsd_perres_{name}.xvg", np.c_[np.arange(1,NR+1), resR[name]], fmt="%d %.6f")
draw("rmsd_per_residue", f"C-alpha RMSD per residue vs first frame, {T}", "Residue", "RMSD (nm)",
     [(np.arange(1,NR+1), resR[c], s) for c, s in sty.items()])
draw("rmsd_per_residue_mean", f"Mean C-alpha RMSD per residue (6 chains), {T}", "Residue", "RMSD (nm)",
     [(np.arange(1,NR+1), np.mean(list(resR.values()), axis=0), dict(color="grey", label="Mean of chains"))])

nmA = np.array([x.strip() for x in u.atoms.names]); rn = u.atoms.resnames
chain = np.arange(len(nmA)) // NA; ridx = u.atoms.resindices - chain*NR
assert ridx.min() == 0 and ridx.max() == NR-1
isH = np.array([x.startswith("H") for x in nmA])
P0 = u.atoms.positions/10; heavy = np.where(~isH)[0]; Hs = np.where(isH)[0]
don = heavy[cKDTree(P0[heavy]).query(P0[Hs])[1]]
kk = np.array([nmA[i][0] in "NO" for i in don]); Hs, don = Hs[kk], don[kk]
acc = np.array([i for i in range(len(nmA)) if nmA[i][0]=="O" or (nmA[i] in ("ND1","NE2") and rn[i].startswith("HI"))])
tot = np.zeros((6,NR)); inter = np.zeros((6,NR)); nb = []
for ts in u.trajectory:
    Pf = u.atoms.positions/10; k = 0
    for hi, lst in enumerate(cKDTree(Pf[Hs]).query_ball_tree(cKDTree(Pf[acc]), 0.25)):
        D_ = don[hi]
        for aj in lst:
            A = acc[aj]
            if A == D_ or (chain[A]==chain[D_] and ridx[A]==ridx[D_]): continue
            v1, v2 = Pf[Hs[hi]]-Pf[D_], Pf[A]-Pf[D_]; dist = np.linalg.norm(v2)
            if dist > 0.35 or v1@v2/(np.linalg.norm(v1)*dist) < np.cos(np.radians(30)): continue
            k += 1
            for Xa in (D_, A):
                tot[chain[Xa], ridx[Xa]] += 1
                if chain[D_] != chain[A]: inter[chain[Xa], ridx[Xa]] += 1
    nb.append(k)
nf = len(nb); print("mean H-bonds/frame:", round(float(np.mean(nb)),1))
for nmx, arr, ttl in (("hbperres",tot,"All H-bonds per residue"), ("hbperres_inter",inter,"Inter-chain H-bonds per residue")):
    for i, c in enumerate(CH): np.savetxt(f"{X}/{nmx}_{c}.xvg", np.c_[np.arange(1,NR+1), arr[i]/nf], fmt="%d %.5f")
    draw(nmx, f"{ttl}, {T}", "Residue", "H-bonds per frame", [(np.arange(1,NR+1), arr[i]/nf, s) for i, s in enumerate(sty.values())])

# ---- DSSP
print("== dssp")
sh(r'''cd "$OUT/frames"
printf "Protein\n" | gmx dssp -s protein.tpr -f fit_cluster_skip5.xtc -n chains.ndx -o "$OUT/xvgs/dssp_Protein.dat" > "$OUT/logs/dssp_Protein.log" 2>&1
cp "$OUT/xvgs/dssp_Protein.dat" "$OUT/xvgs/dssp_chA.dat"
head -c 80 "$OUT/xvgs/dssp_Protein.dat"; echo''')
a = np.array([list(l.rstrip("\n")) for l in open(f"{X}/dssp_Protein.dat") if l.strip()])
a = a[:, ~(a == "=").all(0)]; print("dssp shape:", a.shape); assert a.shape[1] == 6*NR, a.shape
Dd = {c: a[:, i*NR:(i+1)*NR] for i, c in enumerate(CH)}; t = np.arange(len(a))*0.005
for name, ss, yl in (("helix","HGI","Helix residues (%)"), ("strand","EB","Strand residues (%)")):
    fig, ax = plt.subplots(figsize=(12,5), dpi=150)
    for c, s in sty.items():
        ax.plot(t, 100*np.isin(Dd[c], list(ss)).mean(1), lw=1.6, **s)
        print(name, c, "mean %:", round(100*np.isin(Dd[c], list(ss)).mean(), 1))
    ax.set(xlabel="Time (ns)", ylabel=yl, title=f"{name.capitalize()} content vs time, {T}")
    ax.legend(ncol=2); fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_{name}_time.png"); plt.show()
    fig, ax = plt.subplots(figsize=(12,5), dpi=150)
    for c, s in sty.items():
        f = np.isin(Dd[c], list(ss)).mean(0); ax.plot(np.arange(1,NR+1), f, lw=1.4, **s)
        np.savetxt(f"{X}/dssp_{name}_perres_{c}.xvg", np.c_[np.arange(1,NR+1), f], fmt="%d %.4f")
    ax.set(xlabel="Residue", ylabel=f"Fraction of frames in {name}", title=f"{name.capitalize()} per residue, {T}")
    ax.legend(ncol=2); fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_{name}_perres.png"); plt.show()
fig, ax = plt.subplots(figsize=(12,5), dpi=150)
for c, s in sty.items(): ax.plot(t, 100*(1-np.isin(Dd[c], list("HGIEB")).mean(1)), lw=1.6, **s)
ax.set(xlabel="Time (ns)", ylabel="Coil/turn residues (%)", title=f"Coil/turn content vs time, {T}")
ax.legend(ncol=2); fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_coil_time.png"); plt.show()
cmap = ListedColormap(["#eeeeee", "#d62728", "#1f77b4"])
code = lambda m: np.where(np.isin(m, list("HGI")), 1, np.where(np.isin(m, list("EB")), 2, 0))
fig, axs = plt.subplots(6, 1, figsize=(12,13), dpi=150, sharex=True)
for ax, (c, s) in zip(axs, sty.items()):
    ax.imshow(code(Dd[c]), aspect="auto", cmap=cmap, vmin=0, vmax=2, origin="lower", extent=[1,NR,0,1])
    ax.set_ylabel("Time (ns)"); ax.set_title(s["label"], loc="left", fontsize=10)
axs[-1].set_xlabel("Residue"); fig.suptitle(f"DSSP map: red = helix, blue = strand, grey = coil/turn, {T}")
fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_map.png"); plt.show()

# ---- summary + all-states plots
def last(f, col=1): b = xvg(f); return b[len(b)//2:, col].mean()
h = len(t)//2; rows = []
for c in CH:
    rows.append(dict(chain=c, RMSD_nm=last(f"{X}/rmsd_{c}.xvg"), Rg_nm=last(f"{X}/rg_{c}.xvg"),
      SASA_nm2=last(f"{X}/sasa_{c}.xvg", 2), intra_Hbonds=last(f"{X}/hbnum_{c}.xvg"),
      helix_pct=100*np.isin(Dd[c][h:], list("HGI")).mean(), strand_pct=100*np.isin(Dd[c][h:], list("EB")).mean()))
rows.append(dict(chain="hexamer", RMSD_nm=last(f"{X}/rmsd_Protein.xvg"), Rg_nm=last(f"{X}/rg_Protein.xvg"),
                 SASA_nm2=last(f"{X}/sasa_Protein.xvg"), intra_Hbonds=last(f"{X}/hbnum_Protein.xvg")))
df = pd.DataFrame(rows).round(3); df.to_csv(f"{X}/summary_last0.5ns.csv", index=False); print(df)
STATES = [("Alpha helix","H"),("3-10 helix","G"),("Pi helix","I"),("Beta strand","E"),("Beta bridge","B"),
          ("Turn","T"),("Bend","S"),("Polyproline II helix","P")]
Pc = {k_: 100*(a == c_).mean(1) for k_, c_ in STATES}; Pc["Coil / Unstructured"] = 100 - sum(Pc.values())
tp = np.arange(len(a))*5.0
fig, ax = plt.subplots(figsize=(12,6.2), dpi=150)
for i, (k_, y) in enumerate(Pc.items()): ax.plot(tp, y, color=plt.cm.tab10(i), lw=1.6, label=k_)
ax.set(xlabel="Time (ps)", ylabel="Percentage (%)", title=f"Legumin All Secondary Structure States — {T}")
ax.grid(alpha=.3); ax.legend(title="Secondary structure", ncol=2, loc="upper right"); fig.tight_layout()
fig.savefig(f"{OUT}/graphs/dssp_all_states_{TEMP}K_{RUN}.png"); plt.show()
Gp = {"Alpha helix": Pc["Alpha helix"], "Beta structure": Pc["Beta strand"]+Pc["Beta bridge"], "Turns": Pc["Turn"]}
Gp["Loops / Other"] = 100 - sum(Gp.values())
fig, ax = plt.subplots(figsize=(10,6), dpi=150)
for i, (k_, y) in enumerate(Gp.items()): ax.plot(tp, y, color=plt.cm.tab10(i), lw=1.8, label=k_)
ax.set(xlabel="Time (ps)", ylabel="Percentage (%)", title=f"Legumin Secondary Structure — {T}")
ax.grid(alpha=.3); ax.legend(); fig.tight_layout()
fig.savefig(f"{OUT}/graphs/dssp_groups_{TEMP}K_{RUN}.png"); plt.show()
print("DONE ->", OUT)

## 10. Comparison of the first runs
All-runs summary tables and plots for 300 K r1/r2, 373 K r1 and 373 K shear r1.

In [ ]:
import os, itertools, numpy as np, pandas as pd, matplotlib.pyplot as plt
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"): drive.mount('/content/drive')
BASE = "/content/drive/MyDrive/Plant Protein Extrusion MD/analysism"
RUNS = {"300 K r1":"legumin_300_r1","300 K r2":"legumin_300_r2",
        "373 K r1":"legumin_373_r1","373 K shear":"legumin_373_shear_r1"}
COL = {"300 K r1":"#1f77b4","300 K r2":"#7fbfff","373 K r1":"#d62728","373 K shear":"#2ca02c"}
LS  = {"300 K r1":"-","300 K r2":"--","373 K r1":"-","373 K shear":"--"}
CMP = f"{BASE}/comparison_legumin"
for d in ("graphs","csv"): os.makedirs(f"{CMP}/{d}", exist_ok=True)
NR = 496; CH = ["chA","chB","chC","chD","chE","chF"]
PAIRS = list(itertools.combinations(CH, 2)); PN = [f"{a[-1]}-{b[-1]}" for a,b in PAIRS]
STAT = [("Alpha helix","H"),("3-10 helix","G"),("Pi helix","I"),("Beta strand","E"),
        ("Beta bridge","B"),("Turn","T"),("Bend","S"),("PPII helix","P")]
xvg = lambda f: np.loadtxt(f, comments=('#','@'))
ok = os.path.exists
def load_dssp(p):
    a = np.array([list(l.rstrip("\n")) for l in open(p) if l.strip()])
    return a[:, ~(a == "=").all(0)]

# ---------------- load everything ----------------
rows, resrows, comp = [], [], {}
def add(run, m, e, t, y): rows.extend((run, m, e, float(a), float(b)) for a, b in zip(t, y))
spec = {"RMSD (nm)":("rmsd",1,1.0), "Rg (nm)":("rg",1,1000.0),
        "SASA (nm2)":("sasa",2,1.0), "H-bonds (hexamer)":("hbnum",1,1.0)}
RESPAT = {"RMSF (nm)":"rmsf_{c}", "RMSD per residue (nm)":"rmsd_perres_{c}",
          "H-bonds per residue":"hbperres_{c}", "Inter-chain H-bonds per residue":"hbperres_inter_{c}",
          "Helix fraction":"dssp_helix_perres_{c}", "Strand fraction":"dssp_strand_perres_{c}"}
for run, fd in RUNS.items():
    X = f"{BASE}/{fd}/xvgs"
    if not os.path.isdir(X): print("MISSING run folder:", run); continue
    for m, (pre, col, sc) in spec.items():
        for e in CH + ["Protein"]:
            f = f"{X}/{pre}_{e}.xvg"
            if ok(f): a = xvg(f); add(run, m, e, a[:,0]/sc, a[:,col])
            else: print("missing", run, f"{pre}_{e}.xvg")
    for (a_, b_), n in zip(PAIRS, PN):
        f = f"{X}/hbinter_{a_}_{b_}.xvg"
        if ok(f): a = xvg(f); add(run, "Inter H-bonds", n, a[:,0], a[:,1])
    f = f"{X}/dssp_chA.dat"                      # whole-hexamer DSSP (copied in the run script)
    if ok(f):
        a = load_dssp(f); n = len(a); t = np.linspace(0, 1, n)   # frames span 1 ns
        assert a.shape[1] == 6*NR, (run, a.shape)
        ents = {"Protein": a, **{c: a[:, i*NR:(i+1)*NR] for i, c in enumerate(CH)}}
        for e, mm in ents.items():
            hel = 100*np.isin(mm, list("HGI")).mean(1); stn = 100*np.isin(mm, list("EB")).mean(1)
            add(run, "Helix (%)", e, t, hel); add(run, "Strand (%)", e, t, stn)
            add(run, "Coil/turn (%)", e, t, 100-hel-stn)
        comp[run] = {nm: 100*(a[n//2:] == ch).mean() for nm, ch in STAT}
        comp[run]["Coil/other"] = 100 - sum(comp[run].values())
    for c in CH:
        for m, pat in RESPAT.items():
            f = f"{X}/{pat.format(c=c)}.xvg"
            if ok(f):
                v = xvg(f)[:,1]
                if len(v) == NR: resrows.extend((run, m, c, i+1, float(x)) for i, x in enumerate(v))
    f = f"{X}/sasa_perres_Protein.xvg"
    if ok(f):
        d = xvg(f)
        if len(d) == 6*NR:
            for i, c in enumerate(CH):
                resrows.extend((run, "SASA per residue (nm2)", c, j+1, float(x)) for j, x in enumerate(d[i*NR:(i+1)*NR, 1]))

ts  = pd.DataFrame(rows, columns=["run","metric","entity","time_ns","value"])
res = pd.DataFrame(resrows, columns=["run","metric","chain","residue","value"])
ts.to_csv(f"{CMP}/csv/timeseries_long.csv", index=False)
res.to_csv(f"{CMP}/csv/residue_metrics_long.csv", index=False)
pd.DataFrame(comp).T.round(2).to_csv(f"{CMP}/csv/dssp_composition_last_half.csv")

# ---------------- summaries ----------------
def stats(t, y):
    h = t >= t.max()/2
    return dict(mean_full=y.mean(), sd_full=y.std(ddof=1), mean_last_half=y[h].mean(), sd_last_half=y[h].std(ddof=1))
pc = [dict(run=r, metric=m, entity=e, **stats(d.sort_values("time_ns").time_ns.values, d.sort_values("time_ns").value.values))
      for (r, m, e), d in ts.groupby(["run","metric","entity"])]
pd.DataFrame(pc).round(4).to_csv(f"{CMP}/csv/summary_per_chain_and_interface.csv", index=False)

MET = [("RMSD (nm)",["Protein"],"mean","Backbone RMSD, whole hexamer (nm)"),
       ("Rg (nm)",["Protein"],"mean","Radius of gyration, whole hexamer (nm)"),
       ("SASA (nm2)",["Protein"],"mean","SASA, whole hexamer (nm²)"),
       ("H-bonds (hexamer)",["Protein"],"mean","H-bonds within chains (hexamer)"),
       ("Inter H-bonds",PN,"sum","Inter-chain H-bonds (sum of 15 pairs)"),
       ("Helix (%)",["Protein"],"mean","Helix (%)"),
       ("Strand (%)",["Protein"],"mean","Strand (%)"),
       ("Coil/turn (%)",["Protein"],"mean","Coil/turn (%)")]
def curve(run, m, ents, agg):
    d = ts[(ts.run==run) & (ts.metric==m) & (ts.entity.isin(ents))]
    if d.empty: return None
    pr = [e for e in ents if (d.entity==e).any()]
    t0 = d[d.entity==pr[0]].sort_values("time_ns").time_ns.values
    Y = np.array([np.interp(t0, *[d[d.entity==e].sort_values("time_ns")[k].values for k in ("time_ns","value")]) for e in pr])
    return t0, (Y.sum(0) if agg=="sum" else Y.mean(0))
agg_rows = []
for run in RUNS:
    for m, ents, agg, yl in MET:
        c = curve(run, m, ents, agg)
        if c: agg_rows.append(dict(run=run, metric=m, **stats(*c)))
S = pd.DataFrame(agg_rows); S.round(4).to_csv(f"{CMP}/csv/summary_all_runs.csv", index=False)
table = S.pivot(index="run", columns="metric", values="mean_last_half").reindex(list(RUNS))
table = table[[m[0] for m in MET if m[0] in table.columns]]
base = table.loc[["300 K r1","300 K r2"]].mean()
th = table.loc["373 K r1"] - base; sh_ = table.loc["373 K shear"] - table.loc["373 K r1"]
rep = (table.loc["300 K r1"] - table.loc["300 K r2"]).abs()
DD = pd.DataFrame({"mean_300K_r1_r2":base, "val_373K_r1":table.loc["373 K r1"], "val_373K_shear":table.loc["373 K shear"],
                   "thermal_delta(373r1-300)":th, "thermal_pct":100*th/base,
                   "shear_delta(shear-373r1)":sh_, "shear_pct":100*sh_/table.loc["373 K r1"],
                   "replicate_gap(300r1-r2)":rep})
DD["thermal_exceeds_replicate_gap"] = th.abs() > rep
DD["shear_exceeds_replicate_gap"] = sh_.abs() > rep
DD.round(4).to_csv(f"{CMP}/csv/condition_deltas.csv")

# ---------------- graphs: time series ----------------
def ts_ax(ax, m, ents, agg, yl):
    for run in RUNS:
        c = curve(run, m, ents, agg)
        if c: ax.plot(c[0], c[1], color=COL[run], ls=LS[run], lw=1.7, label=run)
    ax.set(xlabel="Time (ns)", ylabel=yl, title=yl); ax.grid(alpha=.3); ax.legend(fontsize=8)
fig, axs = plt.subplots(4, 2, figsize=(14, 17))
for ax, (m, ents, agg, yl) in zip(axs.ravel(), MET): ts_ax(ax, m, ents, agg, yl)
fig.suptitle("Legumin hexamer: all runs", y=1.0); fig.tight_layout()
fig.savefig(f"{CMP}/graphs/00_overview_timeseries.png", dpi=150); plt.show()
for i, (m, ents, agg, yl) in enumerate(MET, 1):
    f2, a2 = plt.subplots(figsize=(10, 5), dpi=150); ts_ax(a2, m, ents, agg, yl); f2.tight_layout()
    f2.savefig(f"{CMP}/graphs/ts_{m.split(' (')[0].replace(' ','_').replace('/','_')}.png"); plt.close(f2)

# ---------------- graphs: bars ----------------
fig, axs = plt.subplots(2, 4, figsize=(16, 7))
for ax, (m, ents, agg, yl) in zip(axs.ravel(), MET):
    d = S[S.metric==m].set_index("run").reindex(list(RUNS)).dropna()
    ax.bar(d.index, d.mean_last_half, yerr=d.sd_last_half, color=[COL[r] for r in d.index], capsize=4)
    ax.set_title(m, fontsize=10); ax.tick_params(axis="x", rotation=30); ax.grid(axis="y", alpha=.3)
fig.suptitle("Last-half mean ± SD (whole hexamer)"); fig.tight_layout()
fig.savefig(f"{CMP}/graphs/01_bar_summary.png", dpi=150); plt.show()

P = pd.DataFrame(pc)
fig, axs = plt.subplots(2, 3, figsize=(18, 8)); w = 0.2
for ax, m in zip(axs.ravel(), ["RMSD (nm)","Rg (nm)","SASA (nm2)","H-bonds (hexamer)","Helix (%)","Strand (%)"]):
    for k, run in enumerate(RUNS):
        d = P[(P.run==run)&(P.metric==m)&(P.entity.isin(CH))].set_index("entity").reindex(CH)
        ax.bar(np.arange(6)+k*w, d.mean_last_half, w, yerr=d.sd_last_half, color=COL[run], label=run, capsize=2)
    ax.set_xticks(np.arange(6)+1.5*w); ax.set_xticklabels([f"Chain {c[-1]}" for c in CH]); ax.set_title(m); ax.grid(axis="y", alpha=.3)
axs[0,0].legend(fontsize=8); fig.suptitle("Per-chain last-half mean ± SD"); fig.tight_layout()
fig.savefig(f"{CMP}/graphs/02_per_chain_bars.png", dpi=150); plt.show()

# heatmap of % change vs 300 K baseline
pct = 100*(table - base)/base
fig, ax = plt.subplots(figsize=(12, 3.8)); v = np.nanpercentile(np.abs(pct.values), 95)
im = ax.imshow(pct.values, cmap="RdBu_r", vmin=-v, vmax=v, aspect="auto")
ax.set_xticks(range(pct.shape[1])); ax.set_xticklabels(pct.columns, rotation=30, ha="right")
ax.set_yticks(range(pct.shape[0])); ax.set_yticklabels(pct.index)
for i in range(pct.shape[0]):
    for j in range(pct.shape[1]): ax.text(j, i, f"{pct.values[i,j]:.1f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, label="% change vs mean of 300 K r1/r2"); ax.set_title("Change relative to 300 K baseline (%)")
fig.tight_layout(); fig.savefig(f"{CMP}/graphs/03_pct_change_heatmap.png", dpi=150); plt.show()

# DSSP composition
if comp:
    C = pd.DataFrame(comp).T.reindex([r for r in RUNS if r in comp])
    ax = C.plot(kind="bar", stacked=True, figsize=(10, 5), colormap="tab20"); ax.set_ylabel("% of residues (last half)")
    ax.set_title("DSSP composition, whole hexamer"); ax.legend(ncol=2, fontsize=8, bbox_to_anchor=(1.01, 1))
    plt.xticks(rotation=20); plt.tight_layout(); plt.savefig(f"{CMP}/graphs/04_dssp_composition.png", dpi=150); plt.show()

# ---------------- graphs: per residue ----------------
rm = res.groupby(["run","metric","residue"]).value.mean()
def rcurve(run, m):
    try: s = rm.loc[(run, m)]; return s.index.values, s.values
    except KeyError: return None
RM = list(RESPAT) + ["SASA per residue (nm2)"]
fig, axs = plt.subplots(4, 2, figsize=(15, 17))
for ax, m in zip(axs.ravel(), RM):
    for run in RUNS:
        c = rcurve(run, m)
        if c: ax.plot(*c, color=COL[run], ls=LS[run], lw=1.3, label=run)
    ax.set(xlabel="Residue", ylabel=m, title=m + " (mean of 6 chains)"); ax.grid(alpha=.3); ax.legend(fontsize=8)
axs.ravel()[-1].axis("off"); fig.tight_layout()
fig.savefig(f"{CMP}/graphs/05_residue_overlays.png", dpi=150); plt.show()

DM = ["RMSF (nm)","SASA per residue (nm2)","Helix fraction","Strand fraction","H-bonds per residue"]
fig, axs = plt.subplots(len(DM), 2, figsize=(15, 3.2*len(DM))); top = []
for i, m in enumerate(DM):
    a300 = [rcurve(r, m) for r in ("300 K r1","300 K r2")]; a300 = [c[1] for c in a300 if c]
    r373, rsh = rcurve("373 K r1", m), rcurve("373 K shear", m)
    if not (a300 and r373 and rsh): continue
    resid = r373[0]; b = np.mean(a300, axis=0)
    for j, (name, dl) in enumerate((("Thermal: 373 K r1 − 300 K mean", r373[1]-b), ("Shear: 373 K shear − 373 K r1", rsh[1]-r373[1]))):
        ax = axs[i, j]; ax.fill_between(resid, dl, 0, where=dl>=0, color="#d62728", alpha=.6)
        ax.fill_between(resid, dl, 0, where=dl<0, color="#1f77b4", alpha=.6)
        ax.axhline(0, color="k", lw=.6); ax.set(xlabel="Residue", ylabel=f"Δ {m}", title=name); ax.grid(alpha=.3)
        for k in np.argsort(-np.abs(dl))[:10]: top.append(dict(contrast=name, metric=m, residue=int(resid[k]), delta=float(dl[k])))
fig.tight_layout(); fig.savefig(f"{CMP}/graphs/06_residue_deltas.png", dpi=150); plt.show()
pd.DataFrame(top).round(4).to_csv(f"{CMP}/csv/top10_residue_changes.csv", index=False)

# ---------------- report ----------------
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
print("\nLast-half means, whole hexamer:\n", table.round(3))
print("\nDeltas vs replicate gap:\n", DD.round(3))
print("\nSaved to:", CMP)

In [ ]:
import os, glob, json, subprocess, numpy as np, pandas as pd, MDAnalysis as mda, matplotlib.pyplot as plt
from MDAnalysis.lib.distances import minimize_vectors
from MDAnalysis.analysis.align import rotation_matrix
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"): drive.mount('/content/drive')
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
BASE = "/content/drive/MyDrive/Plant Protein Extrusion MD"
CMP = f"{BASE}/analysism/comparison_legumin"
for d in ("csv","graphs"): os.makedirs(f"{CMP}/{d}", exist_ok=True)
RUNS = {"373 K shear":(BASE,"md_373_shear_r1_combined","legumin_373_shear_r1"),
        "373 K r1":(BASE,"md_373_r1","legumin_373_r1"),
        "300 K r1":(BASE,"md_300_r1","legumin_300_r1"),
        "300 K r2":(BASE,"md_300_r2","legumin_300_r2")}
rows = []; bshear = None
for run, (P, S, name) in RUNS.items():
    F = f"{BASE}/analysism/{name}/frames"
    ranges = json.load(open(f"{F}/chains.json"))["ranges"]; sl = [slice(*ranges[c]) for c in ("chA","chB","chC","chD","chE","chF")]
    subprocess.run(["bash","-c",f'printf "Protein\\n" | gmx trjconv -s "{F}/protein.tpr" -f "{P}/{S}.xtc" -o /tmp/w.xtc -pbc whole > /tmp/w.log 2>&1'])
    u = mda.Universe(f"{F}/first_cl.gro", "/tmp/w.xtc"); ca = u.select_atoms("name CA").indices
    ref = None
    for ts in u.trajectory[::5]:
        pos = u.atoms.positions.copy(); box = ts.dimensions.astype(np.float32)
        if run == "373 K shear" and bshear is None: bshear = float(box[1])
        a0 = pos[sl[0]].mean(0)
        for s in sl[1:]:
            d = pos[s].mean(0) - a0
            dm = minimize_vectors(d[None,:].astype(np.float32), box)[0]
            pos[s] -= (d - dm)
        x = pos[ca]; x = x - x.mean(0)
        if ref is None: ref = x.copy()
        R, _ = rotation_matrix(x, ref); Rt = R.T
        th = np.arccos(np.clip((np.trace(Rt)-1)/2, -1, 1))
        ax = np.array([Rt[2,1]-Rt[1,2], Rt[0,2]-Rt[2,0], Rt[1,0]-Rt[0,1]])
        n = np.linalg.norm(ax); ax = ax/n if n > 1e-12 else np.zeros(3)
        rv = np.degrees(th)*ax
        rows.append(dict(run=run, time_ns=ts.time/1000, angle_deg=np.degrees(th), rot_x=rv[0], rot_y=rv[1], rot_z=rv[2]))
    os.remove("/tmp/w.xtc"); print("done", run)
df = pd.DataFrame(rows); df.round(3).to_csv(f"{CMP}/csv/rotation_no_fit.csv", index=False)

COL = {"300 K r1":"#1f77b4","300 K r2":"#7fbfff","373 K r1":"#d62728","373 K shear":"#2ca02c"}
rate = -0.5*(3.644/(bshear/10))*180/np.pi            # deg/ns about z, from the real box
fig, axs = plt.subplots(1, 2, figsize=(14,5), dpi=150)
for run, d in df.groupby("run"):
    axs[0].plot(d.time_ns, d.angle_deg, color=COL[run], label=run)
    axs[1].plot(d.time_ns, d.rot_z, color=COL[run], label=run)
tt = np.linspace(0, 1, 50); axs[1].plot(tt, rate*tt, "k--", label=f"expected, rigid body in shear ({rate:.2f} °/ns)")
axs[0].set(xlabel="Time (ns)", ylabel="Total rotation from frame 0 (deg)", title="Hexamer rotation, no rotational fit")
axs[1].set(xlabel="Time (ns)", ylabel="Rotation about z (deg)", title="Rotation about vorticity axis (z)")
for a in axs: a.grid(alpha=.3); a.legend(fontsize=8)
fig.tight_layout(); fig.savefig(f"{CMP}/graphs/07_rotation_no_fit.png"); plt.show()
print("expected rigid rotation about z:", round(rate,2), "deg/ns")
print(df.groupby("run").tail(1)[["run","angle_deg","rot_x","rot_y","rot_z"]].round(1))

## 11. Replicate 2: 373 K r2 and shear r2
Builds the r2 NVT file (`gen-seed = 222`), checks r1 against r2, and runs and checks NPT and production for both r2 conditions.

In [ ]:
import subprocess, os
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
cmd = '''cd "/content/drive/MyDrive/Plant Protein Extrusion MD"
printf "Pres-XY\\n0\\n" | gmx energy -f md_373_shear_r1_combined.edr -o /tmp/presxy.xvg -b 500 2>&1 | tail -6'''
r = subprocess.run(["bash","-c",cmd], capture_output=True, text=True); print(r.stdout, r.stderr[-300:])

In [ ]:
import os, subprocess
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"): drive.mount('/content/drive')
cmd = r'''cd "/content/drive/MyDrive/Plant Protein Extrusion MD"
echo "== setup files"; ls *.mdp *.ndx *.top *.itp 2>/dev/null; ls -d */ 2>/dev/null
echo; echo "== nvt/em/other mdp"; ls | grep -iE "nvt|^em|minim|ions"
for f in npt_373_r1.mdp md_373_r1.mdp md_373_shear_r1.mdp; do echo; echo "===== $f"; grep -vE "^ *;|^ *$" "$f"; done
echo; echo "== nvt mdp (if present)"; for f in nvt*.mdp; do [ -f "$f" ] && { echo "===== $f"; grep -vE "^ *;|^ *$" "$f"; }; done
echo; echo "== log header seeds"; grep -iE "gen[-_]seed|ld[-_]seed|continuation" md_373_r1.mdp npt_373_r1.mdp'''
r = subprocess.run(["bash","-c",cmd], capture_output=True, text=True); print(r.stdout); print(r.stderr[-300:])

In [ ]:
import os, subprocess
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"): drive.mount('/content/drive')
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
cmd = r'''cd "/content/drive/MyDrive/Plant Protein Extrusion MD"
sed 's/^gen-seed *= *111/gen-seed = 222/' nvt_373_r1.mdp > nvt_373_r2.mdp
grep -E "gen-seed|ref-t|gen-temp" nvt_373_r2.mdp
gmx grompp -f nvt_373_r2.mdp -c em3.gro -r em3.gro -p topol.top -o nvt_373_r2.tpr -maxwarn 0 2>&1 | tail -8
echo "== r1 vs r2 tpr differences"; gmx check -s1 nvt_373_r1.tpr -s2 nvt_373_r2.tpr 2>&1 | grep -iE "gen|seed|differ|ld|natoms" | head -15'''
r = subprocess.run(["bash","-c",cmd], capture_output=True, text=True); print(r.stdout, r.stderr[-300:])

In [ ]:
import os, subprocess
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
cmd = r'''cd "/content/drive/MyDrive/Plant Protein Extrusion MD"
printf "em.gro atoms: "; head -2 em.gro | tail -1
echo "== molecules in topol.top"; sed -n '/molecules/,$p' topol.top
echo; echo "== grompp"
gmx grompp -f nvt_373_r2.mdp -c em.gro -r em.gro -p topol.top -o nvt_373_r2.tpr -maxwarn 0 2>&1 | grep -iE "warning|error|fatal|total charge|writing" | head -10
ls -lh nvt_373_r1.tpr nvt_373_r2.tpr
echo; echo "== r1 vs r2 tpr"
gmx check -s1 nvt_373_r1.tpr -s2 nvt_373_r2.tpr 2>&1 | grep -vE "^ *$" | head -30'''
r = subprocess.run(["bash","-c",cmd], capture_output=True, text=True); print(r.stdout, r.stderr[-300:])

In [ ]:
import os, subprocess
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
cmd = r'''cd "/content/drive/MyDrive/Plant Protein Extrusion MD"
echo "== 1. mdp diff r1 vs r2 (should be gen-seed only)"; diff nvt_373_r1.mdp nvt_373_r2.mdp
echo; echo "== 2. tpr comparison, full (t_resinfo lines hidden)"
gmx check -s1 nvt_373_r1.tpr -s2 nvt_373_r2.tpr 2>&1 | grep -vE "t_resinfo|^ *$" | tail -25
echo; echo "== 3. any existing nvt_373_r2 files besides the tpr (mdrun would append to them)"; ls -lh nvt_373_r2* 2>&1
echo; echo "== 4. how r1 NVT was launched"; grep -A3 -m1 "Command line" nvt_373_r1.log; grep -iE "GPU|nb |pme " nvt_373_r1.log | head -6
echo; echo "== 5. how r1 NPT was started"; grep -A3 -m1 "Command line" npt_373_r1.log 2>/dev/null; ls npt_373_r1*.log
echo; echo "== 6. free Drive space"; df -h /content/drive | tail -1'''
r = subprocess.run(["bash","-c",cmd], capture_output=True, text=True); print(r.stdout, r.stderr[-300:])

In [ ]:
import os, subprocess
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
cmd = r'''cd "/content/drive/MyDrive/Plant Protein Extrusion MD"
gmx check -s1 nvt_373_r1.tpr -s2 nvt_373_r2.tpr > /tmp/chk.txt 2>&1
echo "x lines differing: $(grep -cE '^x\[' /tmp/chk.txt)"
echo "v lines differing: $(grep -cE '^v\[' /tmp/chk.txt)"
echo; echo "== other differences (expect only ld_seed)"
grep -vE "^v\[|^x\[|t_resinfo|^ *$|^comparing|Reading|GROMACS|Executable|Data prefix|Working dir|Command line|gmx check|Note:" /tmp/chk.txt'''
r = subprocess.run(["bash","-c",cmd], capture_output=True, text=True); print(r.stdout, r.stderr[-300:])

In [ ]:
import os, subprocess
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
r = subprocess.run(["bash","-c",'cd "/content/drive/MyDrive/Plant Protein Extrusion MD" && gmx mdrun -deffnm nvt_373_r2 -cpt 5 -ntmpi 1 -ntomp 2 -pin off 2>&1 | tail -5'], capture_output=True, text=True); print(r.stdout)

In [ ]:
import os, subprocess
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
cmd = r'''cd "/content/drive/MyDrive/Plant Protein Extrusion MD"
echo "== NVT r2 files"; ls -lh nvt_373_r2.*
echo; echo "== finished normally?"; tail -3 nvt_373_r2.log | head -2; grep -c "Finished mdrun" nvt_373_r2.log
echo; echo "== NVT r2 temperature (avg over run)"
printf "Temperature\n0\n" | gmx energy -f nvt_373_r2.edr -o /tmp/t_r2.xvg 2>&1 | grep -E "Temperature|Statistics"
echo; echo "== same for r1 (comparison)"
printf "Temperature\n0\n" | gmx energy -f nvt_373_r1.edr -o /tmp/t_r1.xvg 2>&1 | grep -E "Temperature"
echo; echo "== NPT r1 mdp vs what r2 will use"; ls npt_373_r2* 2>&1 | head -3
echo; echo "== NPT r1 launch line"; grep -A2 -m1 "Command line" npt_373_r1.part0002.log'''
r = subprocess.run(["bash","-c",cmd], capture_output=True, text=True); print(r.stdout, r.stderr[-300:])

In [ ]:
import os, subprocess
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
cmd = r'''cd "/content/drive/MyDrive/Plant Protein Extrusion MD"
cp npt_373_r1.mdp npt_373_r2.mdp
echo "== mdp diff r1 vs r2 (expect none)"; diff npt_373_r1.mdp npt_373_r2.mdp && echo IDENTICAL
echo; echo "== grompp"
gmx grompp -f npt_373_r2.mdp -c nvt_373_r2.gro -r nvt_373_r2.gro -t nvt_373_r2.cpt -p topol.top -o npt_373_r2.tpr -maxwarn 0 > /tmp/g.txt 2>&1
grep -iE "warning|error|fatal|note" /tmp/g.txt | head -10; tail -3 /tmp/g.txt
ls -lh npt_373_r1.tpr npt_373_r2.tpr
echo; echo "== inputrec/topology differences vs r1 (x, v, resinfo hidden)"
gmx check -s1 npt_373_r1.tpr -s2 npt_373_r2.tpr 2>&1 | grep -vE "^v\[|^x\[|t_resinfo|^ *$|^comparing|Reading|GROMACS|Executable|Data prefix|Working dir|Command line|gmx check|Note:|^ *[0-9.e+-]+ +[0-9.e+-]+ +[0-9.e+-]+\)"
echo; echo "== atoms"; gmx check -s npt_373_r2.tpr 2>&1 | grep -iE "natoms"'''
r = subprocess.run(["bash","-c",cmd], capture_output=True, text=True); print(r.stdout, r.stderr[-300:])

In [ ]:
import os, subprocess
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
r = subprocess.run(["bash","-c",'cd "/content/drive/MyDrive/Plant Protein Extrusion MD" && gmx mdrun -deffnm npt_373_r2 -cpt 5 -ntmpi 1 -ntomp 2 -pin off 2>&1 | tail -5'], capture_output=True, text=True); print(r.stdout)

In [ ]:
import os, subprocess
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
cmd = r'''cd "/content/drive/MyDrive/Plant Protein Extrusion MD"
echo "== NPT r2 files"; ls -lh npt_373_r2.*
echo; echo "== finished normally? (expect 1)"; grep -c "Finished mdrun" npt_373_r2.log
echo; echo "== averages"
printf "Temperature\nPressure\nDensity\n0\n" | gmx energy -f npt_373_r2.edr -o /tmp/npt_r2.xvg 2>&1 | grep -E "^(Temperature|Pressure|Density)"
echo; echo "== r1 NPT edr files (for comparison)"; ls npt_373_r1*.edr'''
r = subprocess.run(["bash","-c",cmd], capture_output=True, text=True); print(r.stdout, r.stderr[-300:])

In [ ]:
import os, subprocess
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
cmd = r'''cd "/content/drive/MyDrive/Plant Protein Extrusion MD"
echo "== shear r1 mdp"; cat md_373_shear_r1.mdp
echo; echo "== shear mdp vs thermal mdp (what the shear adds)"; diff md_373_r1.mdp md_373_shear_r1.mdp
echo; echo "== shear r1 launch line (original)"; grep -A2 -m1 "Command line" md_373_shear_r1.log
echo; echo "== shear r1 restart line (part0002)"; grep -A2 -m1 "Command line" md_373_shear_r1.part0002.log
echo; echo "== did shear r1 start from the same structure as thermal r1?"
gmx check -s1 md_373_r1.tpr -s2 md_373_shear_r1.tpr 2>&1 | grep -cE "^x\["
echo; echo "== any r2 shear files (expect none)"; ls md_373_shear_r2* 2>&1'''
r = subprocess.run(["bash","-c",cmd], capture_output=True, text=True); print(r.stdout, r.stderr[-300:])

In [ ]:
import os, subprocess
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
cmd = r'''cd "/content/drive/MyDrive/Plant Protein Extrusion MD"
sed 's/^ld-seed = 111/ld-seed = 222/' md_373_shear_r1.mdp > md_373_shear_r2.mdp
echo "== mdp diff r1 vs r2 (expect ld-seed only)"; diff md_373_shear_r1.mdp md_373_shear_r2.mdp
echo; echo "== grompp"
gmx grompp -f md_373_shear_r2.mdp -c npt_373_r2.gro -t npt_373_r2.cpt -p topol.top -o md_373_shear_r2.tpr -maxwarn 0 > /tmp/g.txt 2>&1
grep -iE "warning|error|fatal|note" /tmp/g.txt | head -10; tail -3 /tmp/g.txt
ls -lh md_373_shear_r1.tpr md_373_shear_r2.tpr
echo; echo "== inputrec differences vs r1 (x, v, resinfo hidden)"
gmx check -s1 md_373_shear_r1.tpr -s2 md_373_shear_r2.tpr 2>&1 | grep -vE "^v\[|^x\[|t_resinfo|^ *$|^comparing|Reading|GROMACS|Executable|Data prefix|Working dir|Command line|gmx check|Note:|^ *[0-9.e+-]+ +[0-9.e+-]+ +[0-9.e+-]+\)"
echo; echo "== did r1 shear and r1 thermal share velocities? (v lines differing, 0 means same NPT start)"
gmx check -s1 md_373_r1.tpr -s2 md_373_shear_r1.tpr 2>&1 | grep -cE "^v\["
echo; echo "== r2 shear files (expect mdp and tpr only)"; ls md_373_shear_r2*'''
r = subprocess.run(["bash","-c",cmd], capture_output=True, text=True); print(r.stdout, r.stderr[-300:])

In [ ]:
import os, subprocess
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
r = subprocess.run(["bash","-c",'cd "/content/drive/MyDrive/Plant Protein Extrusion MD" && gmx mdrun -deffnm md_373_shear_r2 -cpt 5 -ntmpi 1 -ntomp 2 -pin off 2>&1 | tail -5'], capture_output=True, text=True); print(r.stdout)

## 12. Replicate 2 analysis and output checks
Single-cell analysis for 373 K shear r2 and 373 K thermal r2, plus checks that all expected graphs and `.xvg` files exist.

In [ ]:
import os, subprocess
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"): drive.mount('/content/drive')
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
cmd = r'''cd "/content/drive/MyDrive/Plant Protein Extrusion MD" || { echo "FOLDER NOT FOUND - wrong Drive account?"; exit 1; }
echo "== shear r2 files"; ls -lh --time-style=+%H:%M md_373_shear_r2*
echo; echo "== finished normally? (expect 1 per log)"
for f in md_373_shear_r2.log md_373_shear_r2.part0002.log; do [ -f $f ] && echo "$f: $(grep -c 'Finished mdrun' $f)"; done
echo; echo "== xtc first / last frame"
for f in md_373_shear_r2.xtc md_373_shear_r2.part0002.xtc md_373_shear_r2_combined.xtc; do [ -f $f ] && { echo $f; gmx check -f $f 2>&1 | grep -E "Reading frame +0 |Last frame|^Step"; }; done
echo; echo "== checkpoint time"; gmx check -f md_373_shear_r2.cpt 2>&1 | grep -E "Last frame"
echo; echo "== energies"
for f in md_373_shear_r2.edr md_373_shear_r2.part0002.edr; do [ -f $f ] && { echo $f; printf "Temperature\n0\n" | gmx energy -f $f -o /tmp/t.xvg 2>&1 | grep -E "Statistics|^Temperature"; }; done
echo; echo "== final box (6th number = tilt, expect about 3.644)"
for f in md_373_shear_r2.part0002.gro md_373_shear_r2.gro; do [ -f $f ] && { echo $f; tail -1 $f; }; done
echo; echo "== backup folder"; ls backups 2>&1 | head'''
r = subprocess.run(["bash","-c",cmd], capture_output=True, text=True); print(r.stdout, r.stderr[-300:])

In [ ]:
import os, sys, json, itertools, subprocess
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from google.colab import drive
TEMP, RUN = "373", "shear_r2"
TPR_NAME  = "md_373_shear_r2"                 # tpr
TRAJ_NAME = "md_373_shear_r2"                 # ran in one part, no combine needed
NCH = 6; DEFORM = 3.644                        # hexamer; nm/ns tilt from the mdp
if not os.path.exists("/content/drive/MyDrive"): drive.mount('/content/drive')
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
if os.system("gmx --version > /dev/null 2>&1") != 0: raise SystemExit("GROMACS not found - run your GROMACS setup cell first, then run this cell again")
os.system('ln -sfn "/content/drive/MyDrive/Plant Protein Extrusion MD" /content/pm')
OUT = f"/content/pm/analysism/legumin_{TEMP}_{RUN}"; T = "373 K shear (r2)"
os.environ.update(P="/content/pm", S=TPR_NAME, TRAJ=TRAJ_NAME, OUT=OUT)
for d in ["graphs","logs","frames","xvgs"]: os.makedirs(f"{OUT}/{d}", exist_ok=True)
subprocess.run([sys.executable,"-m","pip","install","-q","MDAnalysis","scipy"])
import MDAnalysis as mda
from MDAnalysis.coordinates.XTC import XTCReader
from MDAnalysis.lib.distances import minimize_vectors
from MDAnalysis.analysis.align import rotation_matrix
from scipy.spatial import cKDTree
from matplotlib.colors import ListedColormap
def sh(cmd, tail=0):
    r = subprocess.run(["bash","-c",cmd], capture_output=True, text=True)
    o = (r.stdout + r.stderr).strip().split("\n")
    print("\n".join(o[-tail:] if tail else o))

# ---- check files, mdp, and trajectory continuity
print("== files"); sh('ls -lh $P/$S.tpr $P/$TRAJ.xtc; grep -iE "ref[-_]t|deform" $P/$S.mdp')
rd = XTCReader(f"{OUT.rsplit('/analysism',1)[0]}/{TRAJ_NAME}.xtc")
times = np.array([ts.time for ts in rd]); dims = np.array([ts.dimensions.copy() for ts in rd])
print("frames:", len(times), "| time", times[0], "->", times[-1], "| natoms:", rd.n_atoms)
assert times[0] == 0 and abs(times[-1]-1000) < 1e-3 and len(times) == 1001 and np.all(np.diff(times) == 1), "trajectory not continuous 0-1000 ps"
print("box a,b,c start:", dims[0,:3].round(2), "end:", dims[-1,:3].round(2))
print("angles start:", dims[0,3:].round(2), "end:", dims[-1,3:].round(2))
gd = DEFORM/(dims[0,1]/10); print(f"shear rate {gd:.4f} /ns -> expected rigid rotation {-0.5*gd*180/np.pi:.2f} deg/ns about z")

# ---- protein tpr, whole, first frame
print("== whole")
sh(r'''cd "$OUT/frames"
printf "Protein\n" | gmx convert-tpr -s "$P/$S.tpr" -o protein.tpr 2>&1 | tail -1
printf "Protein\n" | gmx trjconv -s protein.tpr -f "$P/$TRAJ.xtc" -o whole2.xtc -pbc whole 2>&1 | tail -1
printf "Protein\n" | gmx trjconv -s protein.tpr -f whole2.xtc -o first_w.gro -dump 0 2>&1 | tail -1''')

# ---- chain detection from .gro
lines = open(f"{OUT}/frames/first_w.gro").read().split("\n")
n = int(lines[1]); body = lines[2:2+n]
resnum = np.array([int(l[0:5]) for l in body]); nm = [l[10:15].strip() for l in body]
starts = [0] + [i for i in range(1, n) if resnum[i] < resnum[i-1]]; ends = starts[1:] + [n]
nres = [len(set(resnum[s:e])) for s, e in zip(starts, ends)]
print("atoms:", n, "| chains:", len(starts), "| atoms/chain:", [e-s for s,e in zip(starts,ends)], "| res/chain:", nres)
assert len(starts) == NCH, f"expected {NCH} chains - send me the line above"
assert len(set(e-s for s,e in zip(starts,ends))) == 1, "chains differ in size - send me the line above"
NR = nres[0]; CH = ["ch"+"ABCDEF"[i] for i in range(NCH)]; G = {}; ranges = {}
for k, (s, e) in enumerate(zip(starts, ends)):
    r = range(s+1, e+1); c = CH[k]; ranges[c] = [s, e]
    G[c] = list(r); G[c+"_bb"] = [i for i in r if nm[i-1] in ("N","CA","C")]; G[c+"_CA"] = [i for i in r if nm[i-1]=="CA"]
r = range(1, n+1)
G["Protein"] = list(r); G["Protein_bb"] = [i for i in r if nm[i-1] in ("N","CA","C")]; G["Protein_CA"] = [i for i in r if nm[i-1]=="CA"]
with open(f"{OUT}/frames/chains.ndx","w") as f:
    for g, idx in G.items():
        f.write(f"[ {g} ]\n")
        for k in range(0, len(idx), 15): f.write(" ".join(f"{i:5d}" for i in idx[k:k+15]) + "\n")
json.dump(dict(ranges=ranges, nres=NR, natoms=n), open(f"{OUT}/frames/chains.json","w"))

# ---- keep chains together (triclinic-safe)
sl = [slice(*ranges[c]) for c in CH]
u = mda.Universe(f"{OUT}/frames/first_w.gro", f"{OUT}/frames/whole2.xtc")
with mda.Writer(f"{OUT}/frames/together.xtc", u.atoms.n_atoms) as W:
    for ts in u.trajectory[::5]:
        pos = u.atoms.positions.copy(); box = ts.dimensions.astype(np.float32); a0 = pos[sl[0]].mean(0)
        for s in sl[1:]:
            d = pos[s].mean(0) - a0
            dm = minimize_vectors(d[None,:].astype(np.float32), box)[0]; pos[s] -= (d - dm)
        u.atoms.positions = pos; W.write(u.atoms)

# ---- fit
print("== fit")
sh(r'''cd "$OUT/frames"
printf "Backbone\nProtein\n" | gmx trjconv -s protein.tpr -f together.xtc -o fit_cluster_skip5.xtc -fit rot+trans 2>&1 | tail -1
printf "Protein\n" | gmx trjconv -s protein.tpr -f fit_cluster_skip5.xtc -o first_cl.gro -dump 0 2>&1 | tail -1
rm -f whole2.xtc together.xtc
gmx check -f fit_cluster_skip5.xtc 2>&1 | grep -E "Last frame|^Step"''')

# ---- hexamer integrity check
u = mda.Universe(f"{OUT}/frames/first_cl.gro", f"{OUT}/frames/fit_cluster_skip5.xtc")
pairs = list(itertools.combinations(range(NCH), 2)); worst = {p: 0.0 for p in pairs}; lonely = 0
for ts in u.trajectory:
    Pp = u.atoms.positions/10; ch = [Pp[s] for s in sl]
    m = {p: cKDTree(ch[p[0]]).query(ch[p[1]])[0].min() for p in pairs}
    for p in pairs: worst[p] = max(worst[p], m[p])
    for i in range(NCH):
        if min(m[p] for p in pairs if i in p) > 0.5: lonely += 1
print({"ABCDEF"[a]+"ABCDEF"[b]: round(float(v),2) for (a,b), v in worst.items()})
print("chains with no neighbour within 0.5 nm:", lonely)
assert lonely == 0, "a chain drifted away"

# ---- GROMACS analysis
print("== gmx analysis (slow)")
PAIRSTR = " ".join(f'"{a} {b}"' for a, b in itertools.combinations(CH, 2))
os.environ.update(CHS=" ".join(["Protein"]+CH), PAIRSTR=PAIRSTR)
sh(r'''cd "$OUT/frames"; X="$OUT/xvgs"; L="$OUT/logs"; T=fit_cluster_skip5.xtc
for c in $CHS; do
  printf "${c}_bb\n${c}_bb\n" | gmx rms  -s first_cl.gro -f $T -n chains.ndx -o $X/rmsd_$c.xvg -tu ns > $L/rmsd_$c.log 2>&1
  printf "${c}_CA\n${c}_CA\n" | gmx rmsf -s first_cl.gro -f $T -n chains.ndx -o $X/rmsf_$c.xvg -res   > $L/rmsf_$c.log 2>&1
  printf "$c\n"               | gmx gyrate -s protein.tpr -f $T -n chains.ndx -o $X/rg_$c.xvg          > $L/rg_$c.log 2>&1
  gmx sasa -s protein.tpr -f $T -n chains.ndx -surface "group Protein" -output "group $c" -o $X/sasa_$c.xvg -or $X/tmp_$c.xvg -tu ns > $L/sasa_$c.log 2>&1
  gmx hbond -s protein.tpr -f $T -n chains.ndx -r "group $c" -t "group $c" -num $X/hbnum_$c.xvg -tu ns > $L/hbond_$c.log 2>&1
done
eval "for p in $PAIRSTR; do set -- \$p
  gmx hbond -s protein.tpr -f $T -n chains.ndx -r \"group \$1\" -t \"group \$2\" -num $X/hbinter_\$1_\$2.xvg -tu ns > $L/hbinter_\$1_\$2.log 2>&1
done"
mv $X/tmp_Protein.xvg $X/sasa_perres_Protein.xvg; rm -f $X/tmp_ch*.xvg
for f in rmsd_Protein rmsd_chA rmsf_chA rg_Protein sasa_chA hbnum_Protein hbinter_chA_chB; do echo $f $(tail -1 $X/$f.xvg); done''')

# ---- plots
X = f"{OUT}/xvgs"
cols = ["#d62728","#1f77b4","#2ca02c","#ff9896","#7fbfff","#98df8a"]
sty = {c: dict(color=cols[i], ls="-" if i%2==0 else ":", label="Chain "+c[-1]) for i, c in enumerate(CH)}
xvg = lambda f: np.loadtxt(f, comments=('#','@'))
def draw(out, title, xl, yl, items):
    fig, ax = plt.subplots(figsize=(12,5), dpi=150)
    for x, y, s in items: ax.plot(x, y, lw=1.6, **s)
    ax.set(xlabel=xl, ylabel=yl, title=title); ax.legend(ncol=2); fig.tight_layout()
    fig.savefig(f"{OUT}/graphs/{out}.png"); plt.show()
chains = lambda pre, col=1, sc=1: [(xvg(f"{X}/{pre}_{c}.xvg")[:,0]/sc, xvg(f"{X}/{pre}_{c}.xvg")[:,col], s) for c, s in sty.items()]
def whole(pre, col=1, sc=1):
    a = xvg(f"{X}/{pre}_Protein.xvg"); return [(a[:,0]/sc, a[:,col], dict(color="grey", label="Whole hexamer"))]
for pre, yl, ttl, sc in (("rmsd","RMSD (nm)","Backbone RMSD",1), ("rg","Rg (nm)","Radius of gyration",1000),
                         ("sasa","SASA (nm²)","SASA",1), ("hbnum","H-bonds","Intra-chain H-bonds",1)):
    cc = 2 if pre == "sasa" else 1
    draw(f"{pre}_per_chain", f"{ttl} per chain, {T}", "Time (ns)", yl, chains(pre, cc, sc))
    draw(f"{pre}_protein", f"{ttl}, whole hexamer, {T}", "Time (ns)", yl, whole(pre, 1, sc))
draw("rmsf_per_chain", f"C-alpha RMSF per residue, {T}", "Residue", "RMSF (nm)", chains("rmsf"))
d = xvg(f"{X}/sasa_perres_Protein.xvg"); assert len(d) == NCH*NR, len(d)
draw("sasa_per_residue", f"SASA per residue, {T}", "Residue", "SASA (nm²)",
     [(np.arange(1,NR+1), d[i*NR:(i+1)*NR,1], s) for i, s in enumerate(sty.values())])
fig, ax = plt.subplots(figsize=(12,6), dpi=150)
for k, (a_, b_) in enumerate(itertools.combinations(CH, 2)):
    h = xvg(f"{X}/hbinter_{a_}_{b_}.xvg"); ax.plot(h[:,0], h[:,1], lw=1.2, color=plt.cm.tab10(k), label=f"{a_[-1]}–{b_[-1]}")
ax.set(xlabel="Time (ns)", ylabel="H-bonds", title=f"Inter-chain H-bonds, {T}"); ax.legend(ncol=3, fontsize=8)
fig.tight_layout(); fig.savefig(f"{OUT}/graphs/hbonds_inter_chain.png"); plt.show()

# ---- per-residue RMSD and H-bonds
u = mda.Universe(f"{OUT}/frames/first_cl.gro", f"{OUT}/frames/fit_cluster_skip5.xtc")
NA = u.atoms.n_atoms // NCH; cs = {c: slice(i*NA, (i+1)*NA) for i, c in enumerate(CH)}; resR = {}
for name, s in cs.items():
    chn = u.atoms[s]; bb = chn.select_atoms("name N CA C"); ca = chn.select_atoms("name CA")
    u.trajectory[0]; c0 = bb.positions.mean(0); bb0 = bb.positions - c0; ca0 = ca.positions - c0
    d2 = np.zeros(len(ca)); k = 0
    for ts in u.trajectory:
        c = bb.positions.mean(0); R, _ = rotation_matrix(bb.positions - c, bb0)
        d2 += (((ca.positions - c) @ R.T - ca0)**2).sum(1); k += 1
    resR[name] = np.sqrt(d2/k)/10
    np.savetxt(f"{X}/rmsd_perres_{name}.xvg", np.c_[np.arange(1,NR+1), resR[name]], fmt="%d %.6f")
draw("rmsd_per_residue", f"C-alpha RMSD per residue vs first frame, {T}", "Residue", "RMSD (nm)",
     [(np.arange(1,NR+1), resR[c], s) for c, s in sty.items()])
draw("rmsd_per_residue_mean", f"Mean C-alpha RMSD per residue ({NCH} chains), {T}", "Residue", "RMSD (nm)",
     [(np.arange(1,NR+1), np.mean(list(resR.values()), axis=0), dict(color="grey", label="Mean of chains"))])

nmA = np.array([x.strip() for x in u.atoms.names]); rn = u.atoms.resnames
chain = np.arange(len(nmA)) // NA; ridx = u.atoms.resindices - chain*NR
assert ridx.min() == 0 and ridx.max() == NR-1
isH = np.array([x.startswith("H") for x in nmA])
P0 = u.atoms.positions/10; heavy = np.where(~isH)[0]; Hs = np.where(isH)[0]
don = heavy[cKDTree(P0[heavy]).query(P0[Hs])[1]]
kk = np.array([nmA[i][0] in "NO" for i in don]); Hs, don = Hs[kk], don[kk]
acc = np.array([i for i in range(len(nmA)) if nmA[i][0]=="O" or (nmA[i] in ("ND1","NE2") and rn[i].startswith("HI"))])
tot = np.zeros((NCH,NR)); inter = np.zeros((NCH,NR)); nb = []
for ts in u.trajectory:
    Pf = u.atoms.positions/10; k = 0
    for hi, lst in enumerate(cKDTree(Pf[Hs]).query_ball_tree(cKDTree(Pf[acc]), 0.25)):
        D_ = don[hi]
        for aj in lst:
            A = acc[aj]
            if A == D_ or (chain[A]==chain[D_] and ridx[A]==ridx[D_]): continue
            v1, v2 = Pf[Hs[hi]]-Pf[D_], Pf[A]-Pf[D_]; dist = np.linalg.norm(v2)
            if dist > 0.35 or v1@v2/(np.linalg.norm(v1)*dist) < np.cos(np.radians(30)): continue
            k += 1
            for Xa in (D_, A):
                tot[chain[Xa], ridx[Xa]] += 1
                if chain[D_] != chain[A]: inter[chain[Xa], ridx[Xa]] += 1
    nb.append(k)
nf = len(nb); print("mean H-bonds/frame:", round(float(np.mean(nb)),1))
for nmx, arr, ttl in (("hbperres",tot,"All H-bonds per residue"), ("hbperres_inter",inter,"Inter-chain H-bonds per residue")):
    for i, c in enumerate(CH): np.savetxt(f"{X}/{nmx}_{c}.xvg", np.c_[np.arange(1,NR+1), arr[i]/nf], fmt="%d %.5f")
    draw(nmx, f"{ttl}, {T}", "Residue", "H-bonds per frame", [(np.arange(1,NR+1), arr[i]/nf, s) for i, s in enumerate(sty.values())])

# ---- DSSP
print("== dssp")
sh(r'''cd "$OUT/frames"
printf "Protein\n" | gmx dssp -s protein.tpr -f fit_cluster_skip5.xtc -n chains.ndx -o "$OUT/xvgs/dssp_Protein.dat" > "$OUT/logs/dssp_Protein.log" 2>&1
cp "$OUT/xvgs/dssp_Protein.dat" "$OUT/xvgs/dssp_chA.dat"
head -c 80 "$OUT/xvgs/dssp_Protein.dat"; echo''')
a = np.array([list(l.rstrip("\n")) for l in open(f"{X}/dssp_Protein.dat") if l.strip()])
a = a[:, ~(a == "=").all(0)]; print("dssp shape:", a.shape); assert a.shape[1] == NCH*NR, a.shape
Dd = {c: a[:, i*NR:(i+1)*NR] for i, c in enumerate(CH)}; t = np.arange(len(a))*0.005
for name, ss, yl in (("helix","HGI","Helix residues (%)"), ("strand","EB","Strand residues (%)")):
    fig, ax = plt.subplots(figsize=(12,5), dpi=150)
    for c, s in sty.items():
        ax.plot(t, 100*np.isin(Dd[c], list(ss)).mean(1), lw=1.6, **s)
        print(name, c, "mean %:", round(100*np.isin(Dd[c], list(ss)).mean(), 1))
    ax.set(xlabel="Time (ns)", ylabel=yl, title=f"{name.capitalize()} content vs time, {T}")
    ax.legend(ncol=2); fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_{name}_time.png"); plt.show()
    fig, ax = plt.subplots(figsize=(12,5), dpi=150)
    for c, s in sty.items():
        f = np.isin(Dd[c], list(ss)).mean(0); ax.plot(np.arange(1,NR+1), f, lw=1.4, **s)
        np.savetxt(f"{X}/dssp_{name}_perres_{c}.xvg", np.c_[np.arange(1,NR+1), f], fmt="%d %.4f")
    ax.set(xlabel="Residue", ylabel=f"Fraction of frames in {name}", title=f"{name.capitalize()} per residue, {T}")
    ax.legend(ncol=2); fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_{name}_perres.png"); plt.show()
fig, ax = plt.subplots(figsize=(12,5), dpi=150)
for c, s in sty.items(): ax.plot(t, 100*(1-np.isin(Dd[c], list("HGIEB")).mean(1)), lw=1.6, **s)
ax.set(xlabel="Time (ns)", ylabel="Coil/turn residues (%)", title=f"Coil/turn content vs time, {T}")
ax.legend(ncol=2); fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_coil_time.png"); plt.show()
cmap = ListedColormap(["#eeeeee", "#d62728", "#1f77b4"])
code = lambda m: np.where(np.isin(m, list("HGI")), 1, np.where(np.isin(m, list("EB")), 2, 0))
fig, axs = plt.subplots(NCH, 1, figsize=(12,2.2*NCH+1), dpi=150, sharex=True)
for ax, (c, s) in zip(np.atleast_1d(axs), sty.items()):
    ax.imshow(code(Dd[c]), aspect="auto", cmap=cmap, vmin=0, vmax=2, origin="lower", extent=[1,NR,0,1])
    ax.set_ylabel("Time (ns)"); ax.set_title(s["label"], loc="left", fontsize=10)
np.atleast_1d(axs)[-1].set_xlabel("Residue"); fig.suptitle(f"DSSP map: red = helix, blue = strand, grey = coil/turn, {T}")
fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_map.png"); plt.show()

# ---- summary + all-states plots
def last(f, col=1): b = xvg(f); return b[len(b)//2:, col].mean()
h = len(t)//2; rows = []
for c in CH:
    rows.append(dict(chain=c, RMSD_nm=last(f"{X}/rmsd_{c}.xvg"), Rg_nm=last(f"{X}/rg_{c}.xvg"),
      SASA_nm2=last(f"{X}/sasa_{c}.xvg", 2), intra_Hbonds=last(f"{X}/hbnum_{c}.xvg"),
      helix_pct=100*np.isin(Dd[c][h:], list("HGI")).mean(), strand_pct=100*np.isin(Dd[c][h:], list("EB")).mean()))
rows.append(dict(chain="hexamer", RMSD_nm=last(f"{X}/rmsd_Protein.xvg"), Rg_nm=last(f"{X}/rg_Protein.xvg"),
                 SASA_nm2=last(f"{X}/sasa_Protein.xvg"), intra_Hbonds=last(f"{X}/hbnum_Protein.xvg")))
df = pd.DataFrame(rows).round(3); df.to_csv(f"{X}/summary_last0.5ns.csv", index=False); print(df)
STATES = [("Alpha helix","H"),("3-10 helix","G"),("Pi helix","I"),("Beta strand","E"),("Beta bridge","B"),
          ("Turn","T"),("Bend","S"),("Polyproline II helix","P")]
Pc = {k_: 100*(a == c_).mean(1) for k_, c_ in STATES}; Pc["Coil / Unstructured"] = 100 - sum(Pc.values())
tp = np.arange(len(a))*5.0
fig, ax = plt.subplots(figsize=(12,6.2), dpi=150)
for i, (k_, y) in enumerate(Pc.items()): ax.plot(tp, y, color=plt.cm.tab10(i), lw=1.6, label=k_)
ax.set(xlabel="Time (ps)", ylabel="Percentage (%)", title=f"Legumin All Secondary Structure States — {T}")
ax.grid(alpha=.3); ax.legend(title="Secondary structure", ncol=2, loc="upper right"); fig.tight_layout()
fig.savefig(f"{OUT}/graphs/dssp_all_states_{TEMP}K_{RUN}.png"); plt.show()
Gp = {"Alpha helix": Pc["Alpha helix"], "Beta structure": Pc["Beta strand"]+Pc["Beta bridge"], "Turns": Pc["Turn"]}
Gp["Loops / Other"] = 100 - sum(Gp.values())
fig, ax = plt.subplots(figsize=(10,6), dpi=150)
for i, (k_, y) in enumerate(Gp.items()): ax.plot(tp, y, color=plt.cm.tab10(i), lw=1.8, label=k_)
ax.set(xlabel="Time (ps)", ylabel="Percentage (%)", title=f"Legumin Secondary Structure — {T}")
ax.grid(alpha=.3); ax.legend(); fig.tight_layout()
fig.savefig(f"{OUT}/graphs/dssp_groups_{TEMP}K_{RUN}.png"); plt.show()
print("DONE ->", OUT)

In [ ]:
import os, subprocess
cmd = r'''cd "/content/drive/MyDrive/Plant Protein Extrusion MD/analysism" || exit 1
D=legumin_373_shear_r2
echo "== graphs ($(ls $D/graphs/*.png 2>/dev/null | wc -l), expect 23)"
echo; echo "== xvg files that are empty or missing (expect nothing)"
for c in Protein chA chB chC chD chE chF; do for p in rmsd rmsf rg sasa hbnum; do [ -s $D/xvgs/${p}_$c.xvg ] || echo "MISSING/EMPTY: ${p}_$c"; done; done
for a in chA chB chC chD chE chF; do for b in chA chB chC chD chE chF; do [[ "$a" < "$b" ]] && { [ -s $D/xvgs/hbinter_${a}_${b}.xvg ] || echo "MISSING/EMPTY: hbinter_${a}_${b}"; }; done; done
for f in sasa_perres_Protein.xvg dssp_Protein.dat summary_last0.5ns.csv; do [ -s $D/xvgs/$f ] || echo "MISSING/EMPTY: $f"; done
echo; echo "== errors in gmx logs (expect nothing)"; grep -liE "fatal error|segmentation|not found|invalid" $D/logs/*.log
echo; echo "== analysis folders"; ls -d */
for d in $(ls -d */ | tr -d / | grep -E "^legumin_373"); do
  echo; echo "== $d"; cat $d/xvgs/summary_last0.5ns.csv 2>/dev/null || echo "(no summary)"
done'''
r = subprocess.run(["bash","-c",cmd], capture_output=True, text=True); print(r.stdout, r.stderr[-300:])

In [ ]:
import os, subprocess
cmd = r'''cd "/content/drive/MyDrive/Plant Protein Extrusion MD/analysism" || exit 1
for d in legumin_300_r1 legumin_300_r2; do echo "== $d"; grep -E "^(chain|hexamer)" $d/xvgs/summary_last0.5ns.csv; done'''
r = subprocess.run(["bash","-c",cmd], capture_output=True, text=True); print(r.stdout, r.stderr[-300:])

In [ ]:
import os, subprocess
cmd = r'''cd "/content/drive/MyDrive/Plant Protein Extrusion MD/analysism" || exit 1
for d in vicilin_300_r1 vicilin_300_r2; do echo "== $d"; grep -E "^(chain|trimer)" $d/xvgs/summary_last0.5ns.csv; done'''
r = subprocess.run(["bash","-c",cmd], capture_output=True, text=True); print(r.stdout, r.stderr[-300:])

In [ ]:
import os, subprocess
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"): drive.mount('/content/drive')
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
cmd = r'''command -v gmx >/dev/null || { echo "STOP: GROMACS NOT FOUND - run your GROMACS setup cell first"; exit 1; }
cd "/content/drive/MyDrive/Plant Protein Extrusion MD" || { echo "STOP: folder not found - wrong Drive account?"; exit 1; }
ls npt_373_r2.gro npt_373_r2.cpt md_373_r1.mdp md_373_r1.tpr || { echo "STOP: a needed input file is missing"; exit 1; }
if ls md_373_r2.* >/dev/null 2>&1; then echo "STOP: md_373_r2 files already exist:"; ls md_373_r2.*; exit 0; fi
echo; echo "== r1 thermal mdp (key lines)"; grep -iE "^ *(define|nsteps|continuation|gen[-_]vel|pcoupl|tc-grps|ref-t|ld-seed)" md_373_r1.mdp
echo; echo "== r1 thermal launch line"; grep -A1 -m1 "Command line" md_373_r1.log
sed 's/^ld-seed = 111/ld-seed = 222/' md_373_r1.mdp > md_373_r2.mdp
grep -q "^ld-seed = 222" md_373_r2.mdp || { echo "STOP: no 'ld-seed = 111' line found to change"; rm -f md_373_r2.mdp; exit 0; }
R=""; grep -qiE "^ *define" md_373_r1.mdp && R="-r npt_373_r2.gro"
echo; echo "== restraint reference flag used: '$R'"
echo; echo "== mdp diff r1 vs r2 (expect ld-seed only)"; diff md_373_r1.mdp md_373_r2.mdp
echo; echo "== grompp"
gmx grompp -f md_373_r2.mdp -c npt_373_r2.gro $R -t npt_373_r2.cpt -p topol.top -o md_373_r2.tpr -maxwarn 0 > /tmp/g.txt 2>&1
grep -iE "warning|error|fatal|note" /tmp/g.txt | head -10; tail -3 /tmp/g.txt
ls -lh md_373_r1.tpr md_373_r2.tpr
echo; echo "== inputrec differences vs r1 (x, v, resinfo hidden)"
gmx check -s1 md_373_r1.tpr -s2 md_373_r2.tpr 2>&1 | grep -vE "^v\[|^x\[|t_resinfo|^ *$|^comparing|Reading|GROMACS|Executable|Data prefix|Working dir|Command line|gmx check|Note:|^ *[0-9.e+-]+ +[0-9.e+-]+ +[0-9.e+-]+\)"
echo; echo "== r2 thermal files (expect mdp and tpr only)"; ls md_373_r2.*'''
r = subprocess.run(["bash","-c",cmd], capture_output=True, text=True); print(r.stdout, r.stderr[-300:])

In [ ]:
import os, subprocess
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
cmd = r'''command -v gmx >/dev/null || { echo "STOP: GROMACS NOT FOUND - run your GROMACS setup cell first"; exit 1; }
cd "/content/drive/MyDrive/Plant Protein Extrusion MD" || exit 1
echo "== files"; ls -l --time-style=+%d-%b_%H:%M md_373_r2.mdp md_373_r2.tpr npt_373_r2.gro npt_373_r2.cpt
echo; echo "== mdp diff r1 vs r2 (expect ld-seed 111 vs 222 only)"; diff md_373_r1.mdp md_373_r2.mdp && echo IDENTICAL
echo; echo "== tpr settings"; gmx dump -s md_373_r2.tpr 2>/dev/null | head -900 | grep -E "nsteps|ld-seed|continuation|gen-vel|natoms" | head -8
echo; echo "== other md_373_r2 outputs (expect none)"; ls md_373_r2.* | grep -vE "\.(mdp|tpr)$"; echo done'''
p = subprocess.Popen(["bash","-c",cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout: print(line, end="")

In [ ]:
import os, subprocess
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
cmd = r'''command -v gmx >/dev/null || { echo "STOP: GROMACS NOT FOUND - run your GROMACS setup cell first"; exit 1; }
cd "/content/drive/MyDrive/Plant Protein Extrusion MD" || { echo "STOP: folder not found - wrong Drive account?"; exit 1; }
if ls md_373_r2.* | grep -qvE "\.(mdp|tpr)$"; then echo "STOP: md_373_r2 output files already exist:"; ls md_373_r2.*; exit 0; fi
gmx mdrun -deffnm md_373_r2 -cpt 5 -ntmpi 1 -ntomp 2 -pin off 2>&1 | tail -5'''
r = subprocess.run(["bash","-c",cmd], capture_output=True, text=True); print(r.stdout, r.stderr[-300:])

In [ ]:
import os, sys, json, itertools, subprocess
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from google.colab import drive
TEMP, RUN = "373", "r2"
TPR_NAME  = "md_373_r2"                       # thermal r2 tpr
TRAJ_NAME = "md_373_r2"                       # ran in one part
NCH = 6; DEFORM = 0.0                          # hexamer; thermal run, no shear
if not os.path.exists("/content/drive/MyDrive"): drive.mount('/content/drive')
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
if os.system("gmx --version > /dev/null 2>&1") != 0: raise SystemExit("GROMACS not found - run your GROMACS setup cell first, then run this cell again")
os.system('ln -sfn "/content/drive/MyDrive/Plant Protein Extrusion MD" /content/pm')
PM = "/content/drive/MyDrive/Plant Protein Extrusion MD"
if os.path.exists(f"{PM}/md_373_r2.part0002.xtc"): raise SystemExit("md_373_r2 was restarted (part0002 files exist) - combine the parts first; send me the file list")
if not os.path.exists(f"{PM}/md_373_r2.log") or "Finished mdrun" not in open(f"{PM}/md_373_r2.log").read(): raise SystemExit("md_373_r2.log is missing or has no Finished mdrun line - the run did not finish")
OUT = f"/content/pm/analysism/legumin_{TEMP}_{RUN}"; T = "373 K thermal (r2)"
os.environ.update(P="/content/pm", S=TPR_NAME, TRAJ=TRAJ_NAME, OUT=OUT)
for d in ["graphs","logs","frames","xvgs"]: os.makedirs(f"{OUT}/{d}", exist_ok=True)
subprocess.run([sys.executable,"-m","pip","install","-q","MDAnalysis","scipy"])
import MDAnalysis as mda
from MDAnalysis.coordinates.XTC import XTCReader
from MDAnalysis.lib.distances import minimize_vectors
from MDAnalysis.analysis.align import rotation_matrix
from scipy.spatial import cKDTree
from matplotlib.colors import ListedColormap
def sh(cmd, tail=0):
    r = subprocess.run(["bash","-c",cmd], capture_output=True, text=True)
    o = (r.stdout + r.stderr).strip().split("\n")
    print("\n".join(o[-tail:] if tail else o))

# ---- check files, mdp, and trajectory continuity
print("== files"); sh('ls -lh $P/$S.tpr $P/$TRAJ.xtc; grep -iE "ref[-_]t|deform" $P/$S.mdp')
rd = XTCReader(f"{OUT.rsplit('/analysism',1)[0]}/{TRAJ_NAME}.xtc")
times = np.array([ts.time for ts in rd]); dims = np.array([ts.dimensions.copy() for ts in rd])
print("frames:", len(times), "| time", times[0], "->", times[-1], "| natoms:", rd.n_atoms)
assert rd.n_atoms < 100000, "xtc is not protein-only (whole system) - send me this number"
assert times[0] == 0 and abs(times[-1]-1000) < 1e-3 and len(times) == 1001 and np.all(np.diff(times) == 1), "trajectory not continuous 0-1000 ps"
print("box a,b,c start:", dims[0,:3].round(2), "end:", dims[-1,:3].round(2))
print("angles start:", dims[0,3:].round(2), "end:", dims[-1,3:].round(2))
if DEFORM:
    gd = DEFORM/(dims[0,1]/10); print(f"shear rate {gd:.4f} /ns -> expected rigid rotation {-0.5*gd*180/np.pi:.2f} deg/ns about z")
else: print("thermal run: box angles should stay 60/60/90")

# ---- protein tpr, whole, first frame
print("== whole")
sh(r'''cd "$OUT/frames"
printf "Protein\n" | gmx convert-tpr -s "$P/$S.tpr" -o protein.tpr 2>&1 | tail -1
printf "Protein\n" | gmx trjconv -s protein.tpr -f "$P/$TRAJ.xtc" -o whole2.xtc -pbc whole 2>&1 | tail -1
printf "Protein\n" | gmx trjconv -s protein.tpr -f whole2.xtc -o first_w.gro -dump 0 2>&1 | tail -1''')

# ---- chain detection from .gro
lines = open(f"{OUT}/frames/first_w.gro").read().split("\n")
n = int(lines[1]); body = lines[2:2+n]
resnum = np.array([int(l[0:5]) for l in body]); nm = [l[10:15].strip() for l in body]
starts = [0] + [i for i in range(1, n) if resnum[i] < resnum[i-1]]; ends = starts[1:] + [n]
nres = [len(set(resnum[s:e])) for s, e in zip(starts, ends)]
print("atoms:", n, "| chains:", len(starts), "| atoms/chain:", [e-s for s,e in zip(starts,ends)], "| res/chain:", nres)
assert len(starts) == NCH, f"expected {NCH} chains - send me the line above"
assert len(set(e-s for s,e in zip(starts,ends))) == 1, "chains differ in size - send me the line above"
NR = nres[0]; CH = ["ch"+"ABCDEF"[i] for i in range(NCH)]; G = {}; ranges = {}
for k, (s, e) in enumerate(zip(starts, ends)):
    r = range(s+1, e+1); c = CH[k]; ranges[c] = [s, e]
    G[c] = list(r); G[c+"_bb"] = [i for i in r if nm[i-1] in ("N","CA","C")]; G[c+"_CA"] = [i for i in r if nm[i-1]=="CA"]
r = range(1, n+1)
G["Protein"] = list(r); G["Protein_bb"] = [i for i in r if nm[i-1] in ("N","CA","C")]; G["Protein_CA"] = [i for i in r if nm[i-1]=="CA"]
with open(f"{OUT}/frames/chains.ndx","w") as f:
    for g, idx in G.items():
        f.write(f"[ {g} ]\n")
        for k in range(0, len(idx), 15): f.write(" ".join(f"{i:5d}" for i in idx[k:k+15]) + "\n")
json.dump(dict(ranges=ranges, nres=NR, natoms=n), open(f"{OUT}/frames/chains.json","w"))

# ---- keep chains together (triclinic-safe)
sl = [slice(*ranges[c]) for c in CH]
u = mda.Universe(f"{OUT}/frames/first_w.gro", f"{OUT}/frames/whole2.xtc")
with mda.Writer(f"{OUT}/frames/together.xtc", u.atoms.n_atoms) as W:
    for ts in u.trajectory[::5]:
        pos = u.atoms.positions.copy(); box = ts.dimensions.astype(np.float32); a0 = pos[sl[0]].mean(0)
        for s in sl[1:]:
            d = pos[s].mean(0) - a0
            dm = minimize_vectors(d[None,:].astype(np.float32), box)[0]; pos[s] -= (d - dm)
        u.atoms.positions = pos; W.write(u.atoms)

# ---- fit
print("== fit")
sh(r'''cd "$OUT/frames"
printf "Backbone\nProtein\n" | gmx trjconv -s protein.tpr -f together.xtc -o fit_cluster_skip5.xtc -fit rot+trans 2>&1 | tail -1
printf "Protein\n" | gmx trjconv -s protein.tpr -f fit_cluster_skip5.xtc -o first_cl.gro -dump 0 2>&1 | tail -1
rm -f whole2.xtc together.xtc
gmx check -f fit_cluster_skip5.xtc 2>&1 | grep -E "Last frame|^Step"''')

# ---- hexamer integrity check
u = mda.Universe(f"{OUT}/frames/first_cl.gro", f"{OUT}/frames/fit_cluster_skip5.xtc")
pairs = list(itertools.combinations(range(NCH), 2)); worst = {p: 0.0 for p in pairs}; lonely = 0
for ts in u.trajectory:
    Pp = u.atoms.positions/10; ch = [Pp[s] for s in sl]
    m = {p: cKDTree(ch[p[0]]).query(ch[p[1]])[0].min() for p in pairs}
    for p in pairs: worst[p] = max(worst[p], m[p])
    for i in range(NCH):
        if min(m[p] for p in pairs if i in p) > 0.5: lonely += 1
print({"ABCDEF"[a]+"ABCDEF"[b]: round(float(v),2) for (a,b), v in worst.items()})
print("chains with no neighbour within 0.5 nm:", lonely)
assert lonely == 0, "a chain drifted away"

# ---- GROMACS analysis
print("== gmx analysis (slow)")
PAIRSTR = " ".join(f'"{a} {b}"' for a, b in itertools.combinations(CH, 2))
os.environ.update(CHS=" ".join(["Protein"]+CH), PAIRSTR=PAIRSTR)
sh(r'''cd "$OUT/frames"; X="$OUT/xvgs"; L="$OUT/logs"; T=fit_cluster_skip5.xtc
for c in $CHS; do
  printf "${c}_bb\n${c}_bb\n" | gmx rms  -s first_cl.gro -f $T -n chains.ndx -o $X/rmsd_$c.xvg -tu ns > $L/rmsd_$c.log 2>&1
  printf "${c}_CA\n${c}_CA\n" | gmx rmsf -s first_cl.gro -f $T -n chains.ndx -o $X/rmsf_$c.xvg -res   > $L/rmsf_$c.log 2>&1
  printf "$c\n"               | gmx gyrate -s protein.tpr -f $T -n chains.ndx -o $X/rg_$c.xvg          > $L/rg_$c.log 2>&1
  gmx sasa -s protein.tpr -f $T -n chains.ndx -surface "group Protein" -output "group $c" -o $X/sasa_$c.xvg -or $X/tmp_$c.xvg -tu ns > $L/sasa_$c.log 2>&1
  gmx hbond -s protein.tpr -f $T -n chains.ndx -r "group $c" -t "group $c" -num $X/hbnum_$c.xvg -tu ns > $L/hbond_$c.log 2>&1
done
eval "for p in $PAIRSTR; do set -- \$p
  gmx hbond -s protein.tpr -f $T -n chains.ndx -r \"group \$1\" -t \"group \$2\" -num $X/hbinter_\$1_\$2.xvg -tu ns > $L/hbinter_\$1_\$2.log 2>&1
done"
mv $X/tmp_Protein.xvg $X/sasa_perres_Protein.xvg; rm -f $X/tmp_ch*.xvg
for f in rmsd_Protein rmsd_chA rmsf_chA rg_Protein sasa_chA hbnum_Protein hbinter_chA_chB; do echo $f $(tail -1 $X/$f.xvg); done''')

# ---- plots
X = f"{OUT}/xvgs"
cols = ["#d62728","#1f77b4","#2ca02c","#ff9896","#7fbfff","#98df8a"]
sty = {c: dict(color=cols[i], ls="-" if i%2==0 else ":", label="Chain "+c[-1]) for i, c in enumerate(CH)}
xvg = lambda f: np.loadtxt(f, comments=('#','@'))
def draw(out, title, xl, yl, items):
    fig, ax = plt.subplots(figsize=(12,5), dpi=150)
    for x, y, s in items: ax.plot(x, y, lw=1.6, **s)
    ax.set(xlabel=xl, ylabel=yl, title=title); ax.legend(ncol=2); fig.tight_layout()
    fig.savefig(f"{OUT}/graphs/{out}.png"); plt.show()
chains = lambda pre, col=1, sc=1: [(xvg(f"{X}/{pre}_{c}.xvg")[:,0]/sc, xvg(f"{X}/{pre}_{c}.xvg")[:,col], s) for c, s in sty.items()]
def whole(pre, col=1, sc=1):
    a = xvg(f"{X}/{pre}_Protein.xvg"); return [(a[:,0]/sc, a[:,col], dict(color="grey", label="Whole hexamer"))]
for pre, yl, ttl, sc in (("rmsd","RMSD (nm)","Backbone RMSD",1), ("rg","Rg (nm)","Radius of gyration",1000),
                         ("sasa","SASA (nm²)","SASA",1), ("hbnum","H-bonds","Intra-chain H-bonds",1)):
    cc = 2 if pre == "sasa" else 1
    draw(f"{pre}_per_chain", f"{ttl} per chain, {T}", "Time (ns)", yl, chains(pre, cc, sc))
    draw(f"{pre}_protein", f"{ttl}, whole hexamer, {T}", "Time (ns)", yl, whole(pre, 1, sc))
draw("rmsf_per_chain", f"C-alpha RMSF per residue, {T}", "Residue", "RMSF (nm)", chains("rmsf"))
d = xvg(f"{X}/sasa_perres_Protein.xvg"); assert len(d) == NCH*NR, len(d)
draw("sasa_per_residue", f"SASA per residue, {T}", "Residue", "SASA (nm²)",
     [(np.arange(1,NR+1), d[i*NR:(i+1)*NR,1], s) for i, s in enumerate(sty.values())])
fig, ax = plt.subplots(figsize=(12,6), dpi=150)
for k, (a_, b_) in enumerate(itertools.combinations(CH, 2)):
    h = xvg(f"{X}/hbinter_{a_}_{b_}.xvg"); ax.plot(h[:,0], h[:,1], lw=1.2, color=plt.cm.tab10(k), label=f"{a_[-1]}–{b_[-1]}")
ax.set(xlabel="Time (ns)", ylabel="H-bonds", title=f"Inter-chain H-bonds, {T}"); ax.legend(ncol=3, fontsize=8)
fig.tight_layout(); fig.savefig(f"{OUT}/graphs/hbonds_inter_chain.png"); plt.show()

# ---- per-residue RMSD and H-bonds
u = mda.Universe(f"{OUT}/frames/first_cl.gro", f"{OUT}/frames/fit_cluster_skip5.xtc")
NA = u.atoms.n_atoms // NCH; cs = {c: slice(i*NA, (i+1)*NA) for i, c in enumerate(CH)}; resR = {}
for name, s in cs.items():
    chn = u.atoms[s]; bb = chn.select_atoms("name N CA C"); ca = chn.select_atoms("name CA")
    u.trajectory[0]; c0 = bb.positions.mean(0); bb0 = bb.positions - c0; ca0 = ca.positions - c0
    d2 = np.zeros(len(ca)); k = 0
    for ts in u.trajectory:
        c = bb.positions.mean(0); R, _ = rotation_matrix(bb.positions - c, bb0)
        d2 += (((ca.positions - c) @ R.T - ca0)**2).sum(1); k += 1
    resR[name] = np.sqrt(d2/k)/10
    np.savetxt(f"{X}/rmsd_perres_{name}.xvg", np.c_[np.arange(1,NR+1), resR[name]], fmt="%d %.6f")
draw("rmsd_per_residue", f"C-alpha RMSD per residue vs first frame, {T}", "Residue", "RMSD (nm)",
     [(np.arange(1,NR+1), resR[c], s) for c, s in sty.items()])
draw("rmsd_per_residue_mean", f"Mean C-alpha RMSD per residue ({NCH} chains), {T}", "Residue", "RMSD (nm)",
     [(np.arange(1,NR+1), np.mean(list(resR.values()), axis=0), dict(color="grey", label="Mean of chains"))])

nmA = np.array([x.strip() for x in u.atoms.names]); rn = u.atoms.resnames
chain = np.arange(len(nmA)) // NA; ridx = u.atoms.resindices - chain*NR
assert ridx.min() == 0 and ridx.max() == NR-1
isH = np.array([x.startswith("H") for x in nmA])
P0 = u.atoms.positions/10; heavy = np.where(~isH)[0]; Hs = np.where(isH)[0]
don = heavy[cKDTree(P0[heavy]).query(P0[Hs])[1]]
kk = np.array([nmA[i][0] in "NO" for i in don]); Hs, don = Hs[kk], don[kk]
acc = np.array([i for i in range(len(nmA)) if nmA[i][0]=="O" or (nmA[i] in ("ND1","NE2") and rn[i].startswith("HI"))])
tot = np.zeros((NCH,NR)); inter = np.zeros((NCH,NR)); nb = []
for ts in u.trajectory:
    Pf = u.atoms.positions/10; k = 0
    for hi, lst in enumerate(cKDTree(Pf[Hs]).query_ball_tree(cKDTree(Pf[acc]), 0.25)):
        D_ = don[hi]
        for aj in lst:
            A = acc[aj]
            if A == D_ or (chain[A]==chain[D_] and ridx[A]==ridx[D_]): continue
            v1, v2 = Pf[Hs[hi]]-Pf[D_], Pf[A]-Pf[D_]; dist = np.linalg.norm(v2)
            if dist > 0.35 or v1@v2/(np.linalg.norm(v1)*dist) < np.cos(np.radians(30)): continue
            k += 1
            for Xa in (D_, A):
                tot[chain[Xa], ridx[Xa]] += 1
                if chain[D_] != chain[A]: inter[chain[Xa], ridx[Xa]] += 1
    nb.append(k)
nf = len(nb); print("mean H-bonds/frame:", round(float(np.mean(nb)),1))
for nmx, arr, ttl in (("hbperres",tot,"All H-bonds per residue"), ("hbperres_inter",inter,"Inter-chain H-bonds per residue")):
    for i, c in enumerate(CH): np.savetxt(f"{X}/{nmx}_{c}.xvg", np.c_[np.arange(1,NR+1), arr[i]/nf], fmt="%d %.5f")
    draw(nmx, f"{ttl}, {T}", "Residue", "H-bonds per frame", [(np.arange(1,NR+1), arr[i]/nf, s) for i, s in enumerate(sty.values())])

# ---- DSSP
print("== dssp")
sh(r'''cd "$OUT/frames"
printf "Protein\n" | gmx dssp -s protein.tpr -f fit_cluster_skip5.xtc -n chains.ndx -o "$OUT/xvgs/dssp_Protein.dat" > "$OUT/logs/dssp_Protein.log" 2>&1
cp "$OUT/xvgs/dssp_Protein.dat" "$OUT/xvgs/dssp_chA.dat"
head -c 80 "$OUT/xvgs/dssp_Protein.dat"; echo''')
a = np.array([list(l.rstrip("\n")) for l in open(f"{X}/dssp_Protein.dat") if l.strip()])
a = a[:, ~(a == "=").all(0)]; print("dssp shape:", a.shape); assert a.shape[1] == NCH*NR, a.shape
Dd = {c: a[:, i*NR:(i+1)*NR] for i, c in enumerate(CH)}; t = np.arange(len(a))*0.005
for name, ss, yl in (("helix","HGI","Helix residues (%)"), ("strand","EB","Strand residues (%)")):
    fig, ax = plt.subplots(figsize=(12,5), dpi=150)
    for c, s in sty.items():
        ax.plot(t, 100*np.isin(Dd[c], list(ss)).mean(1), lw=1.6, **s)
        print(name, c, "mean %:", round(100*np.isin(Dd[c], list(ss)).mean(), 1))
    ax.set(xlabel="Time (ns)", ylabel=yl, title=f"{name.capitalize()} content vs time, {T}")
    ax.legend(ncol=2); fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_{name}_time.png"); plt.show()
    fig, ax = plt.subplots(figsize=(12,5), dpi=150)
    for c, s in sty.items():
        f = np.isin(Dd[c], list(ss)).mean(0); ax.plot(np.arange(1,NR+1), f, lw=1.4, **s)
        np.savetxt(f"{X}/dssp_{name}_perres_{c}.xvg", np.c_[np.arange(1,NR+1), f], fmt="%d %.4f")
    ax.set(xlabel="Residue", ylabel=f"Fraction of frames in {name}", title=f"{name.capitalize()} per residue, {T}")
    ax.legend(ncol=2); fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_{name}_perres.png"); plt.show()
fig, ax = plt.subplots(figsize=(12,5), dpi=150)
for c, s in sty.items(): ax.plot(t, 100*(1-np.isin(Dd[c], list("HGIEB")).mean(1)), lw=1.6, **s)
ax.set(xlabel="Time (ns)", ylabel="Coil/turn residues (%)", title=f"Coil/turn content vs time, {T}")
ax.legend(ncol=2); fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_coil_time.png"); plt.show()
cmap = ListedColormap(["#eeeeee", "#d62728", "#1f77b4"])
code = lambda m: np.where(np.isin(m, list("HGI")), 1, np.where(np.isin(m, list("EB")), 2, 0))
fig, axs = plt.subplots(NCH, 1, figsize=(12,2.2*NCH+1), dpi=150, sharex=True)
for ax, (c, s) in zip(np.atleast_1d(axs), sty.items()):
    ax.imshow(code(Dd[c]), aspect="auto", cmap=cmap, vmin=0, vmax=2, origin="lower", extent=[1,NR,0,1])
    ax.set_ylabel("Time (ns)"); ax.set_title(s["label"], loc="left", fontsize=10)
np.atleast_1d(axs)[-1].set_xlabel("Residue"); fig.suptitle(f"DSSP map: red = helix, blue = strand, grey = coil/turn, {T}")
fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_map.png"); plt.show()

# ---- summary + all-states plots
def last(f, col=1): b = xvg(f); return b[len(b)//2:, col].mean()
h = len(t)//2; rows = []
for c in CH:
    rows.append(dict(chain=c, RMSD_nm=last(f"{X}/rmsd_{c}.xvg"), Rg_nm=last(f"{X}/rg_{c}.xvg"),
      SASA_nm2=last(f"{X}/sasa_{c}.xvg", 2), intra_Hbonds=last(f"{X}/hbnum_{c}.xvg"),
      helix_pct=100*np.isin(Dd[c][h:], list("HGI")).mean(), strand_pct=100*np.isin(Dd[c][h:], list("EB")).mean()))
rows.append(dict(chain="hexamer", RMSD_nm=last(f"{X}/rmsd_Protein.xvg"), Rg_nm=last(f"{X}/rg_Protein.xvg"),
                 SASA_nm2=last(f"{X}/sasa_Protein.xvg"), intra_Hbonds=last(f"{X}/hbnum_Protein.xvg")))
df = pd.DataFrame(rows).round(3); df.to_csv(f"{X}/summary_last0.5ns.csv", index=False); print(df)
STATES = [("Alpha helix","H"),("3-10 helix","G"),("Pi helix","I"),("Beta strand","E"),("Beta bridge","B"),
          ("Turn","T"),("Bend","S"),("Polyproline II helix","P")]
Pc = {k_: 100*(a == c_).mean(1) for k_, c_ in STATES}; Pc["Coil / Unstructured"] = 100 - sum(Pc.values())
tp = np.arange(len(a))*5.0
fig, ax = plt.subplots(figsize=(12,6.2), dpi=150)
for i, (k_, y) in enumerate(Pc.items()): ax.plot(tp, y, color=plt.cm.tab10(i), lw=1.6, label=k_)
ax.set(xlabel="Time (ps)", ylabel="Percentage (%)", title=f"Legumin All Secondary Structure States — {T}")
ax.grid(alpha=.3); ax.legend(title="Secondary structure", ncol=2, loc="upper right"); fig.tight_layout()
fig.savefig(f"{OUT}/graphs/dssp_all_states_{TEMP}K_{RUN}.png"); plt.show()
Gp = {"Alpha helix": Pc["Alpha helix"], "Beta structure": Pc["Beta strand"]+Pc["Beta bridge"], "Turns": Pc["Turn"]}
Gp["Loops / Other"] = 100 - sum(Gp.values())
fig, ax = plt.subplots(figsize=(10,6), dpi=150)
for i, (k_, y) in enumerate(Gp.items()): ax.plot(tp, y, color=plt.cm.tab10(i), lw=1.8, label=k_)
ax.set(xlabel="Time (ps)", ylabel="Percentage (%)", title=f"Legumin Secondary Structure — {T}")
ax.grid(alpha=.3); ax.legend(); fig.tight_layout()
fig.savefig(f"{OUT}/graphs/dssp_groups_{TEMP}K_{RUN}.png"); plt.show()
print("DONE ->", OUT)

## 13. All-runs comparison
Summary tables and plots across both replicates of every condition.

In [ ]:
import os, itertools, numpy as np, pandas as pd, matplotlib.pyplot as plt
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"): drive.mount('/content/drive')
BASE = "/content/drive/MyDrive/Plant Protein Extrusion MD/analysism"
RUNS = {"300 K r1":"legumin_300_r1", "300 K r2":"legumin_300_r2",
        "373 K r1":"legumin_373_r1", "373 K r2":"legumin_373_r2",
        "373 K shear r1":"legumin_373_shear_r1", "373 K shear r2":"legumin_373_shear_r2"}
COL = {"300 K r1":"#1f77b4", "300 K r2":"#7fbfff", "373 K r1":"#d62728", "373 K r2":"#ff9896",
       "373 K shear r1":"#2ca02c", "373 K shear r2":"#98df8a"}
LS  = {r: ("-" if r.endswith("r1") else "--") for r in RUNS}
COND = {"300 K thermal":("300 K r1","300 K r2"), "373 K thermal":("373 K r1","373 K r2"),
        "373 K shear":("373 K shear r1","373 K shear r2")}
CMP = f"{BASE}/comparison_legumin_with_r2"
for d in ("graphs","csv"): os.makedirs(f"{CMP}/{d}", exist_ok=True)
NR = 496; NCH = 6; CH = ["chA","chB","chC","chD","chE","chF"]
PAIRS = list(itertools.combinations(CH, 2)); PN = [f"{a[-1]}-{b[-1]}" for a, b in PAIRS]
STAT = [("Alpha helix","H"),("3-10 helix","G"),("Pi helix","I"),("Beta strand","E"),
        ("Beta bridge","B"),("Turn","T"),("Bend","S"),("PPII helix","P")]
xvg = lambda f: np.loadtxt(f, comments=('#','@'))
ok = os.path.exists
def load_dssp(p):
    a = np.array([list(l.rstrip("\n")) for l in open(p) if l.strip()])
    return a[:, ~(a == "=").all(0)]
miss = [r for r, fd in RUNS.items() if not os.path.isdir(f"{BASE}/{fd}/xvgs")]
if miss: raise SystemExit(f"MISSING run folder(s): {miss} - send me the 'ls analysism' output")

# ---------------- load everything ----------------
rows, resrows, comp = [], [], {}
def add(run, m, e, t, y): rows.extend((run, m, e, float(a), float(b)) for a, b in zip(t, y))
spec = {"RMSD (nm)":("rmsd",1,1.0), "Rg (nm)":("rg",1,1000.0),
        "SASA (nm2)":("sasa",2,1.0), "H-bonds (hexamer)":("hbnum",1,1.0)}
RESPAT = {"RMSF (nm)":"rmsf_{c}", "RMSD per residue (nm)":"rmsd_perres_{c}",
          "H-bonds per residue":"hbperres_{c}", "Inter-chain H-bonds per residue":"hbperres_inter_{c}",
          "Helix fraction":"dssp_helix_perres_{c}", "Strand fraction":"dssp_strand_perres_{c}"}
for run, fd in RUNS.items():
    X = f"{BASE}/{fd}/xvgs"
    for m, (pre, col, sc) in spec.items():
        for e in CH + ["Protein"]:
            f = f"{X}/{pre}_{e}.xvg"
            if ok(f): a = xvg(f); add(run, m, e, a[:,0]/sc, a[:,col])
            else: print("missing", run, f"{pre}_{e}.xvg")
    for (a_, b_), n in zip(PAIRS, PN):
        f = f"{X}/hbinter_{a_}_{b_}.xvg"
        if ok(f): a = xvg(f); add(run, "Inter H-bonds", n, a[:,0], a[:,1])
        else: print("missing", run, f"hbinter_{a_}_{b_}.xvg")
    f = next((p for p in (f"{X}/dssp_chA.dat", f"{X}/dssp_Protein.dat") if ok(p)), None)
    if f:
        a = load_dssp(f)
        if a.shape[1] != NCH*NR: print("DSSP shape mismatch, skipped:", run, a.shape)
        else:
            n = len(a); t = np.linspace(0, 1, n)
            ents = {"Protein": a, **{c: a[:, i*NR:(i+1)*NR] for i, c in enumerate(CH)}}
            for e, mm in ents.items():
                hel = 100*np.isin(mm, list("HGI")).mean(1); stn = 100*np.isin(mm, list("EB")).mean(1)
                add(run, "Helix (%)", e, t, hel); add(run, "Strand (%)", e, t, stn)
                add(run, "Coil/turn (%)", e, t, 100-hel-stn)
            comp[run] = {nm: 100*(a[n//2:] == ch).mean() for nm, ch in STAT}
            comp[run]["Coil/other"] = 100 - sum(comp[run].values())
    else: print("missing DSSP file", run)
    for c in CH:
        for m, pat in RESPAT.items():
            f = f"{X}/{pat.format(c=c)}.xvg"
            if ok(f):
                v = xvg(f)[:,1]
                if len(v) == NR: resrows.extend((run, m, c, i+1, float(x)) for i, x in enumerate(v))
    f = f"{X}/sasa_perres_Protein.xvg"
    if ok(f):
        d = xvg(f)
        if len(d) == NCH*NR:
            for i, c in enumerate(CH):
                resrows.extend((run, "SASA per residue (nm2)", c, j+1, float(x)) for j, x in enumerate(d[i*NR:(i+1)*NR, 1]))
ts  = pd.DataFrame(rows, columns=["run","metric","entity","time_ns","value"])
res = pd.DataFrame(resrows, columns=["run","metric","chain","residue","value"])
ts.to_csv(f"{CMP}/csv/timeseries_long.csv", index=False)
res.to_csv(f"{CMP}/csv/residue_metrics_long.csv", index=False)
pd.DataFrame(comp).T.round(2).to_csv(f"{CMP}/csv/dssp_composition_last_half.csv")

# ---------------- summaries ----------------
def stats(t, y):
    h = t >= t.max()/2
    return dict(mean_full=y.mean(), sd_full=y.std(ddof=1), mean_last_half=y[h].mean(), sd_last_half=y[h].std(ddof=1))
pc = [dict(run=r, metric=m, entity=e, **stats(d.sort_values("time_ns").time_ns.values, d.sort_values("time_ns").value.values))
      for (r, m, e), d in ts.groupby(["run","metric","entity"])]
pd.DataFrame(pc).round(4).to_csv(f"{CMP}/csv/summary_per_chain_and_interface.csv", index=False)
MET = [("RMSD (nm)",["Protein"],"mean","Backbone RMSD, whole hexamer (nm)"),
       ("Rg (nm)",["Protein"],"mean","Radius of gyration, whole hexamer (nm)"),
       ("SASA (nm2)",["Protein"],"mean","SASA, whole hexamer (nm²)"),
       ("H-bonds (hexamer)",["Protein"],"mean","H-bonds within the hexamer"),
       ("Inter H-bonds",PN,"sum","Inter-chain H-bonds (sum of 15 pairs)"),
       ("Helix (%)",["Protein"],"mean","Helix (%)"),
       ("Strand (%)",["Protein"],"mean","Strand (%)"),
       ("Coil/turn (%)",["Protein"],"mean","Coil/turn (%)")]
def curve(run, m, ents, agg):
    d = ts[(ts.run==run) & (ts.metric==m) & (ts.entity.isin(ents))]
    if d.empty: return None
    pr = [e for e in ents if (d.entity==e).any()]
    t0 = d[d.entity==pr[0]].sort_values("time_ns").time_ns.values
    Y = np.array([np.interp(t0, *[d[d.entity==e].sort_values("time_ns")[k].values for k in ("time_ns","value")]) for e in pr])
    return t0, (Y.sum(0) if agg=="sum" else Y.mean(0))
agg_rows = []
for run in RUNS:
    for m, ents, agg, yl in MET:
        c = curve(run, m, ents, agg)
        if c: agg_rows.append(dict(run=run, metric=m, **stats(*c)))
S = pd.DataFrame(agg_rows); S.round(4).to_csv(f"{CMP}/csv/summary_all_runs.csv", index=False)
table = S.pivot(index="run", columns="metric", values="mean_last_half").reindex(list(RUNS))
table = table[[m[0] for m in MET if m[0] in table.columns]]
table.round(4).to_csv(f"{CMP}/csv/last_half_means_by_run.csv")

# ---------------- conditions: replicate spread vs effect ----------------
crow = []
for m in table.columns:
    g = {c: (table.loc[a, m], table.loc[b, m]) for c, (a, b) in COND.items()}
    mean = {c: float(np.mean(v)) for c, v in g.items()}; gap = {c: abs(v[0]-v[1]) for c, v in g.items()}
    for name, hi, lo in (("thermal: 373 K - 300 K", "373 K thermal", "300 K thermal"),
                         ("shear: 373 K shear - 373 K thermal", "373 K shear", "373 K thermal")):
        dlt = mean[hi] - mean[lo]
        sep = bool(min(g[hi]) > max(g[lo]) or max(g[hi]) < min(g[lo]))
        crow.append(dict(metric=m, contrast=name, baseline=mean[lo], value=mean[hi], delta=dlt,
                         pct=100*dlt/mean[lo], gap_baseline=gap[lo], gap_value=gap[hi],
                         bigger_than_both_gaps=abs(dlt) > max(gap[lo], gap[hi]), replicates_separated=sep))
CT = pd.DataFrame(crow); CT.round(4).to_csv(f"{CMP}/csv/condition_contrasts.csv", index=False)

# ---------------- graphs: time series ----------------
def ts_ax(ax, m, ents, agg, yl):
    for run in RUNS:
        c = curve(run, m, ents, agg)
        if c: ax.plot(c[0], c[1], color=COL[run], ls=LS[run], lw=1.5, label=run)
    ax.set(xlabel="Time (ns)", ylabel=yl, title=yl); ax.grid(alpha=.3); ax.legend(fontsize=7)
fig, axs = plt.subplots(4, 2, figsize=(14, 17))
for ax, (m, ents, agg, yl) in zip(axs.ravel(), MET): ts_ax(ax, m, ents, agg, yl)
fig.suptitle("Legumin hexamer: all runs", y=1.0); fig.tight_layout()
fig.savefig(f"{CMP}/graphs/00_overview_timeseries.png", dpi=150); plt.show()
for i, (m, ents, agg, yl) in enumerate(MET, 1):
    f2, a2 = plt.subplots(figsize=(10, 5), dpi=150); ts_ax(a2, m, ents, agg, yl); f2.tight_layout()
    f2.savefig(f"{CMP}/graphs/ts_{m.split(' (')[0].replace(' ','_').replace('/','_')}.png"); plt.close(f2)

# ---------------- graphs: bars ----------------
fig, axs = plt.subplots(2, 4, figsize=(18, 7))
for ax, (m, ents, agg, yl) in zip(axs.ravel(), MET):
    d = S[S.metric==m].set_index("run").reindex(list(RUNS)).dropna()
    ax.bar(d.index, d.mean_last_half, yerr=d.sd_last_half, color=[COL[r] for r in d.index], capsize=3)
    ax.set_title(m, fontsize=10); ax.tick_params(axis="x", rotation=40); ax.grid(axis="y", alpha=.3)
fig.suptitle("Last-half mean ± SD over time (whole hexamer)"); fig.tight_layout()
fig.savefig(f"{CMP}/graphs/01_bar_summary.png", dpi=150); plt.show()
P = pd.DataFrame(pc)
fig, axs = plt.subplots(2, 3, figsize=(18, 8)); w = 0.13
for ax, m in zip(axs.ravel(), ["RMSD (nm)","Rg (nm)","SASA (nm2)","H-bonds (hexamer)","Helix (%)","Strand (%)"]):
    for k, run in enumerate(RUNS):
        d = P[(P.run==run)&(P.metric==m)&(P.entity.isin(CH))].set_index("entity").reindex(CH)
        ax.bar(np.arange(NCH)+k*w, d.mean_last_half, w, yerr=d.sd_last_half, color=COL[run], label=run, capsize=2)
    ax.set_xticks(np.arange(NCH)+2.5*w); ax.set_xticklabels([f"Chain {c[-1]}" for c in CH]); ax.set_title(m); ax.grid(axis="y", alpha=.3)
axs[0,0].legend(fontsize=7); fig.suptitle("Per-chain last-half mean ± SD"); fig.tight_layout()
fig.savefig(f"{CMP}/graphs/02_per_chain_bars.png", dpi=150); plt.show()

# heatmap of % change vs 300 K baseline
base = table.loc[["300 K r1","300 K r2"]].mean(); pct = 100*(table - base)/base
fig, ax = plt.subplots(figsize=(12, 4.4)); v = np.nanpercentile(np.abs(pct.values), 95)
im = ax.imshow(pct.values, cmap="RdBu_r", vmin=-v, vmax=v, aspect="auto")
ax.set_xticks(range(pct.shape[1])); ax.set_xticklabels(pct.columns, rotation=30, ha="right")
ax.set_yticks(range(pct.shape[0])); ax.set_yticklabels(pct.index)
for i in range(pct.shape[0]):
    for j in range(pct.shape[1]): ax.text(j, i, f"{pct.values[i,j]:.1f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, label="% change vs mean of 300 K r1/r2"); ax.set_title("Change relative to the 300 K baseline (%)")
fig.tight_layout(); fig.savefig(f"{CMP}/graphs/03_pct_change_heatmap.png", dpi=150); plt.show()

# DSSP composition
if comp:
    C = pd.DataFrame(comp).T.reindex([r for r in RUNS if r in comp])
    ax = C.plot(kind="bar", stacked=True, figsize=(11, 5), colormap="tab20"); ax.set_ylabel("% of residues (last half)")
    ax.set_title("DSSP composition, whole hexamer"); ax.legend(ncol=2, fontsize=8, bbox_to_anchor=(1.01, 1))
    plt.xticks(rotation=25); plt.tight_layout(); plt.savefig(f"{CMP}/graphs/04_dssp_composition.png", dpi=150); plt.show()

# replicate dot plot: the two replicates of each condition side by side
fig, axs = plt.subplots(2, 4, figsize=(17, 8)); cn = list(COND)
for ax, m in zip(axs.ravel(), table.columns):
    for k, (c, (a_, b_)) in enumerate(COND.items()):
        ya, yb = table.loc[a_, m], table.loc[b_, m]
        ax.plot([k, k], [ya, yb], color="grey", lw=1)
        ax.scatter([k], [ya], color=COL[a_], s=60, zorder=3, label="r1" if k == 0 else None, edgecolor="k")
        ax.scatter([k], [yb], color=COL[b_], s=60, zorder=3, marker="s", label="r2" if k == 0 else None, edgecolor="k")
    ax.set_xticks(range(3)); ax.set_xticklabels([c.replace(" ", "\n", 1) for c in cn], fontsize=8)
    ax.set_xlim(-.5, 2.5); ax.set_title(m, fontsize=10); ax.grid(axis="y", alpha=.3)
axs[0,0].legend(fontsize=8)
fig.suptitle("Replicates per condition (last-half mean, whole hexamer): a real effect separates the pairs"); fig.tight_layout()
fig.savefig(f"{CMP}/graphs/08_replicate_dotplot.png", dpi=150); plt.show()

# ---------------- graphs: per residue ----------------
rm = res.groupby(["run","metric","residue"]).value.mean()
def rcurve(run, m):
    try: s = rm.loc[(run, m)]; return s.index.values, s.values
    except KeyError: return None
def rmean(runs, m):
    cs = [rcurve(r, m) for r in runs]; cs = [c for c in cs if c]
    return (cs[0][0], np.mean([c[1] for c in cs], axis=0)) if cs else None
RM = list(RESPAT) + ["SASA per residue (nm2)"]
fig, axs = plt.subplots(4, 2, figsize=(15, 17))
for ax, m in zip(axs.ravel(), RM):
    for run in RUNS:
        c = rcurve(run, m)
        if c: ax.plot(*c, color=COL[run], ls=LS[run], lw=1.2, label=run)
    ax.set(xlabel="Residue", ylabel=m, title=m + " (mean of 6 chains)"); ax.grid(alpha=.3); ax.legend(fontsize=7)
axs.ravel()[-1].axis("off"); fig.tight_layout()
fig.savefig(f"{CMP}/graphs/05_residue_overlays.png", dpi=150); plt.show()
DM = ["RMSF (nm)","SASA per residue (nm2)","Helix fraction","Strand fraction","H-bonds per residue"]
fig, axs = plt.subplots(len(DM), 2, figsize=(15, 3.2*len(DM))); top = []
for i, m in enumerate(DM):
    a300, a373, ash = rmean(COND["300 K thermal"], m), rmean(COND["373 K thermal"], m), rmean(COND["373 K shear"], m)
    if not (a300 and a373 and ash): continue
    resid = a373[0]
    for j, (name, dl) in enumerate((("Thermal: 373 K mean - 300 K mean", a373[1]-a300[1]), ("Shear: 373 K shear mean - 373 K thermal mean", ash[1]-a373[1]))):
        ax = axs[i, j]; ax.fill_between(resid, dl, 0, where=dl>=0, color="#d62728", alpha=.6)
        ax.fill_between(resid, dl, 0, where=dl<0, color="#1f77b4", alpha=.6)
        ax.axhline(0, color="k", lw=.6); ax.set(xlabel="Residue", ylabel=f"Δ {m}", title=name); ax.grid(alpha=.3)
        for k in np.argsort(-np.abs(dl))[:10]: top.append(dict(contrast=name, metric=m, residue=int(resid[k]), delta=float(dl[k])))
fig.tight_layout(); fig.savefig(f"{CMP}/graphs/06_residue_deltas.png", dpi=150); plt.show()
pd.DataFrame(top).round(4).to_csv(f"{CMP}/csv/top10_residue_changes.csv", index=False)

# ---------------- report ----------------
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
print("\nLast-half means, whole hexamer:\n", table.round(3))
sh = CT[CT.contrast.str.startswith("shear")].set_index("metric")[["baseline","value","delta","pct","gap_baseline","gap_value","bigger_than_both_gaps","replicates_separated"]]
th = CT[CT.contrast.str.startswith("thermal")].set_index("metric")[["baseline","value","delta","pct","gap_baseline","gap_value","bigger_than_both_gaps","replicates_separated"]]
print("\nShear effect (373 K shear vs 373 K thermal), against replicate gaps:\n", sh.round(3))
print("\nThermal effect (373 K vs 300 K), against replicate gaps:\n", th.round(3))
print("\nSaved to:", CMP)